In [16]:
import os
import datetime
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import Layout, VBox, HBox, HTML, Output
from IPython.display import display
import trino

# ==========================================
# 1. DATABASE CONNECTION & QUERY ENGINE
# ==========================================
class AhaMoveTrinoEngine:
    def __init__(self):
        self.host = os.getenv("TRINO_HOST", "trino-coordinator")
        self.port = int(os.getenv("TRINO_PORT", 8080))
        self.user = os.getenv("TRINO_USER", "admin")
        self.catalog = os.getenv("TRINO_CATALOG", "lakehouse")
        self.schema = os.getenv("TRINO_SCHEMA", "gold")

    def get_connection(self):
        return trino.dbapi.connect(
            host=self.host,
            port=self.port,
            user=self.user,
            catalog=self.catalog,
            schema=self.schema
        )

    def execute_query(self, query: str) -> pd.DataFrame:
        """Thực thi SQL Trino và trả về Pandas DataFrame"""
        try:
            with self.get_connection() as conn:
                cursor = conn.cursor()
                cursor.execute(query)
                cols = [desc[0] for desc in cursor.description]
                data = cursor.fetchall()
                return pd.DataFrame(data, columns=cols)
        except Exception as e:
            # Fallback sang Mock Data nếu không kết nối được Trino DB
            return pd.DataFrame()

db_engine = AhaMoveTrinoEngine()

# ==========================================
# 2. BRAND COLOR PALETTE & STYLING
# ==========================================
COLOR_PRIMARY = "#FF6B00"       # AhaMove Orange
COLOR_SECONDARY = "#0B192C"     # Deep Navy
COLOR_SUCCESS = "#2ECC71"       # Green
COLOR_DANGER = "#E74C3C"        # Red
COLOR_WARNING = "#F1C40F"       # Yellow
COLOR_GRAY = "#7F8C8D"          # Slate Gray

# ==========================================
# 3. INTERACTIVE GLOBAL FILTER CONTROLLERS
# ==========================================
date_start_widget = widgets.DatePicker(
    description='Từ ngày:',
    value=datetime.date(2026, 8, 1),
    layout=Layout(width='30%')
)

date_end_widget = widgets.DatePicker(
    description='Đến ngày:',
    value=datetime.date(2026, 8, 19),
    layout=Layout(width='30%')
)

service_type_widget = widgets.Dropdown(
    options=['ALL', 'BIKE_INSTANT', 'BIKE_SAMEDAY', 'TRUCK_500KG', 'TRUCK_1000KG'],
    value='ALL',
    description='Dịch vụ:',
    layout=Layout(width='30%')
)

partner_widget = widgets.Dropdown(
    options=['ALL', 'C2C Individual', 'Lazada Hub', 'Shopee Express', 'TikiNOW', 'Sendo'],
    value='ALL',
    description='Đối tác:',
    layout=Layout(width='30%')
)

btn_refresh = widgets.Button(
    description='Tải dữ liệu',
    button_style='warning',
    icon='refresh',
    layout=Layout(width='15%', margin='0px 0px 0px 10px')
)

dashboard_output = Output()

# ==========================================
# 4. SQL QUERY GENERATORS & VISUALIZATIONS
# ==========================================
def build_where_clause(start_date, end_date, service_id, partner_name):
    where_clauses = [f"d.full_date BETWEEN DATE '{start_date}' AND DATE '{end_date}'"]
    if service_id != 'ALL':
        where_clauses.append(f"f.service_id = '{service_id}'")
    if partner_name == 'C2C Individual':
        where_clauses.append("p.partner_name IS NULL")
    elif partner_name != 'ALL':
        where_clauses.append(f"p.partner_name = '{partner_name}'")
    return " AND ".join(where_clauses)

def render_section_1(start_date, end_date, service_id, partner_name):
    where_sql = build_where_clause(start_date, end_date, service_id, partner_name)
    
    # Chart 1: P50 & P90 Time To Board
    q1 = f"""
    SELECT 
        d.full_date,
        approx_percentile(date_diff('second', f.accept_time, f.board_time)/60.0, 0.50) AS p50_lta,
        approx_percentile(date_diff('second', f.accept_time, f.board_time)/60.0, 0.90) AS p90_lta
    FROM gold.fact_orders f
    JOIN gold.dim_date d ON f.create_date_sk = d.date_sk
    LEFT JOIN gold.dim_partner p ON f.partner_sk = p.partner_sk
    WHERE f.accept_time IS NOT NULL AND f.board_time IS NOT NULL AND f.board_time >= f.accept_time
      AND {where_sql}
    GROUP BY d.full_date
    ORDER BY d.full_date
    """
    df1 = db_engine.execute_query(q1)
    
    # Mock Data chuẩn AhaMove
    if df1.empty:
        dates = pd.date_range(start_date, end_date)
        n_days = len(dates)
        # P50 dao động 5.5 - 8.5 phút, P90 dao động 15 - 24 phút
        p50 = 6.5 + np.sin(np.linspace(0, 3, n_days)) * 1.2 + np.random.normal(0, 0.4, n_days)
        p90 = 17.5 + np.sin(np.linspace(0, 3, n_days)) * 2.8 + np.random.normal(0, 0.8, n_days)
        df1 = pd.DataFrame({
            'full_date': dates,
            'p50_lta': np.clip(p50, 5.0, 10.0),
            'p90_lta': np.clip(p90, 14.0, 26.0)
        })

    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=df1['full_date'], y=df1['p50_lta'], name='P50 (50% số lượng đơn - Phút)', line=dict(color=COLOR_SUCCESS, width=2.5)))
    fig1.add_trace(go.Scatter(x=df1['full_date'], y=df1['p90_lta'], name='P90 (90% số lượng đơn - Phút)', line=dict(color=COLOR_DANGER, width=2.5, dash='dash')))
    fig1.update_layout(title="<b>1. Biến động LTA (P50 vs P90 Time-to-Board) theo Ngày</b>", template="plotly_white", hovermode="x unified")

    # Chart 2: Pre-Boarding Cancel Rate
    q2 = f"""
    SELECT 
        d.full_date,
        CAST(COUNT(CASE WHEN f.cancel_time IS NOT NULL AND (f.board_time IS NULL OR f.cancel_time < f.board_time) AND f.cancel_by_user = TRUE THEN 1 END) AS DOUBLE) / NULLIF(COUNT(f.order_sk), 0) * 100 AS user_cancel_rate,
        CAST(COUNT(CASE WHEN f.cancel_time IS NOT NULL AND (f.board_time IS NULL OR f.cancel_time < f.board_time) AND (f.cancel_by_user = FALSE OR f.cancel_by_user IS NULL) THEN 1 END) AS DOUBLE) / NULLIF(COUNT(f.order_sk), 0) * 100 AS driver_system_cancel_rate
    FROM gold.fact_orders f
    JOIN gold.dim_date d ON f.create_date_sk = d.date_sk
    LEFT JOIN gold.dim_partner p ON f.partner_sk = p.partner_sk
    WHERE {where_sql}
    GROUP BY d.full_date
    ORDER BY d.full_date
    """
    df2 = db_engine.execute_query(q2)
    
    if df2.empty:
        dates = pd.date_range(start_date, end_date)
        n_days = len(dates)
        # User cancel 3.5% - 6.5%, Driver/Sys cancel 2.0% - 4.0%
        u_cancel = 4.8 + np.random.normal(0, 0.8, n_days)
        d_cancel = 2.7 + np.random.normal(0, 0.4, n_days)
        df2 = pd.DataFrame({
            'full_date': dates,
            'user_cancel_rate': np.clip(u_cancel, 3.0, 7.5),
            'driver_system_cancel_rate': np.clip(d_cancel, 1.5, 4.5)
        })

    fig2 = go.Figure()
    fig2.add_trace(go.Bar(x=df2['full_date'], y=df2['user_cancel_rate'], name='% Hủy do User', marker_color=COLOR_PRIMARY))
    fig2.add_trace(go.Bar(x=df2['full_date'], y=df2['driver_system_cancel_rate'], name='% Hủy do Tài xế/Hệ thống', marker_color=COLOR_SECONDARY))
    fig2.add_shape(type="line", x0=df2['full_date'].min(), x1=df2['full_date'].max(), y0=5, y1=5, line=dict(color="red", width=2, dash="dot"))
    fig2.update_layout(barmode='stack', title="<b>2. Tỷ lệ Hủy đơn Pre-Boarding (Ngưỡng cảnh báo User Cancel > 5%)</b>", template="plotly_white", yaxis_title="% Tỷ lệ Hủy")

    # Threshold Check & Alert Banner
    latest_p90_wow = (df1['p90_lta'].iloc[-1] - df1['p90_lta'].iloc[0]) / df1['p90_lta'].iloc[0] * 100 if len(df1) > 1 else 0
    latest_user_cancel = df2['user_cancel_rate'].iloc[-1]
    
    alert_html = ""
    if latest_p90_wow > 20:
        alert_html += f"<div style='background-color:#FADBD8; padding:10px; border-radius:5px; margin-bottom:10px;'><b style='color:#78281F;'>ALERT (P90 LTA):</b> P90 tăng {latest_p90_wow:.1f}% so với đầu kỳ (vượt ngưỡng 20%). Yêu cầu kiểm tra ngay Section 2 & 3.</div>"
    if latest_user_cancel > 5:
        alert_html += f"<div style='background-color:#FCF3CF; padding:10px; border-radius:5px; margin-bottom:10px;'><b style='color:#7D6608;'>ALERT (User Cancel):</b> Tỷ lệ User hủy đơn đạt {latest_user_cancel:.1f}% (>5%). Kích hoạt ngay đề xuất tăng pricing request_fee!</div>"
    
    return HTML(alert_html), fig1, fig2

def render_section_2(start_date, end_date, service_id, partner_name):
    where_sql = build_where_clause(start_date, end_date, service_id, partner_name)
    
    q3 = f"""
    SELECT 
        f.order_id,
        COALESCE(s.supplier_id, 'UNKNOWN') AS supplier_id,
        ST_Distance(
            to_spherical_geography(ST_Point(f.accept_lng, f.accept_lat)), 
            to_spherical_geography(ST_Point(f.from_lng, f.from_lat))
        ) AS dispatch_distance_meters,
        date_diff('second', f.accept_time, f.board_time) / 60.0 AS time_to_board_minutes
    FROM gold.fact_orders f
    JOIN gold.dim_date d ON f.create_date_sk = d.date_sk
    LEFT JOIN gold.dim_partner p ON f.partner_sk = p.partner_sk
    LEFT JOIN gold.dim_supplier s ON f.supplier_sk = s.supplier_sk
    WHERE f.accept_time IS NOT NULL AND f.board_time IS NOT NULL
      AND f.accept_lat IS NOT NULL AND f.from_lat IS NOT NULL
      AND {where_sql}
    LIMIT 3000
    """
    df3 = db_engine.execute_query(q3)
    
    # Mock Data chuẩn Spatial Multi-Apping
    if df3.empty:
        n = 600
        # Khoảng cách gán đơn từ 100m đến 4.5km
        dist = np.random.gamma(shape=2.0, scale=800.0, size=n)
        dist = np.clip(dist, 100, 5000)
        
        # Thời gian di chuyển chuẩn (~250m/phút + thời gian chờ/kẹt xe)
        tb = (dist / 250.0) + np.random.exponential(scale=2.5, size=n) + 2.0
        
        # Chèn 6% điểm dữ liệu gian lận Multi-Apping (<1km nhưng >10p)
        n_fraud = int(n * 0.06)
        dist[:n_fraud] = np.random.uniform(150, 850, n_fraud)
        tb[:n_fraud] = np.random.uniform(11.5, 26.0, n_fraud)
        
        df3 = pd.DataFrame({
            'order_id': [f'ORD_AHM_{10000 + i}' for i in range(n)],
            'supplier_id': [f'SUP_{np.random.randint(100, 250)}' for i in range(n)],
            'dispatch_distance_meters': dist,
            'time_to_board_minutes': tb
        })

    df3['is_suspect'] = (df3['dispatch_distance_meters'] < 1000) & (df3['time_to_board_minutes'] > 10)
    
    fig3 = px.scatter(
        df3, 
        x='dispatch_distance_meters', 
        y='time_to_board_minutes', 
        color='is_suspect',
        color_discrete_map={True: COLOR_DANGER, False: COLOR_SECONDARY},
        hover_data=['order_id', 'supplier_id'],
        trendline="ols",
        title="<b>3. Spatial Analysis: Dispatch Distance vs Time to Board (Phát hiện Multi-Apping)</b>"
    )
    fig3.add_shape(type="rect", x0=0, x1=1000, y0=10, y1=df3['time_to_board_minutes'].max(), fillcolor="red", opacity=0.15, line_width=0)
    fig3.update_layout(xaxis_title="Khoảng cách gán đơn (Meters)", yaxis_title="Thời gian đến lấy hàng (Phút)", template="plotly_white")
    
    suspect_suppliers = df3[df3['is_suspect']]['supplier_id'].unique().tolist()
    suspect_msg = f"<b>Cảnh báo Tình nghi Multi-Apping/Gian lận:</b> Phát hiện <b>{len(suspect_suppliers)}</b> tài xế (< 1km nhưng > 10 phút lấy hàng). Danh sách Top Supplier ID: {', '.join(suspect_suppliers[:5])}..."
    
    return HTML(f"<div style='background-color:#EBF5FB; padding:10px; border-radius:5px;'>{suspect_msg}</div>"), fig3

def render_section_3(start_date, end_date, service_id, partner_name):
    where_sql = build_where_clause(start_date, end_date, service_id, partner_name)
    
    # Chart 4: B2B Friction Horizontal Bar Chart
    q4 = f"""
    SELECT 
        COALESCE(p.partner_name, 'C2C Individual') AS partner_name,
        AVG(date_diff('second', f.accept_time, f.board_time) / 60.0) AS avg_board_time
    FROM gold.fact_orders f
    JOIN gold.dim_date d ON f.create_date_sk = d.date_sk
    LEFT JOIN gold.dim_partner p ON f.partner_sk = p.partner_sk
    WHERE f.accept_time IS NOT NULL AND f.board_time IS NOT NULL
      AND ST_Distance(to_spherical_geography(ST_Point(f.accept_lng, f.accept_lat)), to_spherical_geography(ST_Point(f.from_lng, f.from_lat))) < 2000
      AND {where_sql}
    GROUP BY COALESCE(p.partner_name, 'C2C Individual')
    ORDER BY avg_board_time ASC
    """
    df4 = db_engine.execute_query(q4)
    if df4.empty:
        df4 = pd.DataFrame({
            'partner_name': ['C2C Individual', 'TikiNOW', 'Sendo', 'Shopee Express', 'Lazada Hub'],
            'avg_board_time': [5.2, 6.4, 7.9, 9.1, 12.3]
        }).sort_values('avg_board_time')

    fig4 = px.bar(
        df4, 
        y='partner_name', 
        x='avg_board_time', 
        orientation='h',
        text_auto='.1f',
        color='avg_board_time',
        color_continuous_scale='Reds',
        title="<b>4. B2B vs C2C Boarding Friction (Thời gian lấy hàng trung bình bán kính < 2km)</b>"
    )
    fig4.update_layout(xaxis_title="Thời gian lấy hàng trung bình (Phút)", yaxis_title="Partner", template="plotly_white")

    # Chart 5: Idle Behavior Heatmap
    q5 = f"""
    SELECT 
        d.day_name,
        d.day_of_week,
        EXTRACT(HOUR FROM f.accept_time) AS accept_hour,
        CAST(COUNT(CASE WHEN date_diff('second', f.accept_time, f.board_time) > 900 THEN 1 END) AS DOUBLE) / NULLIF(COUNT(f.order_sk), 0) * 100 AS idle_rate
    FROM gold.fact_orders f
    JOIN gold.dim_date d ON f.create_date_sk = d.date_sk
    LEFT JOIN gold.dim_partner p ON f.partner_sk = p.partner_sk
    WHERE f.accept_time IS NOT NULL AND f.board_time IS NOT NULL
      AND ST_Distance(to_spherical_geography(ST_Point(f.accept_lng, f.accept_lat)), to_spherical_geography(ST_Point(f.from_lng, f.from_lat))) < 2000
      AND {where_sql}
    GROUP BY d.day_name, d.day_of_week, EXTRACT(HOUR FROM f.accept_time)
    ORDER BY d.day_of_week, accept_hour
    """
    df5 = db_engine.execute_query(q5)
    
    if df5.empty:
        days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        hours = list(range(24))
        grid = []
        for d in days:
            for h in hours:
                # Giờ cao điểm trưa (11h-13h) và chiều tối (17h-19h) có % ngâm đơn cao vọt
                if h in [11, 12, 13]:
                    val = np.random.uniform(22, 32)
                elif h in [17, 18, 19]:
                    val = np.random.uniform(18, 26)
                elif 8 <= h <= 20:
                    val = np.random.uniform(7, 14)
                else:
                    val = np.random.uniform(2, 6)
                grid.append({'day_name': d, 'accept_hour': h, 'idle_rate': val})
        df5 = pd.DataFrame(grid)

    pivot_df5 = df5.pivot(index='day_name', columns='accept_hour', values='idle_rate')
    # Sắp xếp lại thứ tự ngày trong tuần cho chuẩn
    days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    pivot_df5 = pivot_df5.reindex([d for d in days_order if d in pivot_df5.index])
    
    fig5 = px.imshow(
        pivot_df5,
        labels=dict(x="Khung giờ chấp nhận đơn (0-23h)", y="Thứ trong tuần", color="% Ngâm đơn (>15p)"),
        color_continuous_scale='YlOrRd',
        title="<b>5. Ma trận Tỷ lệ Ngâm đơn (>15 phút, Khoảng cách < 2km) theo Khung Giờ</b>"
    )
    fig5.update_layout(template="plotly_white")

    return fig4, fig5

# ==========================================
# 5. SECTION 4: GRANULAR ORDER TRACKING TABLE
# ==========================================
def render_section_4(start_date, end_date, service_id, partner_name):
    where_sql = build_where_clause(start_date, end_date, service_id, partner_name)
    
    q6 = f"""
    SELECT 
        f.order_id,
        f.service_id,
        COALESCE(p.partner_name, 'C2C Individual') AS partner_name,
        COALESCE(s.supplier_id, 'UNKNOWN') AS supplier_id,
        f.accept_time,
        f.board_time,
        date_diff('second', f.accept_time, f.board_time) AS lta_seconds,
        ROUND(ST_Distance(
            to_spherical_geography(ST_Point(f.accept_lng, f.accept_lat)), 
            to_spherical_geography(ST_Point(f.from_lng, f.from_lat))
        ), 1) AS dispatch_distance_m,
        CASE 
            WHEN f.board_time IS NOT NULL THEN 'BOARDED'
            WHEN f.cancel_time IS NOT NULL AND f.cancel_by_user = TRUE THEN 'CANCELLED_USER'
            WHEN f.cancel_time IS NOT NULL THEN 'CANCELLED_DRIVER_SYS'
            ELSE 'IN_TRANSIT_TO_PICKUP'
        END AS phase_status,
        CASE 
            WHEN ST_Distance(to_spherical_geography(ST_Point(f.accept_lng, f.accept_lat)), to_spherical_geography(ST_Point(f.from_lng, f.from_lat))) < 1000 
                 AND date_diff('second', f.accept_time, f.board_time) > 600 THEN 'SUSPECT_MULTI_APP'
            WHEN date_diff('second', f.accept_time, f.board_time) > 900 THEN 'HIGH_LTA_DELAY'
            ELSE 'NORMAL'
        END AS anomaly_flag
    FROM gold.fact_orders f
    JOIN gold.dim_date d ON f.create_date_sk = d.date_sk
    LEFT JOIN gold.dim_partner p ON f.partner_sk = p.partner_sk
    LEFT JOIN gold.dim_supplier s ON f.supplier_sk = s.supplier_sk
    WHERE f.accept_time IS NOT NULL
      AND {where_sql}
    ORDER BY f.accept_time DESC
    LIMIT 200
    """
    df6 = db_engine.execute_query(q6)

    # Fallback Mock Data chuẩn AhaMove Order Tracking
    if df6.empty:
        n = 50
        start_dt = datetime.datetime.combine(start_date, datetime.time(8, 0))
        end_dt = datetime.datetime.combine(end_date, datetime.time(20, 0))
        time_span = max(int((end_dt - start_dt).total_seconds()), 3600)
        
        random_seconds = np.random.uniform(0, time_span, n)
        accept_times = [start_dt + datetime.timedelta(seconds=s) for s in random_seconds]
        
        services = ['BIKE_INSTANT', 'BIKE_SAMEDAY', 'TRUCK_500KG', 'TRUCK_1000KG']
        partners = ['C2C Individual', 'Lazada Hub', 'Shopee Express', 'TikiNOW', 'Sendo']
        
        records = []
        for i in range(n):
            acc_t = accept_times[i]
            srv = np.random.choice(services, p=[0.65, 0.20, 0.10, 0.05])
            prt = np.random.choice(partners, p=[0.45, 0.20, 0.15, 0.10, 0.10])
            sup_id = f"SUP_{np.random.randint(100, 999)}"
            ord_id = f"ORD_AHM_{100000 + i}"
            
            rand_scenario = np.random.random()
            
            if rand_scenario < 0.70:  # Normal Boarded
                dist = np.random.uniform(300, 3200)
                lta = int(dist / 4.5 + np.random.uniform(60, 240))
                phase = 'BOARDED'
                flag = 'NORMAL'
                board_t = acc_t + datetime.timedelta(seconds=lta)
            elif rand_scenario < 0.80:  # Suspect Multi-App (<1000m & >600s)
                dist = np.random.uniform(150, 850)
                lta = int(np.random.uniform(620, 1150))
                phase = 'BOARDED'
                flag = 'SUSPECT_MULTI_APP'
                board_t = acc_t + datetime.timedelta(seconds=lta)
            elif rand_scenario < 0.90:  # High LTA Delay (>900s)
                dist = np.random.uniform(1800, 4800)
                lta = int(np.random.uniform(930, 1750))
                phase = 'BOARDED'
                flag = 'HIGH_LTA_DELAY'
                board_t = acc_t + datetime.timedelta(seconds=lta)
            else:  # Cancelled
                dist = np.random.uniform(200, 2500)
                lta = None
                board_t = None
                phase = np.random.choice(['CANCELLED_USER', 'CANCELLED_DRIVER_SYS'], p=[0.65, 0.35])
                flag = 'NORMAL'
            
            records.append({
                'order_id': ord_id,
                'service_id': srv,
                'partner_name': prt,
                'supplier_id': sup_id,
                'accept_time': acc_t.strftime('%Y-%m-%d %H:%M:%S'),
                'board_time': board_t.strftime('%Y-%m-%d %H:%M:%S') if board_t else None,
                'lta_seconds': lta,
                'dispatch_distance_m': round(dist, 1),
                'phase_status': phase,
                'anomaly_flag': flag
            })
        
        df6 = pd.DataFrame(records)
        df6 = df6.sort_values(by='accept_time', ascending=False).reset_index(drop=True)

    # Highlight trạng thái trên Pandas Table Styler
    def highlight_status(val):
        if val == 'SUSPECT_MULTI_APP':
            return 'background-color: #FADBD8; color: #78281F; font-weight: bold;'
        elif val == 'HIGH_LTA_DELAY':
            return 'background-color: #FCF3CF; color: #7D6608; font-weight: bold;'
        elif val == 'BOARDED':
            return 'color: #2ECC71; font-weight: bold;'
        elif 'CANCELLED' in str(val):
            return 'color: #E74C3C;'
        return ''

    styled_df = df6.style.map(
        highlight_status, 
        subset=['anomaly_flag', 'phase_status']
    ).format({
        'lta_seconds': lambda x: f"{x:.0f}" if pd.notna(x) else "-",
        'dispatch_distance_m': '{:.1f}'
    })

    table_title = HTML("<h3 style='color:#0B192C; margin-top:25px;'>6. Bảng theo dõi Chi tiết Đơn hàng (ACCEPTED ➔ BOARDED Phase)</h3>")
    return table_title, styled_df

# ==========================================
# 6. DASHBOARD CONTROLLER & RENDER PIPELINE
# ==========================================
def update_dashboard(b=None):
    dashboard_output.clear_output(wait=True)
    with dashboard_output:
        s_date = date_start_widget.value
        e_date = date_end_widget.value
        service = service_type_widget.value
        partner = partner_widget.value
        
        # Section 1: Trends & Alerts
        alert_banner, fig1, fig2 = render_section_1(s_date, e_date, service, partner)
        display(alert_banner)
        display(fig1)
        display(fig2)
        
        # Section 2: Spatial Fraud & Multi-App
        suspect_banner, fig3 = render_section_2(s_date, e_date, service, partner)
        display(suspect_banner)
        display(fig3)
        
        # Section 3: B2B Friction & Heatmap
        fig4, fig5 = render_section_3(s_date, e_date, service, partner)
        display(fig4)
        display(fig5)
        
        # Section 4: Order Tracking Table
        tbl_title, styled_table = render_section_4(s_date, e_date, service, partner)
        display(tbl_title)
        display(styled_table)

btn_refresh.on_click(update_dashboard)

# Layout Setup
filter_bar = HBox([date_start_widget, date_end_widget, service_type_widget, partner_widget, btn_refresh], 
                  layout=Layout(background_color='#F4F6F7', padding='10px', border_radius='5px', margin='0px 0px 15px 0px'))

header = HTML("<h1 style='color:#FF6B00; text-align:center;'>AHAMOVE LTA & BOARDING FRICTION MONITORING DASHBOARD</h1>")

# Display App
dashboard_app = VBox([header, filter_bar, dashboard_output])
display(dashboard_app)

# Trigger initial load
update_dashboard()

In [17]:
import os
import datetime
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Layout, VBox, HBox, HTML, Output
from IPython.display import display
import trino

# ==========================================
# 1. CONFIGURATION & CONSTANTS
# ==========================================
COLOR_PRIMARY = "#FF6B00"       # AhaMove Orange (P90 Tail Risk)
COLOR_SUCCESS = "#2ECC71"       # Green (P50 Baseline)
COLOR_DANGER = "#E74C3C"        # Red (Tipping Point / Threshold)
COLOR_NAVY = "#0B192C"          # Deep Navy
COLOR_GRAY = "#95A5A6"          # Neutral Gray

# Cấu hình chuẩn hóa SLA & Tipping Point cho 4 dịch vụ Xe Máy
SERVICE_CONFIG = {
    'SIEU_TOC': {
        'name': 'Siêu Tốc',
        'p50_target': 6.0,
        'p90_target': 11.0,
        'tipping_point': 15.0,
        'grid_pos': (1, 1)
    },
    'NHANH': {
        'name': 'Nhanh',
        'p50_target': 8.0,
        'p90_target': 16.0,
        'tipping_point': 22.0,
        'grid_pos': (1, 2)
    },
    '4H': {
        'name': '4H',
        'p50_target': 18.0,
        'p90_target': 35.0,
        'tipping_point': 45.0,
        'grid_pos': (2, 1)
    },
    'DONG_GIA': {
        'name': 'Đồng Giá',
        'p50_target': 25.0,
        'p90_target': 45.0,
        'tipping_point': 60.0,
        'grid_pos': (2, 2)
    }
}

# ==========================================
# 2. DATABASE CONNECTION ENGINE
# ==========================================
class AhaMoveTrinoEngine:
    def __init__(self):
        self.host = os.getenv("TRINO_HOST", "trino-coordinator")
        self.port = int(os.getenv("TRINO_PORT", 8080))
        self.user = os.getenv("TRINO_USER", "admin")
        self.catalog = os.getenv("TRINO_CATALOG", "lakehouse")
        self.schema = os.getenv("TRINO_SCHEMA", "gold")

    def get_connection(self):
        return trino.dbapi.connect(
            host=self.host,
            port=self.port,
            user=self.user,
            catalog=self.catalog,
            schema=self.schema
        )

    def execute_query(self, query: str) -> pd.DataFrame:
        """Thực thi SQL Trino và trả về Pandas DataFrame"""
        try:
            with self.get_connection() as conn:
                cursor = conn.cursor()
                cursor.execute(query)
                cols = [desc[0] for desc in cursor.description]
                data = cursor.fetchall()
                return pd.DataFrame(data, columns=cols)
        except Exception:
            # Fallback Mock Data nếu không có kết nối DB live
            return pd.DataFrame()

db_engine = AhaMoveTrinoEngine()

# ==========================================
# 3. INTERACTIVE FILTER CONTROLLERS
# ==========================================
date_start_widget = widgets.DatePicker(
    description='Từ ngày:',
    value=datetime.date(2026, 8, 1),
    layout=Layout(width='23%')
)

date_end_widget = widgets.DatePicker(
    description='Đến ngày:',
    value=datetime.date(2026, 8, 19),
    layout=Layout(width='23%')
)

district_widget = widgets.Dropdown(
    options=['ALL', 'Quận 1', 'Quận 3', 'Quận 7', 'Thủ Đức', 'Tân Bình', 'Bình Thạnh'],
    value='ALL',
    description='Khu vực:',
    layout=Layout(width='23%')
)

time_bucket_widget = widgets.Dropdown(
    options=['ALL', 'Peak (10h-13h & 17h-19h)', 'Non-Peak'],
    value='ALL',
    description='Khung giờ:',
    layout=Layout(width='23%')
)

btn_refresh = widgets.Button(
    description='Cập Nhật Dashboard',
    button_style='warning',
    icon='refresh',
    layout=Layout(width='100%', margin='10px 0px')
)

dashboard_output = Output()

# ==========================================
# 4. QUERY BUILDER & MOCK DATA GENERATOR
# ==========================================
def build_lta_trino_query(start_date, end_date, district, time_bucket) -> str:
    """Tạo Trino SQL Query chuẩn hóa cho First Mile LTA"""
    where_clauses = [
        "city_id = 'SGN'",
        "vehicle_type = 'BIKE'",
        "service_id IN ('SIEU_TOC', 'NHANH', '4H', 'DONG_GIA')",
        "status = 'COMPLETED'",
        "board_time > accept_time",
        "DATE_DIFF('hour', accept_time, board_time) < 4",
        f"DATE(accept_time AT TIME ZONE 'Asia/Ho_Chi_Minh') BETWEEN DATE '{start_date}' AND DATE '{end_date}'"
    ]
    
    if district != 'ALL':
        where_clauses.append(f"pickup_district = '{district}'")
        
    if time_bucket == 'Peak (10h-13h & 17h-19h)':
        where_clauses.append("(EXTRACT(HOUR FROM accept_time AT TIME ZONE 'Asia/Ho_Chi_Minh') IN (10, 11, 12, 17, 18))")
    elif time_bucket == 'Non-Peak':
        where_clauses.append("(EXTRACT(HOUR FROM accept_time AT TIME ZONE 'Asia/Ho_Chi_Minh') NOT IN (10, 11, 12, 17, 18))")

    where_str = "\n  AND ".join(where_clauses)

    return f"""
    SELECT 
        service_id,
        DATE(accept_time AT TIME ZONE 'Asia/Ho_Chi_Minh') AS full_date,
        ROUND(APPROX_PERCENTILE(DATE_DIFF('second', accept_time, board_time) / 60.0, 0.50), 2) AS lta_p50_min,
        ROUND(APPROX_PERCENTILE(DATE_DIFF('second', accept_time, board_time) / 60.0, 0.90), 2) AS lta_p90_min
    FROM fact_orders
    WHERE 1=1
      AND {where_str}
    GROUP BY 1, 2
    ORDER BY 1, 2
    """

def generate_mock_lta_data(start_date, end_date) -> pd.DataFrame:
    """Tạo Mock Data mô phỏng thực tế khi không có kết nối Trino DB"""
    dates = pd.date_range(start_date, end_date)
    records = []
    
    np.random.seed(42)
    for s_id, cfg in SERVICE_CONFIG.items():
        n = len(dates)
        # Giả lập xu hướng với nhiễu ngẫu nhiên và vài điểm bùng phát vỡ Tipping Point
        p50_base = cfg['p50_target'] + np.random.normal(0, 0.5, n)
        p90_base = cfg['p90_target'] + np.random.normal(0, 1.5, n)
        
        # Thêm biến động ngập nước/kẹt xe ngày 8 & 15
        if n > 8:
            p90_base[7] += cfg['tipping_point'] * 0.25
        if n > 15:
            p90_base[14] += cfg['tipping_point'] * 0.35

        for i, dt in enumerate(dates):
            records.append({
                'service_id': s_id,
                'full_date': dt.strftime('%Y-%m-%d'),
                'lta_p50_min': round(max(1.0, p50_base[i]), 2),
                'lta_p90_min': round(max(p50_base[i] + 1.0, p90_base[i]), 2)
            })
            
    return pd.DataFrame(records)

# ==========================================
# 5. GRID DASHBOARD VISUALIZATION
# ==========================================
def render_lta_dashboard(b):
    with dashboard_output:
        dashboard_output.clear_output(wait=True)
        
        start_dt = date_start_widget.value
        end_dt = date_end_widget.value
        dist = district_widget.value
        t_bucket = time_bucket_widget.value

        # 1. Lấy dữ liệu
        query = build_lta_trino_query(start_dt, end_dt, dist, t_bucket)
        df = db_engine.execute_query(query)
        
        if df.empty:
            df = generate_mock_lta_data(start_dt, end_dt)

        df['full_date'] = pd.to_datetime(df['full_date']).dt.strftime('%Y-%m-%d')

        # 2. Cấu trúc Subplots Grid 2x2
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=[
                f"<b>{cfg['name']}</b> (Target P50: {cfg['p50_target']}m | P90: {cfg['p90_target']}m)"
                for cfg in SERVICE_CONFIG.values()
            ],
            vertical_spacing=0.15,
            horizontal_spacing=0.08
        )

        # 3. Vẽ biểu đồ cho từng dịch vụ
        for s_id, cfg in SERVICE_CONFIG.items():
            row, col = cfg['grid_pos']
            sub_df = df[df['service_id'] == s_id].sort_values('full_date')

            if sub_df.empty:
                continue

            # Line P50 (Target Baseline - Green)
            fig.add_trace(
                go.Scatter(
                    x=sub_df['full_date'],
                    y=sub_df['lta_p50_min'],
                    name='P50 (Median)',
                    mode='lines+markers',
                    line=dict(color=COLOR_SUCCESS, width=2),
                    legendgroup='P50',
                    showlegend=(row == 1 and col == 1),
                    hovertemplate="Ngày: %{x}<br>P50 LTA: %{y} phút<extra></extra>"
                ),
                row=row, col=col
            )

            # Line P90 (Tail-Risk - Orange)
            fig.add_trace(
                go.Scatter(
                    x=sub_df['full_date'],
                    y=sub_df['lta_p90_min'],
                    name='P90 (Tail-Risk)',
                    mode='lines+markers',
                    line=dict(color=COLOR_PRIMARY, width=2.5),
                    legendgroup='P90',
                    showlegend=(row == 1 and col == 1),
                    hovertemplate="Ngày: %{x}<br>P90 LTA: %{y} phút<extra></extra>"
                ),
                row=row, col=col
            )

            # Horizontal Reference Line (Tipping Point - Red)
            fig.add_hline(
                y=cfg['tipping_point'],
                line_dash="dash",
                line_color=COLOR_DANGER,
                line_width=1.5,
                annotation_text=f"Red Alert (> {cfg['tipping_point']}m)",
                annotation_position="top right",
                annotation_font_color=COLOR_DANGER,
                row=row, col=col
            )

            # Highlight các điểm vi phạm Tipping Point (Red Alert Markers)
            alert_df = sub_df[sub_df['lta_p90_min'] > cfg['tipping_point']]
            if not alert_df.empty:
                fig.add_trace(
                    go.Scatter(
                        x=alert_df['full_date'],
                        y=alert_df['lta_p90_min'],
                        mode='markers',
                        marker=dict(color=COLOR_DANGER, size=10, symbol='x'),
                        name='Vỡ Tipping Point',
                        legendgroup='Alert',
                        showlegend=(row == 1 and col == 1),
                        hovertemplate="<b>CẢNH BÁO RED ALERT</b><br>Ngày: %{x}<br>P90 LTA: %{y} phút<extra></extra>"
                    ),
                    row=row, col=col
                )

            # Cập nhật Trục Y cho từng subplot
            fig.update_yaxes(title_text="Phút", row=row, col=col, gridcolor="#F0F0F0")
            fig.update_xaxes(showgrid=True, gridcolor="#F0F0F0", tickangle=-45, row=row, col=col)

        # 4. Định dạng Layout Tổng thể
        fig.update_layout(
            height=750,
            title_text="<b>AHAMOVE SGN - FIRST MILE LEAD TIME (TIME-TO-BOARD) MONITORING</b>",
            title_font_size=18,
            title_font_color=COLOR_NAVY,
            template="plotly_white",
            hovermode="x unified",
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.05,
                xanchor="right",
                x=1
            ),
            margin=dict(t=100, b=50, l=50, r=50)
        )

        fig.show()

# Bind sự kiện Click Button
btn_refresh.on_click(render_lta_dashboard)

# ==========================================
# 6. DISPLAY DASHBOARD
# ==========================================
header_html = HTML("""
<div style="background-color: #0B192C; padding: 12px 20px; border-radius: 6px; margin-bottom: 12px;">
    <h3 style="color: #FF6B00; margin: 0; font-family: Arial, sans-serif;">🚚 AhaMove Operational Excellence - First Mile LTA Dashboard</h3>
    <p style="color: #FFFFFF; margin: 4px 0 0 0; font-size: 13px;">Theo dõi thời gian Chấp nhận đơn đến khi Có mặt tại điểm lấy (Time-to-Board) cho dịch vụ Xe máy TP.HCM</p>
</div>
""")

filter_bar = HBox([date_start_widget, date_end_widget, district_widget, time_bucket_widget])
ui = VBox([header_html, filter_bar, btn_refresh, dashboard_output])

display(ui)
# Trigger vẽ biểu đồ lần đầu
render_lta_dashboard(None)

In [10]:
import os
import datetime
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Layout, VBox, HBox, HTML, Accordion, Output
from IPython.display import display
import trino

# ==========================================
# 1. CONFIGURATION & CONSTANTS
# ==========================================
COLOR_PRIMARY = "#FF6B00"       # AhaMove Orange (P90 Tail Risk)
COLOR_SUCCESS = "#2ECC71"       # Green (P50 Baseline)
COLOR_DANGER = "#E74C3C"        # Red (Tipping Point / Alert)
COLOR_NAVY = "#0B192C"          # Deep Navy
COLOR_BG_CARD = "#F8F9FA"       # Light Gray for KPI Cards

SERVICE_CONFIG = {
    'SIEU_TOC': {'name': 'Siêu Tốc', 'p50_target': 6.0, 'p90_target': 11.0, 'tipping_point': 15.0, 'grid_pos': (1, 1)},
    'NHANH': {'name': 'Nhanh', 'p50_target': 8.0, 'p90_target': 16.0, 'tipping_point': 22.0, 'grid_pos': (1, 2)},
    '4H': {'name': '4H', 'p50_target': 18.0, 'p90_target': 35.0, 'tipping_point': 45.0, 'grid_pos': (2, 1)},
    'DONG_GIA': {'name': 'Đồng Giá', 'p50_target': 25.0, 'p90_target': 45.0, 'tipping_point': 60.0, 'grid_pos': (2, 2)}
}

# ==========================================
# 2. DATA GENERATOR & LOGIC (SPC + ALERT)
# ==========================================
def get_dashboard_data(start_date, end_date, district, time_bucket) -> pd.DataFrame:
    """Tạo Mock Data chuẩn hóa có tính toán SPC Baseline (mu_4w + 2*sigma)"""
    dates = pd.date_range(start_date, end_date)
    records = []
    np.random.seed(42)
    
    for s_id, cfg in SERVICE_CONFIG.items():
        n = len(dates)
        p50_base = cfg['p50_target'] + np.random.normal(0, 0.4, n)
        p90_base = cfg['p90_target'] + np.random.normal(0, 1.2, n)
        
        # Mô phỏng điểm bùng phát sự cố (Red Alert)
        if n > 5: p90_base[4] += cfg['tipping_point'] * 0.35  # Vỡ Tipping Point & SPC
        if n > 12: p90_base[11] += cfg['tipping_point'] * 0.40

        for i, dt in enumerate(dates):
            p50_val = round(max(1.0, p50_base[i]), 2)
            p90_val = round(max(p50_val + 1.0, p90_base[i]), 2)
            
            # Giả lập baseline 4 tuần (mu_4w) và độ lệch chuẩn (sigma)
            mu_4w = cfg['p90_target'] * 0.95
            sigma = 1.5
            spc_threshold = round(mu_4w + 2 * sigma, 2)
            
            # Logic Dual-Condition Trigger Red Alert
            is_red_alert = (p90_val > cfg['tipping_point']) and (p90_val > spc_threshold)
            
            records.append({
                'service_id': s_id,
                'full_date': dt.strftime('%Y-%m-%d'),
                'lta_p50_min': p50_val,
                'lta_p90_min': p90_val,
                'spc_threshold': spc_threshold,
                'is_red_alert': is_red_alert
            })
            
    return pd.DataFrame(records)

# ==========================================
# 3. LAYER 1: EXECUTIVE KPI CARDS COMPONENT
# ==========================================
def render_kpi_cards_html(df: pd.DataFrame) -> str:
    """Tạo Tầng 1: HTML Executive Summary KPI Cards cho Boardview"""
    total_records = len(df)
    red_alerts_count = df['is_red_alert'].sum() if not df.empty else 0
    
    # Tính % tuân thủ SLA (P90 <= Tipping Point)
    if total_records > 0:
        sla_pass = sum(df['lta_p90_min'] <= df['service_id'].map(lambda x: SERVICE_CONFIG[x]['tipping_point']))
        sla_compliance = round((sla_pass / total_records) * 100, 1)
        late_rate = round(100.0 - sla_compliance, 1)
    else:
        sla_compliance, late_rate = 100.0, 0.0

    sla_color = COLOR_SUCCESS if sla_compliance >= 90 else COLOR_DANGER
    alert_color = COLOR_DANGER if red_alerts_count > 0 else COLOR_SUCCESS

    return f"""
    <div style="display: flex; gap: 15px; margin-bottom: 15px; font-family: Arial, sans-serif;">
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 12px 18px; border-radius: 8px; border-left: 5px solid {sla_color}; box-shadow: 0 1px 3px rgba(0,0,0,0.1);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">SLA Compliance (P90)</span>
            <div style="font-size: 22px; font-weight: bold; color: {sla_color}; margin-top: 2px;">{sla_compliance}% <span style="font-size: 14px;">🟢</span></div>
        </div>
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 12px 18px; border-radius: 8px; border-left: 5px solid {alert_color}; box-shadow: 0 1px 3px rgba(0,0,0,0.1);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">Red Alerts Phát Hiện</span>
            <div style="font-size: 22px; font-weight: bold; color: {alert_color}; margin-top: 2px;">{red_alerts_count} Sự cố <span style="font-size: 14px;">🔴</span></div>
        </div>
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 12px 18px; border-radius: 8px; border-left: 5px solid {COLOR_NAVY}; box-shadow: 0 1px 3px rgba(0,0,0,0.1);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">Tỷ Lệ Đơn Vỡ LTA Chặng Đầu</span>
            <div style="font-size: 22px; font-weight: bold; color: {COLOR_NAVY}; margin-top: 2px;">{late_rate}% <span style="font-size: 14px;">⚪</span></div>
        </div>
    </div>
    """

# ==========================================
# 4. LAYER 3: COLLAPSIBLE READ-ME COMPONENT
# ==========================================
def create_documentation_accordion() -> Accordion:
    """Tạo Tầng 3: Documentation dạng Collapsible Accordion cho BA / Newbie"""
    
    doc_section_1 = HTML("""
    <div style="font-family: Arial, sans-serif; font-size: 13px; line-height: 1.5; color: #2C3E50;">
        <p><b>1. Ý nghĩa các đường hiển thị:</b></p>
        <ul>
            <li><b style="color:#2ECC71;">Đường xanh lá (P50 - Median):</b> Thời gian chuẩn của 50% đơn hàng thông thường. Biểu thị năng lực vận hành nền.</li>
            <li><b style="color:#FF6B00;">Đường cam (P90 - Tail Risk):</b> Thời gian của 10% đơn hàng chậm nhất. Nhóm có rủi ro hủy đơn cao nhất.</li>
            <li><b style="color:#E74C3C;">Vạch nét đứt đỏ (Tipping Point):</b> Ngưỡng thời gian tới hạn. Vượt quá mốc này, tỷ lệ người gửi hủy đơn vọt theo hàm số mũ.</li>
            <li><b style="color:#E74C3C;">Biểu tượng X đỏ:</b> Cảnh báo đỏ (Red Alert) kích hoạt khi P90 vi phạm ngưỡng vận hành & bất thường thống kê.</li>
        </ul>
        <table style="width:100%; border-collapse: collapse; margin-top: 8px; font-size: 12px;">
            <tr style="background-color: #0B192C; color: white;">
                <th style="padding: 6px; border: 1px solid #ddd;">Dịch vụ</th>
                <th style="padding: 6px; border: 1px solid #ddd;">P50 Target</th>
                <th style="padding: 6px; border: 1px solid #ddd;">P90 Target</th>
                <th style="padding: 6px; border: 1px solid #ddd;">Ngưỡng Báo Đỏ</th>
                <th style="padding: 6px; border: 1px solid #ddd;">Lý do Business & Điểm gãy tâm lý</th>
            </tr>
            <tr>
                <td style="padding: 6px; border: 1px solid #ddd;"><b>Siêu Tốc</b></td>
                <td style="padding: 6px; border: 1px solid #ddd;">6.0 phút</td>
                <td style="padding: 6px; border: 1px solid #ddd;">11.0 phút</td>
                <td style="padding: 6px; border: 1px solid #ddd; color: red;"><b>> 15.0 phút</b></td>
                <td style="padding: 6px; border: 1px solid #ddd;">Đơn hỏa tốc/đồ ăn. Quá 15m LTA, tỷ lệ hủy đơn vọt từ 2% lên > 11%.</td>
            </tr>
            <tr style="background-color: #F9F9F9;">
                <td style="padding: 6px; border: 1px solid #ddd;"><b>Nhanh</b></td>
                <td style="padding: 6px; border: 1px solid #ddd;">8.0 phút</td>
                <td style="padding: 6px; border: 1px solid #ddd;">16.0 phút</td>
                <td style="padding: 6px; border: 1px solid #ddd; color: red;"><b>> 22.0 phút</b></td>
                <td style="padding: 6px; border: 1px solid #ddd;">Khách nhạy cảm giá. SLA 90m cho phép LTA tối đa 22m.</td>
            </tr>
            <tr>
                <td style="padding: 6px; border: 1px solid #ddd;"><b>4H</b></td>
                <td style="padding: 6px; border: 1px solid #ddd;">18.0 phút</td>
                <td style="padding: 6px; border: 1px solid #ddd;">35.0 phút</td>
                <td style="padding: 6px; border: 1px solid #ddd; color: red;"><b>> 45.0 phút</b></td>
                <td style="padding: 6px; border: 1px solid #ddd;">Gom 5–8 đơn. Quá 45m LTA làm vỡ chuỗi lộ trình giao 4 giờ (TSP route).</td>
            </tr>
            <tr style="background-color: #F9F9F9;">
                <td style="padding: 6px; border: 1px solid #ddd;"><b>Đồng Giá</b></td>
                <td style="padding: 6px; border: 1px solid #ddd;">25.0 phút</td>
                <td style="padding: 6px; border: 1px solid #ddd;">45.0 phút</td>
                <td style="padding: 6px; border: 1px solid #ddd; color: red;"><b>> 60.0 phút</b></td>
                <td style="padding: 6px; border: 1px solid #ddd;">Lấy hàng theo khung hẹn. Chỉ cần hoàn tất trước giờ Cut-off slot giao.</td>
            </tr>
        </table>
    </div>
    """)

    doc_section_2 = HTML("""
    <div style="font-family: Arial, sans-serif; font-size: 13px; line-height: 1.5; color: #2C3E50;">
        <p>Cảnh báo đỏ (dấu X đỏ) chỉ kích hoạt khi thỏa mãn <b>đồng thời 2 điều kiện (Dual-Condition)</b>:</p>
        <div style="background: #FFF3CD; padding: 10px; border-left: 4px solid #FFC107; font-family: monospace; margin: 8px 0;">
            Trigger Red Alert ⟺ P90 (Thực tế) > max( Tipping Point Dịch Vụ ,  μ4w + 2σ )
        </div>
        <ul>
            <li><b>Điều kiện 1 (Business Limit):</b> P90 vượt quá điểm gãy tâm lý hủy đơn của dịch vụ.</li>
            <li><b>Điều kiện 2 (Statistical Anomaly):</b> P90 cao vượt 2 độ lệch chuẩn (+2σ) so với trung bình cùng khung giờ 4 tuần gần nhất (μ4w).</li>
        </ul>
        <p>💡 <b>Cơ chế lọc nhiễu:</b> Khi mưa bão diện rộng làm toàn thành phố kẹt xe, baseline (μ4w + 2σ) tự động tăng lên. Hệ thống sẽ <b>KHÔNG bắn alert giả</b>, giúp Ops tập trung đúng vào các sự cố đứt gãy Supply cục bộ.</p>
    </div>
    """)

    doc_section_3 = HTML("""
    <div style="font-family: Arial, sans-serif; font-size: 13px; line-height: 1.5; color: #2C3E50;">
        <ol>
            <li><b>Bước 1 - Khoanh vùng địa bàn (Filter Region):</b> Chọn bộ lọc Khu vực trên cùng Dashboard để tìm xem P90 đang bị vỡ ở Quận/Hotspot nào.</li>
            <li><b>Bước 2 - Bóc tách nguyên nhân (Supply/Demand Gap):</b> Kiểm tra tỷ lệ tài xế từ chối đơn (Rejection Rate) và Bán kính gán đơn (Matching Radius) tại Hotspot đó.</li>
            <li><b>Bước 3 - Can thiệp khẩn cấp (Ops Action):</b>
                <ul>
                    <li>Bật Surge Bonus (Thưởng nóng 5.000đ – 10.000đ/đơn) tại vùng thiếu Supply.</li>
                    <li>Điều chỉnh thu hẹp/nới rộng bán kính Matching tạm thời trên hệ thống.</li>
                </ul>
            </li>
        </ol>
    </div>
    """)

    accordion = Accordion(children=[doc_section_1, doc_section_2, doc_section_3])
    accordion.set_title(0, "📖 Hướng Dẫn Đọc Biểu Đồ & Bảng Tra Cứu SLA")
    accordion.set_title(1, "⚡ Thuật Toán Bật Cảnh Báo Động (SPC Logic μ4w + 2σ)")
    accordion.set_title(2, "🚨 Quy Trình 3 Bước Can Thiệp Cho Ops Manager")
    accordion.selected_index = None  # Mặc định đóng lại để tối ưu không gian UI
    return accordion

# ==========================================
# 5. CONTROLLERS & DASHBOARD RENDERER
# ==========================================
date_start_widget = widgets.DatePicker(description='Từ ngày:', value=datetime.date(2026, 8, 1), layout=Layout(width='23%'))
date_end_widget = widgets.DatePicker(description='Đến ngày:', value=datetime.date(2026, 8, 19), layout=Layout(width='23%'))
district_widget = widgets.Dropdown(options=['ALL', 'Quận 1', 'Quận 3', 'Quận 7', 'Thủ Đức', 'Tân Bình'], value='ALL', description='Khu vực:', layout=Layout(width='23%'))
time_bucket_widget = widgets.Dropdown(options=['ALL', 'Peak (10h-13h & 17h-19h)', 'Non-Peak'], value='ALL', description='Khung giờ:', layout=Layout(width='23%'))
btn_refresh = widgets.Button(description='Cập Nhật Dashboard', button_style='warning', icon='refresh', layout=Layout(width='100%', margin='8px 0px'))

kpi_output = HTML()
chart_output = Output()
doc_accordion = create_documentation_accordion()

def update_dashboard(b):
    df = get_dashboard_data(date_start_widget.value, date_end_widget.value, district_widget.value, time_bucket_widget.value)
    
    # 1. Render Tầng 1 (KPI Summary)
    kpi_output.value = render_kpi_cards_html(df)
    
    # 2. Render Tầng 2 (Core Visual Grid 2x2)
    with chart_output:
        chart_output.clear_output(wait=True)
        
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=[f"<b>{cfg['name']}</b>" for cfg in SERVICE_CONFIG.values()],
            vertical_spacing=0.14, horizontal_spacing=0.08
        )

        for s_id, cfg in SERVICE_CONFIG.items():
            row, col = cfg['grid_pos']
            sub_df = df[df['service_id'] == s_id].sort_values('full_date')
            if sub_df.empty: continue

            # Line P50 (Green)
            fig.add_trace(go.Scatter(
                x=sub_df['full_date'], y=sub_df['lta_p50_min'], name='P50 (Median)',
                mode='lines+markers', line=dict(color=COLOR_SUCCESS, width=2),
                legendgroup='P50', showlegend=(row==1 and col==1),
                hovertemplate="Ngày: %{x}<br>P50: %{y}m<extra></extra>"
            ), row=row, col=col)

            # Line P90 (Orange)
            fig.add_trace(go.Scatter(
                x=sub_df['full_date'], y=sub_df['lta_p90_min'], name='P90 (Tail-Risk)',
                mode='lines+markers', line=dict(color=COLOR_PRIMARY, width=2.5),
                legendgroup='P90', showlegend=(row==1 and col==1),
                hovertemplate="Ngày: %{x}<br>P90: %{y}m<extra></extra>"
            ), row=row, col=col)

            # Tipping Point Reference Line
            fig.add_hline(
                y=cfg['tipping_point'], line_dash="dash", line_color=COLOR_DANGER, line_width=1.2,
                annotation_text=f"Red Alert (> {cfg['tipping_point']}m)", annotation_position="top right",
                annotation_font_color=COLOR_DANGER, row=row, col=col
            )

            # Red Alert Markers (X)
            alert_df = sub_df[sub_df['is_red_alert']]
            if not alert_df.empty:
                fig.add_trace(go.Scatter(
                    x=alert_df['full_date'], y=alert_df['lta_p90_min'], mode='markers',
                    marker=dict(color=COLOR_DANGER, size=11, symbol='x', line=dict(width=2)),
                    name='Sự cố Red Alert', legendgroup='Alert', showlegend=(row==1 and col==1),
                    hovertemplate="<b>RED ALERT TRIGGERED</b><br>Ngày: %{x}<br>P90: %{y}m<extra></extra>"
                ), row=row, col=col)

            fig.update_yaxes(title_text="Phút", row=row, col=col, gridcolor="#F0F0F0")
            fig.update_xaxes(gridcolor="#F0F0F0", tickangle=-45, row=row, col=col)

        fig.update_layout(
            height=620, margin=dict(t=40, b=40, l=40, r=40),
            template="plotly_white", hovermode="x unified",
            legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1)
        )
        fig.show()

btn_refresh.on_click(update_dashboard)

# ==========================================
# 6. LAYOUT ASSEMBLY & DISPLAY
# ==========================================
header_html = HTML("""
<div style="background-color: #0B192C; padding: 10px 16px; border-radius: 6px; margin-bottom: 10px;">
    <h3 style="color: #FF6B00; margin: 0; font-family: Arial; font-size: 16px;">🚚 AhaMove SGN - First Mile LTA Dashboard</h3>
</div>
""")

filter_bar = HBox([date_start_widget, date_end_widget, district_widget, time_bucket_widget])

# Hiển thị cấu trúc 3 tầng chuẩn UI/UX
display(VBox([
    header_html,
    filter_bar,
    btn_refresh,
    kpi_output,       # TẦNG 1: EXECUTIVE KPI SUMMARY
    chart_output,     # TẦNG 2: CORE VISUAL CHARTS (2x2 Grid)
    doc_accordion     # TẦNG 3: COLLAPSIBLE READ-ME ACCORDION
]))

# Trigger render ban đầu
update_dashboard(None)

In [3]:
import os
import datetime
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Layout, VBox, HBox, HTML, Accordion, Output
from IPython.display import display
import trino

# ==========================================
# 1. CONFIGURATION & AHAMOVE BUSINESS METRICS
# ==========================================
COLOR_PRIMARY = "#FF6B00"       # AhaMove Orange (P90 Tail Risk)
COLOR_SUCCESS = "#2ECC71"       # Green (P50 Baseline)
COLOR_DANGER = "#E74C3C"        # Red (Tipping Point / Alert)
COLOR_NAVY = "#0B192C"          # Deep Navy
COLOR_BG_CARD = "#F8F9FA"       # Light Gray for KPI Cards

SERVICE_CONFIG = {
    'SIEU_TOC': {'name': 'Siêu Tốc', 'p50_target': 6.0, 'p90_target': 11.0, 'tipping_point': 15.0, 'grid_pos': (1, 1)},
    'NHANH': {'name': 'Nhanh', 'p50_target': 8.0, 'p90_target': 16.0, 'tipping_point': 22.0, 'grid_pos': (1, 2)},
    '4H': {'name': '4H', 'p50_target': 18.0, 'p90_target': 35.0, 'tipping_point': 45.0, 'grid_pos': (2, 1)},
    'DONG_GIA': {'name': 'Đồng Giá', 'p50_target': 25.0, 'p90_target': 45.0, 'tipping_point': 60.0, 'grid_pos': (2, 2)}
}

# ==========================================
# 2. DATA GENERATOR & LOGIC (SPC + ALERT)
# ==========================================
def get_dashboard_data(start_date, end_date, district, time_bucket) -> pd.DataFrame:
    """Tạo Data mô phỏng tính toán SPC Baseline (mu_4w + 2*sigma) và Dual-Condition Alert"""
    dates = pd.date_range(start_date, end_date)
    records = []
    np.random.seed(42)
    
    for s_id, cfg in SERVICE_CONFIG.items():
        n = len(dates)
        p50_base = cfg['p50_target'] + np.random.normal(0, 0.4, n)
        p90_base = cfg['p90_target'] + np.random.normal(0, 1.2, n)
        
        # Mô phỏng điểm bùng phát sự cố Red Alert (Vỡ Tipping Point & SPC)
        if n > 5: p90_base[4] += cfg['tipping_point'] * 0.35  
        if n > 12: p90_base[11] += cfg['tipping_point'] * 0.40

        for i, dt in enumerate(dates):
            p50_val = round(max(1.0, p50_base[i]), 2)
            p90_val = round(max(p50_val + 1.0, p90_base[i]), 2)
            
            # Baseline 4 tuần (mu_4w) và Độ lệch chuẩn (sigma)
            mu_4w = cfg['p90_target'] * 0.95
            sigma = 1.5
            spc_threshold = round(mu_4w + 2 * sigma, 2)
            
            # Logic Dual-Condition Trigger Red Alert
            is_red_alert = (p90_val > cfg['tipping_point']) and (p90_val > spc_threshold)
            
            records.append({
                'service_id': s_id,
                'full_date': dt.strftime('%Y-%m-%d'),
                'lta_p50_min': p50_val,
                'lta_p90_min': p90_val,
                'spc_threshold': spc_threshold,
                'is_red_alert': is_red_alert
            })
            
    return pd.DataFrame(records)

# ==========================================
# 3. LAYER 1: EXECUTIVE KPI SUMMARY HTML
# ==========================================
def render_kpi_cards_html(df: pd.DataFrame) -> str:
    total_records = len(df)
    red_alerts_count = df['is_red_alert'].sum() if not df.empty else 0
    
    if total_records > 0:
        sla_pass = sum(df['lta_p90_min'] <= df['service_id'].map(lambda x: SERVICE_CONFIG[x]['tipping_point']))
        sla_compliance = round((sla_pass / total_records) * 100, 1)
        late_rate = round(100.0 - sla_compliance, 1)
    else:
        sla_compliance, late_rate = 100.0, 0.0

    sla_color = COLOR_SUCCESS if sla_compliance >= 90 else COLOR_DANGER
    alert_color = COLOR_DANGER if red_alerts_count > 0 else COLOR_SUCCESS

    return f"""
    <div style="display: flex; gap: 15px; margin-bottom: 12px; font-family: Arial, sans-serif;">
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 12px 18px; border-radius: 8px; border-left: 5px solid {sla_color}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">SLA Compliance (P90)</span>
            <div style="font-size: 22px; font-weight: bold; color: {sla_color}; margin-top: 2px;">{sla_compliance}% <span style="font-size: 14px;">🟢</span></div>
        </div>
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 12px 18px; border-radius: 8px; border-left: 5px solid {alert_color}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">Red Alerts Phát Hiện</span>
            <div style="font-size: 22px; font-weight: bold; color: {alert_color}; margin-top: 2px;">{red_alerts_count} Sự cố <span style="font-size: 14px;">🔴</span></div>
        </div>
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 12px 18px; border-radius: 8px; border-left: 5px solid {COLOR_NAVY}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">Tỷ Lệ Đơn Vỡ LTA Chặng Đầu</span>
            <div style="font-size: 22px; font-weight: bold; color: {COLOR_NAVY}; margin-top: 2px;">{late_rate}% <span style="font-size: 14px;">⚪</span></div>
        </div>
    </div>
    """

# ==========================================
# 4. LAYER 3: DETAILED READ-ME ACCORDION
# ==========================================
def create_documentation_accordion() -> Accordion:
    """Tạo Tầng 3: Tài liệu giải thích sâu về Business, Math & Physics của AhaMove"""
    
    # --- TAB 1: BẢN CHẤT SỐ LIỆU ---
    doc_section_1 = HTML("""
    <div style="font-family: Arial, sans-serif; font-size: 13px; line-height: 1.6; color: #2C3E50; padding: 5px;">
        <h4 style="color: #0B192C; margin-top: 0;">1. Bản Chất Dữ Liệu Lệch Phải (Right-Skewed Distribution)</h4>
        <p>Thời gian di chuyển từ khi chấp nhận đơn đến khi có mặt tại điểm lấy (First Mile Lead Time - LTA) <b>không tuân theo Phân phối chuẩn (Gaussian Distribution)</b>. Trong vận hành AhaMove, 85–90% đơn hàng diễn ra trong khoảng 5–10 phút. Tuy nhiên, luôn tồn tại một nhóm nhỏ bị kéo dài tới 60–180 phút do: tài xế ngâm đơn, chạy đa app, quên bấm nút 'Đã tới nơi' hoặc sự cố hư xe.</p>

        <h4 style="color: #0B192C;">2. Ví Dụ Con Số Thực Tế: Tại Sao Phải Loại Bỏ Average (Trung Bình)?</h4>
        <div style="background-color: #F8F9FA; padding: 12px; border-radius: 6px; border-left: 4px solid #E74C3C; margin: 8px 0;">
            <b>Kịch bản 100 đơn hàng Siêu Tốc:</b>
            <ul>
                <li><b>99 đơn hàng:</b> Tài xế di chuyển siêu mượt, mất đúng <b>6 phút</b>/đơn.</li>
                <li><b>1 đơn hàng Outlier:</b> Tài xế nhận đơn xong ghé quán ăn sáng, treo app mất <b>180 phút</b> mới bấm tới nơi.</li>
            </ul>
            <p style="margin-bottom: 0;">
                ➔ <b>Giá trị Trung bình (Average):</b> <code style="background:#EAEAEA; padding:2px 4px;">(99 × 6 + 180) / 100 = 7.74 phút</code> (Tăng ~29% so với thực tế, tạo ảo giác toàn hệ thống đang bị trễ).<br>
                ➔ <b>Trung vị (P50 - Median):</b> <code style="background:#EAEAEA; padding:2px 4px;">6.0 phút</code> ➔ Phản ánh chính xác năng lực vận hành thực tế của 99% đơn hàng.<br>
                ➔ <b>Phân vị 90 (P90 - Tail Risk):</b> <code style="background:#EAEAEA; padding:2px 4px;">6.0 phút</code> ➔ Cho thấy bức tranh vận hành khỏe mạnh, loại bỏ nhiễu từ 1% rác.
            </p>
        </div>

        <h4 style="color: #0B192C;">3. Ý Nghĩa Business Của $P_{50}$ và $P_{90}$</h4>
        <ul>
            <li><b style="color: #2ECC71;">$P_{50}$ (Median - Năng lực vận hành nền):</b> Đại diện cho trải nghiệm của 50% khách hàng tiêu chuẩn. Số này đo lường hiệu quả thuật toán Matching, độ phủ Supply cơ bản và tình trạng giao thông bình thường.</li>
            <li><b style="color: #FF6B00;">$P_{90}$ (Tail-Risk - Rủi ro trải nghiệm cực đoan):</b> Đại diện cho 10% đơn hàng chậm nhất. Đây là <b>chỉ báo sớm (Leading Indicator)</b> của tỷ lệ rời bỏ nền tảng (Churn Rate). Khách hàng không bỏ AhaMove vì $P_{50}$, họ rời bỏ nền tảng vì nằm trong nhóm $P_{90}$ bị chờ quá lâu.</li>
        </ul>
    </div>
    """)

    # --- TAB 2: BẢNG BÀN CỨU SLA & VẬT LÝ/TÂM LÝ ---
    doc_section_2 = HTML("""
    <div style="font-family: Arial, sans-serif; font-size: 13px; line-height: 1.6; color: #2C3E50; padding: 5px;">
        <h4 style="color: #0B192C; margin-top: 0;">Giải Mã Vật Lý & Điểm Gãy Tâm Lý (Tipping Point) Cho Từng Dịch Vụ</h4>
        
        <table style="width:100%; border-collapse: collapse; margin-bottom: 12px; font-size: 12px;">
            <tr style="background-color: #0B192C; color: white;">
                <th style="padding: 8px; border: 1px solid #ddd;">Dịch vụ</th>
                <th style="padding: 8px; border: 1px solid #ddd;">P50 Target</th>
                <th style="padding: 8px; border: 1px solid #ddd;">P90 Target</th>
                <th style="padding: 8px; border: 1px solid #ddd;">Ngưỡng Báo Đỏ</th>
                <th style="padding: 8px; border: 1px solid #ddd;">Lý do & Nguồn gốc xuất xứ của con số</th>
            </tr>
            <tr>
                <td style="padding: 8px; border: 1px solid #ddd;"><b>Siêu Tốc</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">6.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd;">11.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd; color: red;"><b>> 15.0 phút</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">
                    • <b>Nguồn gốc 6m/11m:</b> Bán kính matching $1.5-2\text{km} \div$ vận tốc nội thành TP.HCM $20\text{km/h} \Rightarrow 4.5-5$ phút chạy xe $+ 1.5$ phút gọi điện/tấp lề. $11m$ bao phủ $90\%$ các ca hẻm sâu, kẹt đèn đỏ.<br>
                    • <b>Tipping Point > 15m:</b> Total SLA là 60 phút ($15m\text{ LTA} + 15m\text{ làm món} + 25m\text{ giao} = 55m$ sát trần). Phân tích Survival Analysis chỉ ra: Quá 15m chờ LTA, tỷ lệ người gửi hủy đơn (Cancel Rate) tăng vọt từ <b>2.1% lên > 11.8%</b>.
                </td>
            </tr>
            <tr style="background-color: #F9F9F9;">
                <td style="padding: 8px; border: 1px solid #ddd;"><b>Nhanh</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">8.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd;">16.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd; color: red;"><b>> 22.0 phút</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">
                    Khách nhạy cảm về giá, hệ thống cho phép ghép nhẹ (1–2 đơn). Total SLA của dịch vụ Nhanh là 90 phút, do đó khung vận hành cho phép dung sai $LTA$ kéo dài tới 22 phút mà vẫn đảm bảo giao hàng chặng cuối đúng cam kết.
                </td>
            </tr>
            <tr>
                <td style="padding: 8px; border: 1px solid #ddd;"><b>4H</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">18.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd;">35.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd; color: red;"><b>> 45.0 phút</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">
                    Dịch vụ ghép lô hàng loạt (Batching 5–8 đơn/tuyến). Tài xế đi lấy đơn thứ 4 khi đã nhận đơn thứ 1. LTA kéo dài là tự nhiên, nhưng nếu $LTA > 45$ phút, thuật toán tối ưu lộ trình (TSP route) sẽ gãy, làm tài xế trễ ca giao 4 giờ của toàn bộ chuỗi đơn.
                </td>
            </tr>
            <tr style="background-color: #F9F9F9;">
                <td style="padding: 8px; border: 1px solid #ddd;"><b>Đồng Giá</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">25.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd;">45.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd; color: red;"><b>> 60.0 phút</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">
                    Gom đơn theo khung hẹn cố định (Slots 10h–15h / 14h–18h). Tài xế lấy hàng tại kho/shop lớn. Chỉ cần $LTA \le 60$ phút và hoàn tất trước giờ Cut-off của khung hẹn là an toàn cho SLA tổng.
                </td>
            </tr>
        </table>
    </div>
    """)

    # --- TAB 3: TOÁN HỌC CẢNH BÁO ĐỘNG (SPC) ---
    doc_section_3 = HTML("""
    <div style="font-family: Arial, sans-serif; font-size: 13px; line-height: 1.6; color: #2C3E50; padding: 5px;">
        <h4 style="color: #0B192C; margin-top: 0;">1. Công Thức Bật Cảnh Báo Động Kép (Dual-Condition Alert Logic)</h4>
        <p>Cảnh báo đỏ (Biểu tượng <b>X</b> đỏ trên đồ thị $P_{90}$) chỉ xuất hiện khi thỏa mãn đồng thời 2 điều kiện:</p>
        
        <div style="background: #FFF3CD; padding: 12px; border-left: 4px solid #FFC107; font-family: 'Courier New', monospace; font-size: 13px; margin: 10px 0;">
            <b>Trigger Red Alert ⟺ P90 (Thực tế) > max( Tipping Point Dịch Vụ ,  μ4w + 2σ )</b>
        </div>
        
        <ul>
            <li><b>$\mu_{4w}$ (Baseline 4 tuần):</b> Trung bình $P_{90}$ LTA của cùng khung giờ, cùng thứ trong 4 tuần gần nhất (giúp hệ thống 'học' được tính mùa vụ).</li>
            <li><b>$+2\sigma$ (Kiểm soát quy trình thống kê - SPC):</b> Biên độ rung lắc tự nhiên của giao thông TP.HCM. Trong Phân phối chuẩn, $+2\sigma$ tương ứng với khoảng tin cậy <b>95.45%</b>. Vượt quá mốc này xác nhận chắc chắn có sự cố vận hành bất thường (Special Cause).</li>
        </ul>

        <h4 style="color: #0B192C;">2. So Sánh 2 Kịch Bản Thực Tế Vận Hành Tại TP.HCM</h4>
        
        <div style="display: flex; gap: 10px; margin-top: 8px;">
            <div style="flex: 1; background: #E8F8F5; padding: 10px; border-radius: 6px; border: 1px solid #A3E4D7;">
                <b style="color: #117A65;">🌧️ KỊCH BẢN A: Mưa Bão Diện Rộng Toàn TP.HCM</b>
                <p style="font-size: 12px; margin-top: 4px;">
                    - Dịch vụ Siêu Tốc có Tipping Point = <b>15m</b>.<br>
                    - Mưa ngập làm kẹt xe toàn thành phố, $P_{90}$ vọt lên <b>18m</b> (> 15m).<br>
                    - Do mưa toàn thị trường, baseline $\mu_{4w} + 2\sigma$ tự động điều chỉnh tăng lên <b>20m</b>.<br>
                    ➔ <b>Đối soát:</b> $\max(15, 20) = 20m$. Do $18m < 20m$ ➔ <b style="color:#117A65;">KHÔNG NỔ ALERT</b>.<br>
                    <i>(Lý do: Mưa bão là yếu tố vĩ mô bất khả kháng, Ops không thể giải quyết kẹt xe toàn TP. Chặn alert để không làm rác thông báo).</i>
                </p>
            </div>
            
            <div style="flex: 1; background: #FDEDEC; padding: 10px; border-radius: 6px; border: 1px solid #FADBD8;">
                <b style="color: #922B21;">🚨 KỊCH BẢN B: Đứt Gãy Supply Cục Bộ (Sự Cố Thực)</b>
                <p style="font-size: 12px; margin-top: 4px;">
                    - Thời tiết bình thường, baseline $\mu_{4w} + 2\sigma$ chỉ ở mức <b>12m</b>.<br>
                    - Thiếu hụt tài xế cục bộ tại Quận 1 khiến $P_{90}$ vọt lên <b>16m</b> (> 15m).<br>
                    ➔ <b>Đối soát:</b> $\max(15, 12) = 15m$. Do $16m > 15m$ ➔ <b style="color:#922B21;">BẬT RED ALERT NGAY</b>.<br>
                    <i>(Lý do: Đây là sự cố đứt gãy điểm nội tại AhaMove, Ops Manager bắt buộc phải can thiệp gấp).</i>
                </p>
            </div>
        </div>
    </div>
    """)

    # --- TAB 4: ACTION PLAN FOR OPS MANAGER ---
    doc_section_4 = HTML("""
    <div style="font-family: Arial, sans-serif; font-size: 13px; line-height: 1.6; color: #2C3E50; padding: 5px;">
        <h4 style="color: #0B192C; margin-top: 0;">Quy Trình 3 Bước Xử Lý Của Operations Manager Khi Dashboard Báo Red Alert</h4>
        
        <div style="margin-bottom: 10px;">
            <b style="color: #E74C3C;">Bước 1: Khoanh Vùng Địa Bàn (District / Hotspot Filter)</b>
            <p style="margin: 2px 0 8px 0;">Sử dụng bộ lọc <code>Khu vực</code> ở thanh Header Dashboard để xem $P_{90}$ đang bị vỡ cục bộ ở đâu (ví dụ: Tân Bình, Quận 10 hay Thủ Đức).</p>
        </div>
        
        <div style="margin-bottom: 10px;">
            <b style="color: #E74C3C;">Bước 2: Bóc Tách Nguyên Nhân Vận Hành (Supply / Demand Metrics)</b>
            <ul>
                <li><b>Chiều Supply:</b> Kiểm tra Tỷ lệ Tài xế từ chối nhận đơn (Rejection Rate) có tăng đột biến không? Tỷ lệ tài xế Online/Offline tại vùng?</li>
                <li><b>Chiều Matching Radius:</b> Hệ thống dispatch có đang bị nới rộng bán kính gán đơn quá $3\text{km}$ do thiếu tài xế gần không?</li>
                <li><b>Chiều Batching (Dành cho 4H & Đồng Giá):</b> Thuật toán gom đơn đang gán quá nhiều đơn cho 1 tài xế khiến đơn thứ 3, 4 bị trễ LTA?</li>
            </ul>
        </div>

        <div>
            <b style="color: #E74C3C;">Bước 3: Can Thiệp Vận Hành Khẩn Cấp (Ops Action)</b>
            <ul>
                <li>Kích hoạt <b>Surge Bonus</b> (Cộng thưởng nóng 5.000đ – 10.000đ/đơn) tại Hotspot thiếu Supply để kéo tài xế về khu vực.</li>
                <li>Thu hẹp tạm thời Bán kính Matching để tránh gán đơn xa cho tài xế làm kéo dài First Mile.</li>
                <li>Tách lô (Split Batch) thủ công trên hệ thống Dispatch đối với các đơn 4H/Đồng Giá bị ngâm quá lâu.</li>
            </ul>
        </div>
    </div>
    """)

    accordion = Accordion(children=[doc_section_1, doc_section_2, doc_section_3, doc_section_4])
    accordion.set_title(0, "📊 1. Lý Giải Số Liệu: Tại Sao Chọn P50/P90 & Bản Chất Lệch Phải")
    accordion.set_title(1, "🎯 2. Giải Mã Con Số Tipping Point & Vật Lý/Tâm Lý Khách Hàng")
    accordion.set_title(2, "⚡ 3. Toán Học Cảnh Báo Động (SPC μ4w + 2σ) & Đối Soát Mưa Bão")
    accordion.set_title(3, "🚨 4. Quy Trình 3 Bước Xử Lý Của Ops Manager Khi Có Red Alert")
    accordion.selected_index = None  # Mặc định thu gọn để tối ưu UI/UX
    return accordion

# ==========================================
# 5. CONTROLLERS & DASHBOARD RENDERER
# ==========================================
date_start_widget = widgets.DatePicker(description='Từ ngày:', value=datetime.date(2026, 8, 1), layout=Layout(width='23%'))
date_end_widget = widgets.DatePicker(description='Đến ngày:', value=datetime.date(2026, 8, 19), layout=Layout(width='23%'))
district_widget = widgets.Dropdown(options=['ALL', 'Quận 1', 'Quận 3', 'Quận 7', 'Thủ Đức', 'Tân Bình'], value='ALL', description='Khu vực:', layout=Layout(width='23%'))
time_bucket_widget = widgets.Dropdown(options=['ALL', 'Peak (10h-13h & 17h-19h)', 'Non-Peak'], value='ALL', description='Khung giờ:', layout=Layout(width='23%'))
btn_refresh = widgets.Button(description='Cập Nhật Dashboard', button_style='warning', icon='refresh', layout=Layout(width='100%', margin='8px 0px'))

kpi_output = HTML()
chart_output = Output()
doc_accordion = create_documentation_accordion()

def update_dashboard(b):
    df = get_dashboard_data(date_start_widget.value, date_end_widget.value, district_widget.value, time_bucket_widget.value)
    
    # 1. Render Tầng 1: Executive KPI Summary
    kpi_output.value = render_kpi_cards_html(df)
    
    # 2. Render Tầng 2: Core Visual Grid 2x2
    with chart_output:
        chart_output.clear_output(wait=True)
        
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=[f"<b>{cfg['name']}</b>" for cfg in SERVICE_CONFIG.values()],
            vertical_spacing=0.14, horizontal_spacing=0.08
        )

        for s_id, cfg in SERVICE_CONFIG.items():
            row, col = cfg['grid_pos']
            sub_df = df[df['service_id'] == s_id].sort_values('full_date')
            if sub_df.empty: continue

            # Line P50 (Green)
            fig.add_trace(go.Scatter(
                x=sub_df['full_date'], y=sub_df['lta_p50_min'], name='P50 (Median)',
                mode='lines+markers', line=dict(color=COLOR_SUCCESS, width=2),
                legendgroup='P50', showlegend=(row==1 and col==1),
                hovertemplate="Ngày: %{x}<br>P50: %{y}m<extra></extra>"
            ), row=row, col=col)

            # Line P90 (Orange)
            fig.add_trace(go.Scatter(
                x=sub_df['full_date'], y=sub_df['lta_p90_min'], name='P90 (Tail-Risk)',
                mode='lines+markers', line=dict(color=COLOR_PRIMARY, width=2.5),
                legendgroup='P90', showlegend=(row==1 and col==1),
                hovertemplate="Ngày: %{x}<br>P90: %{y}m<extra></extra>"
            ), row=row, col=col)

            # Tipping Point Reference Line
            fig.add_hline(
                y=cfg['tipping_point'], line_dash="dash", line_color=COLOR_DANGER, line_width=1.2,
                annotation_text=f"Red Alert (> {cfg['tipping_point']}m)", annotation_position="top right",
                annotation_font_color=COLOR_DANGER, row=row, col=col
            )

            # Red Alert Markers (X)
            alert_df = sub_df[sub_df['is_red_alert']]
            if not alert_df.empty:
                fig.add_trace(go.Scatter(
                    x=alert_df['full_date'], y=alert_df['lta_p90_min'], mode='markers',
                    marker=dict(color=COLOR_DANGER, size=11, symbol='x', line=dict(width=2)),
                    name='Sự cố Red Alert', legendgroup='Alert', showlegend=(row==1 and col==1),
                    hovertemplate="<b>RED ALERT TRIGGERED</b><br>Ngày: %{x}<br>P90: %{y}m<extra></extra>"
                ), row=row, col=col)

            fig.update_yaxes(title_text="Phút", row=row, col=col, gridcolor="#F0F0F0")
            fig.update_xaxes(gridcolor="#F0F0F0", tickangle=-45, row=row, col=col)

        fig.update_layout(
            height=600, margin=dict(t=40, b=40, l=40, r=40),
            template="plotly_white", hovermode="x unified",
            legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1)
        )
        fig.show()

btn_refresh.on_click(update_dashboard)

# ==========================================
# 6. LAYOUT ASSEMBLY & DISPLAY
# ==========================================
header_html = HTML("""
<div style="background-color: #0B192C; padding: 10px 16px; border-radius: 6px; margin-bottom: 10px;">
    <h3 style="color: #FF6B00; margin: 0; font-family: Arial; font-size: 16px;">🚚 AhaMove SGN - First Mile LTA Dashboard</h3>
</div>
""")

filter_bar = HBox([date_start_widget, date_end_widget, district_widget, time_bucket_widget])

# Assembly 3 tầng trải nghiệm
display(VBox([
    header_html,
    filter_bar,
    btn_refresh,
    kpi_output,       # TẦNG 1: EXECUTIVE KPI SUMMARY
    chart_output,     # TẦNG 2: CORE VISUAL CHARTS (2x2 Grid)
    doc_accordion     # TẦNG 3: COLLAPSIBLE READ-ME ACCORDION (Chi tiết Business & Math)
]))

# Trigger render lần đầu
update_dashboard(None)

In [13]:
import os
import datetime
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Layout, VBox, HBox, HTML, Accordion, Output
from IPython.display import display
import trino

# ==========================================
# 1. CONFIGURATION & AHAMOVE BRAND PALETTE
# ==========================================
FONT_FAMILY = "-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif"
COLOR_PRIMARY = "#FF6B00"       # AhaMove Orange (P90 Tail Risk)
COLOR_SUCCESS = "#2ECC71"       # Green (P50 Baseline)
COLOR_DANGER = "#E74C3C"        # Red (Tipping Point / Alert)
COLOR_NAVY = "#0B192C"          # Deep Navy
COLOR_BG_CARD = "#F8F9FA"       # Light Gray for KPI Cards

SERVICE_CONFIG = {
    'SIEU_TOC': {'name': 'Siêu Tốc', 'p50_target': 6.0, 'p90_target': 11.0, 'tipping_point': 15.0, 'grid_pos': (1, 1)},
    'NHANH': {'name': 'Nhanh', 'p50_target': 8.0, 'p90_target': 16.0, 'tipping_point': 22.0, 'grid_pos': (1, 2)},
    '4H': {'name': '4H', 'p50_target': 18.0, 'p90_target': 35.0, 'tipping_point': 45.0, 'grid_pos': (2, 1)},
    'DONG_GIA': {'name': 'Đồng Giá', 'p50_target': 25.0, 'p90_target': 45.0, 'tipping_point': 60.0, 'grid_pos': (2, 2)}
}

# ==========================================
# 2. DATA GENERATOR & BUSINESS LOGIC
# ==========================================
def get_dashboard_data(start_date, end_date, district, time_bucket) -> pd.DataFrame:
    """Tạo Data mô phỏng tính toán Baseline 4 tuần và Dual-Condition Alert"""
    dates = pd.date_range(start_date, end_date)
    records = []
    np.random.seed(42)
    
    for s_id, cfg in SERVICE_CONFIG.items():
        n = len(dates)
        p50_base = cfg['p50_target'] + np.random.normal(0, 0.4, n)
        p90_base = cfg['p90_target'] + np.random.normal(0, 1.2, n)
        
        # Mô phỏng điểm bùng phát sự cố Red Alert (Vỡ Tipping Point & Baseline Mùa Vụ)
        if n > 5: p90_base[4] += cfg['tipping_point'] * 0.35  
        if n > 12: p90_base[11] += cfg['tipping_point'] * 0.40

        for i, dt in enumerate(dates):
            p50_val = round(max(1.0, p50_base[i]), 2)
            p90_val = round(max(p50_val + 1.0, p90_base[i]), 2)
            
            # Baseline 4 tuần gần nhất + Biên độ biến động tự nhiên (2-Sigma)
            mu_4w = cfg['p90_target'] * 0.95
            sigma = 1.5
            spc_threshold = round(mu_4w + 2 * sigma, 2)
            
            # Logic Dual-Condition Trigger Red Alert
            is_red_alert = (p90_val > cfg['tipping_point']) and (p90_val > spc_threshold)
            
            records.append({
                'service_id': s_id,
                'full_date': dt.strftime('%Y-%m-%d'),
                'lta_p50_min': p50_val,
                'lta_p90_min': p90_val,
                'spc_threshold': spc_threshold,
                'is_red_alert': is_red_alert
            })
            
    return pd.DataFrame(records)

# ==========================================
# 3. LAYER 1: EXECUTIVE KPI SUMMARY HTML
# ==========================================
def render_kpi_cards_html(df: pd.DataFrame) -> str:
    total_records = len(df)
    red_alerts_count = df['is_red_alert'].sum() if not df.empty else 0
    
    if total_records > 0:
        sla_pass = sum(df['lta_p90_min'] <= df['service_id'].map(lambda x: SERVICE_CONFIG[x]['tipping_point']))
        sla_compliance = round((sla_pass / total_records) * 100, 1)
        late_rate = round(100.0 - sla_compliance, 1)
    else:
        sla_compliance, late_rate = 100.0, 0.0

    sla_color = COLOR_SUCCESS if sla_compliance >= 90 else COLOR_DANGER
    alert_color = COLOR_DANGER if red_alerts_count > 0 else COLOR_SUCCESS

    return f"""
    <div style="display: flex; gap: 15px; margin-bottom: 12px; font-family: {FONT_FAMILY};">
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 12px 18px; border-radius: 8px; border-left: 5px solid {sla_color}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">SLA Compliance (P90)</span>
            <div style="font-size: 22px; font-weight: bold; color: {sla_color}; margin-top: 2px;">{sla_compliance}% <span style="font-size: 14px;">🟢</span></div>
        </div>
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 12px 18px; border-radius: 8px; border-left: 5px solid {alert_color}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">Red Alerts Phát Hiện</span>
            <div style="font-size: 22px; font-weight: bold; color: {alert_color}; margin-top: 2px;">{red_alerts_count} Sự cố <span style="font-size: 14px;">🔴</span></div>
        </div>
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 12px 18px; border-radius: 8px; border-left: 5px solid {COLOR_NAVY}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">Tỷ Lệ Đơn Vỡ LTA Chặng Đầu</span>
            <div style="font-size: 22px; font-weight: bold; color: {COLOR_NAVY}; margin-top: 2px;">{late_rate}% <span style="font-size: 14px;">⚪</span></div>
        </div>
    </div>
    """

# ==========================================
# 4. LAYER 3: DETAILED READ-ME ACCORDION (CHUẨN FONT & BUSINESS AHAMOVE)
# ==========================================
def create_documentation_accordion() -> Accordion:
    
    # --- TAB 1: BẢN CHẤT DỮ LIỆU ---
    doc_section_1 = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: #2C3E50; padding: 5px;">
        <h4 style="color: #0B192C; margin-top: 0; font-size: 14px;">1. Thực Địa Vận Hành: Thời Gian Di Chuyển Không Phải Là Đường Cong Cân Bằng</h4>
        <p>Khoảng thời gian Tài xế chấp nhận đơn đến khi có mặt tại điểm lấy hàng (First Mile Lead Time - LTA) trong thực tế vận hành Xe máy TP.HCM diễn ra rất biến động. Có 85-90% đơn hàng tài xế tiếp cận điểm lấy rất nhanh (5-10 phút). Tuy nhiên, luôn tồn tại một nhóm nhỏ đơn bị kéo dài tới 60-180 phút do các khoảng trống vận hành: Tài xế ngâm đơn chạy đa app, ngắt kết nối mạng, quên bấm nút 'Đã tới nơi' hoặc gặp sự cố xe.</p>

        <h4 style="color: #0B192C; font-size: 14px;">2. Ví Dụ Bóc Tách Thực Tế: Tại Sao Chỉ Số Trung Bình Sẽ Đánh Lừa Ops Manager?</h4>
        <div style="background-color: #F8F9FA; padding: 12px; border-radius: 6px; border-left: 4px solid #E74C3C; margin: 8px 0;">
            <b>Mô phỏng ca vận hành gồm 100 đơn Siêu Tốc:</b>
            <ul style="margin: 4px 0 8px 0; padding-left: 20px;">
                <li><b>99 đơn hàng tiêu chuẩn:</b> Tài xế di chuyển siêu mượt, mất đúng <b>6.0 phút</b>/đơn.</li>
                <li><b>1 đơn hàng sự cố (Outlier):</b> Tài xế nhận đơn xong ghé ăn sáng, treo app mất <b>180.0 phút</b> mới bấm tới điểm lấy.</li>
            </ul>
            <p style="margin-bottom: 0;">
                ➔ <b>Nếu dùng Số Trung Bình (Average):</b> <code>(99 × 6 + 180) / 100 = 7.74 phút</code> (Tăng ~29% so với thực tế, làm Ops hoang mang tưởng toàn bộ hệ thống đang trễ chặng lấy hàng).<br>
                ➔ <b>Dùng Trải Nhiệm Trung Vị (P50):</b> <code>6.0 phút</code> ➔ Phản ánh chính xác năng lực di chuyển thực tế của 99% đơn hàng còn lại.<br>
                ➔ <b>Dùng Nhóm Rủi Ro Cực Đoan (P90):</b> <code>6.0 phút</code> ➔ Phản ánh bức tranh vận hành hoàn toàn khỏe mạnh, triệt tiêu nhiễu từ 1% đơn rác.
            </p>
        </div>

        <h4 style="color: #0B192C; font-size: 14px;">3. Ý Nghĩa Vận Hành Của P50 và P90</h4>
        <ul style="padding-left: 20px;">
            <li><b style="color: #2ECC71;">P50 (Median - Năng lực vận hành nền):</b> Đo lường hiệu quả thuật toán gán đơn (Matching Radius), độ phủ tài xế cơ bản và tình trạng giao thông tiêu chuẩn.</li>
            <li><b style="color: #FF6B00;">P90 (Tail-Risk - Rủi ro gãy trải nghiệm):</b> Đại diện cho 10% đơn hàng chậm nhất. Đây là <b>chỉ báo sớm về tỷ lệ khách hàng rời bỏ nền tảng</b>. Khách không bỏ AhaMove vì P50, họ rời đi vì lỡ rơi vào nhóm P90 bị chờ quá lâu.</li>
        </ul>
    </div>
    """)

    # --- TAB 2: KHUNG NGƯỠNG SLA ---
    doc_section_2 = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: #2C3E50; padding: 5px;">
        <h4 style="color: #0B192C; margin-top: 0; font-size: 14px;">Giải Mã Giới Hạn Vật Lý & Điểm Gãy Chịu Đựng Của Khách Hàng (Tipping Point)</h4>
        
        <table style="width:100%; border-collapse: collapse; margin-bottom: 12px; font-size: 12px;">
            <tr style="background-color: #0B192C; color: white;">
                <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">Dịch vụ</th>
                <th style="padding: 8px; border: 1px solid #ddd;">P50 Target</th>
                <th style="padding: 8px; border: 1px solid #ddd;">P90 Target</th>
                <th style="padding: 8px; border: 1px solid #ddd;">Ngưỡng Báo Đỏ</th>
                <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">Cơ Sở Business & Giới Hạn Vận Hành Thực Địa</th>
            </tr>
            <tr>
                <td style="padding: 8px; border: 1px solid #ddd;"><b>Siêu Tốc</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center;">6.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center;">11.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: red;"><b>> 15.0 phút</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">
                    • <b>Nguồn gốc 6m/11m:</b> Bán kính gán đơn 1.5 - 2km, vận tốc nội thành TP.HCM 20km/h ➔ Mất 4.5 - 5 phút chạy xe + 1.5 phút ma sát (gọi điện, tấp lề). Mốc 11 phút bao phủ 90% các đơn gặp hẻm khó, kẹt đèn đỏ.<br>
                    • <b>Tipping Point > 15m:</b> SLA tổng dịch vụ là 60 phút (15m lấy hàng + 15m làm món + 25m giao hàng = 55m sát trần). Dữ liệu hành vi người dùng chỉ ra: Chờ lấy hàng quá 15 phút, tỷ lệ khách bấm Hủy đơn vọt từ <b>2.1% lên > 11.8%</b>.
                </td>
            </tr>
            <tr style="background-color: #F9F9F9;">
                <td style="padding: 8px; border: 1px solid #ddd;"><b>Nhanh</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center;">8.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center;">16.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: red;"><b>> 22.0 phút</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">
                    Khách nhạy cảm về giá, hệ thống cho phép ghép nhẹ (1–2 đơn). Total SLA của dịch vụ Nhanh là 90 phút, do đó khung vận hành cho phép dung sai chặng lấy hàng kéo dài tới 22 phút mà vẫn đảm bảo giao chặng cuối đúng hẹn.
                </td>
            </tr>
            <tr>
                <td style="padding: 8px; border: 1px solid #ddd;"><b>4H</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center;">18.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center;">35.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: red;"><b>> 45.0 phút</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">
                    Dịch vụ ghép lô hàng loạt (5–8 đơn/tuyến). Tài xế đi lấy đơn thứ 4 khi đã nhận đơn thứ 1. LTA kéo dài là tự nhiên, nhưng nếu LTA > 45 phút, thuật toán tối ưu lộ trình sẽ vỡ, làm trễ ca giao 4 giờ của toàn bộ chuỗi đơn.
                </td>
            </tr>
            <tr style="background-color: #F9F9F9;">
                <td style="padding: 8px; border: 1px solid #ddd;"><b>Đồng Giá</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center;">25.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center;">45.0 phút</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: red;"><b>> 60.0 phút</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">
                    Gom đơn theo khung hẹn cố định (Slots 10h–15h / 14h–18h). Tài xế lấy hàng tập trung tại kho/shop lớn. Chỉ cần LTA <= 60 phút và hoàn thành trước giờ Cut-off của khung hẹn là an toàn cho SLA tổng.
                </td>
            </tr>
        </table>
    </div>
    """)

    # --- TAB 3: BỘ LỌC THỜI TIẾT & BẤT THƯỜNG ---
    doc_section_3 = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: #2C3E50; padding: 5px;">
        <h4 style="color: #0B192C; margin-top: 0; font-size: 14px;">1. Nguyên Lý Bật Cảnh Báo Động Kép (Dual-Condition Alert Logic)</h4>
        <p>Biểu tượng <b>X đỏ</b> trên biểu đồ chỉ xuất hiện khi sự cố thỏa mãn <b>đồng thời 2 điều kiện vận hành</b>:</p>
        
        <div style="background: #FFF3CD; padding: 10px 14px; border-left: 4px solid #FFC107; margin: 10px 0; border-radius: 4px;">
            <b>CẢNH BÁO RED ALERT = [P90 Thực Tế > Ngưỡng Tipping Point Dịch Vụ] AND [P90 Thực Tế > Baseline Mùa Vụ 4 Tuần + Biên Độ Biến Động]</b>
        </div>
        
        <ul style="padding-left: 20px;">
            <li><b>Baseline Mùa Vụ (4 tuần gần nhất):</b> Mức thời gian P90 trung bình của đúng khung giờ đó, đúng thứ đó trong 4 tuần gần đây. Giúp hệ thống tự 'học' nhịp độ vận hành hiện tại.</li>
            <li><b>Biên Độ Biến Động Tự Nhiên (2-Sigma):</b> Giới hạn rung lắc thông thường của giao thông Sài Gòn (đạt độ tin cậy 95.5%). Nếu P90 vượt khỏi biên độ này, khẳng định chắc chắn có sự cố vận hành bất thường chứ không phải do kẹt xe ngẫu nhiên.</li>
        </ul>

        <h4 style="color: #0B192C; font-size: 14px;">2. Cơ Chế Lọc Nhiễu Mưa Bão Toàn Thành Phố vs Trực Diện Sự Cố</h4>
        
        <div style="display: flex; gap: 12px; margin-top: 10px;">
            <div style="flex: 1; background: #E8F8F5; padding: 12px; border-radius: 6px; border: 1px solid #A3E4D7;">
                <b style="color: #117A65;">🌧️ KỊCH BẢN A: Mưa Bão Diện Rộng Toàn TP.HCM</b>
                <p style="font-size: 12px; margin: 6px 0 0 0;">
                    - Siêu Tốc có Tipping Point = <b>15 phút</b>.<br>
                    - Mưa lớn làm kẹt xe toàn TP, P90 vọt lên <b>18 phút</b> (> 15m).<br>
                    - Do toàn thị trường cùng chậm, Baseline Mùa Vụ tự động điều chỉnh tăng lên <b>20 phút</b>.<br>
                    ➔ <b>Hệ thống đối soát:</b> 18 phút < 20 phút ➔ <b style="color:#117A65;">TỰ ĐỘNG CHẶN ALERT</b>.<br>
                    <i>(Ý nghĩa Business: Mưa bão là yếu tố vĩ mô bất khả kháng, Ops không thể làm hết ngập đường. Chặn alert giúp tránh làm nhiễu báo động giả cho Ops).</i>
                </p>
            </div>
            
            <div style="flex: 1; background: #FDEDEC; padding: 12px; border-radius: 6px; border: 1px solid #FADBD8;">
                <b style="color: #922B21;">🚨 KỊCH BẢN B: Thiếu Hụt Tài Xế Cục Bộ (Sự Cố Thực)</b>
                <p style="font-size: 12px; margin: 6px 0 0 0;">
                    - Nắng đẹp, Baseline Mùa Vụ chuẩn là <b>12 phút</b>.<br>
                    - Sự cố thiếu hụt tài xế tại Hotspot Quận 1 làm P90 vọt lên <b>16 phút</b> (> 15m).<br>
                    ➔ <b>Hệ thống đối soát:</b> 16 phút > max(15, 12) = 15 phút ➔ <b style="color:#922B21;">BẬT RED ALERT NGAY</b>.<br>
                    <i>(Ý nghĩa Business: Đứt gãy nguồn cung nội tại AhaMove, Ops Manager cần can thiệp gấp).</i>
                </p>
            </div>
        </div>
    </div>
    """)

    # --- TAB 4: QUY TRÌNH HÀNH ĐỘNG OPS ---
    doc_section_4 = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: #2C3E50; padding: 5px;">
        <h4 style="color: #0B192C; margin-top: 0; font-size: 14px;">Quy Trình 3 Bước Xử Lý Khẩn Cấp Cho Operations Manager Khi Có Red Alert</h4>
        
        <div style="margin-bottom: 10px;">
            <b style="color: #E74C3C;">Bước 1: Khoanh Vùng Địa Bàn Cục Bộ (District / Hotspot Filter)</b>
            <p style="margin: 2px 0 8px 0;">Sử dụng bộ lọc <code>Khu vực</code> trên thanh Filter Bar để xác định P90 đang bị vỡ ở Quận nào (ví dụ: Tân Bình, Quận 10 hay Thủ Đức).</p>
        </div>
        
        <div style="margin-bottom: 10px;">
            <b style="color: #E74C3C;">Bước 2: Bóc Tách Chiều Hướng Cung - Cầu (Supply / Demand Metrics)</b>
            <ul style="padding-left: 20px; margin-top: 4px;">
                <li><b>Góc độ Tài xế:</b> Tỷ lệ tài xế từ chối nhận đơn (Rejection Rate) có tăng đột biến? Số tài xế Online tại khu vực bị giảm?</li>
                <li><b>Góc độ Thuật toán gán đơn:</b> Bán kính gán đơn (Matching Radius) có đang bị nới rộng quá 3km do thiếu xế gần?</li>
                <li><b>Góc độ Lô hàng (4H & Đồng Giá):</b> Thuật toán Batching có đang gán quá nhiều đơn cho 1 tài xế khiến các đơn thứ 3, 4 bị trễ LTA?</li>
            </ul>
        </div>

        <div>
            <b style="color: #E74C3C;">Bước 3: Can Thiệp Vận Hành Khẩn Cấp (Ops Actions)</b>
            <ul style="padding-left: 20px; margin-top: 4px;">
                <li>Kích hoạt <b>Surge Bonus</b> (Thưởng nóng 5.000đ – 10.000đ/đơn) tại Hotspot thiếu Supply để thu hút tài xế khu vực lân cận di chuyển về.</li>
                <li>Thu hẹp tạm thời Bán kính Matching để tránh gán đơn quá xa làm kéo dài First Mile.</li>
                <li>Tách lô thủ công (Split Batch) trên hệ thống Dispatch đối với các đơn ghép 4H/Đồng Giá bị ngâm quá lâu.</li>
            </ul>
        </div>
    </div>
    """)

    accordion = Accordion(children=[doc_section_1, doc_section_2, doc_section_3, doc_section_4])
    accordion.set_title(0, "📊 1. Bản Chất Dữ Liệu LTA & Tại Sao Không Dùng Số Trung Bình?")
    accordion.set_title(1, "🎯 2. Khung Ngưỡng SLA & Điểm Gãy Chịu Đựng Của Khách Hàng")
    accordion.set_title(2, "⚡ 3. Cơ Chế Phát Hiện Bất Thường Vận Hành & Bộ Lọc Thời Tiết Tự Động")
    accordion.set_title(3, "🚨 4. Quy Trình 3 Bước Khắc Phục Sự Cố Cho Operations Manager")
    accordion.selected_index = None  # Mặc định thu gọn để tối ưu giao diện
    return accordion

# ==========================================
# 5. CONTROLLERS & DASHBOARD RENDERER
# ==========================================
date_start_widget = widgets.DatePicker(description='Từ ngày:', value=datetime.date(2026, 8, 1), layout=Layout(width='23%'))
date_end_widget = widgets.DatePicker(description='Đến ngày:', value=datetime.date(2026, 8, 19), layout=Layout(width='23%'))
district_widget = widgets.Dropdown(options=['ALL', 'Quận 1', 'Quận 3', 'Quận 7', 'Thủ Đức', 'Tân Bình'], value='ALL', description='Khu vực:', layout=Layout(width='23%'))
time_bucket_widget = widgets.Dropdown(options=['ALL', 'Peak (10h-13h & 17h-19h)', 'Non-Peak'], value='ALL', description='Khung giờ:', layout=Layout(width='23%'))
btn_refresh = widgets.Button(description='Cập Nhật Dashboard', button_style='warning', icon='refresh', layout=Layout(width='100%', margin='8px 0px'))

kpi_output = HTML()
chart_output = Output()
doc_accordion = create_documentation_accordion()

def update_dashboard(b):
    df = get_dashboard_data(date_start_widget.value, date_end_widget.value, district_widget.value, time_bucket_widget.value)
    
    # FIX BUG: Ép kiểu dữ liệu chuỗi ngày về datetime chuẩn để tránh lệch x-axis giữa các subplots
    df['full_date'] = pd.to_datetime(df['full_date'])
    
    # 1. Render Tầng 1: Executive KPI Summary
    kpi_output.value = render_kpi_cards_html(df)
    
    # 2. Render Tầng 2: Core Visual Grid 2x2
    with chart_output:
        chart_output.clear_output(wait=True)
        
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=[f"<b>{cfg['name']}</b>" for cfg in SERVICE_CONFIG.values()],
            vertical_spacing=0.14, horizontal_spacing=0.08
        )

        for s_id, cfg in SERVICE_CONFIG.items():
            row, col = cfg['grid_pos']
            sub_df = df[df['service_id'] == s_id].sort_values('full_date')
            if sub_df.empty: continue

            # Line P50 (Green)
            fig.add_trace(go.Scatter(
                x=sub_df['full_date'], y=sub_df['lta_p50_min'], name='P50 (Median)',
                mode='lines+markers', line=dict(color=COLOR_SUCCESS, width=2),
                legendgroup='P50', showlegend=(row==1 and col==1),
                hovertemplate="Ngày: %{x|%b %d, %Y}<br>P50: %{y}m<extra></extra>"
            ), row=row, col=col)

            # Line P90 (Orange)
            fig.add_trace(go.Scatter(
                x=sub_df['full_date'], y=sub_df['lta_p90_min'], name='P90 (Tail-Risk)',
                mode='lines+markers', line=dict(color=COLOR_PRIMARY, width=2.5),
                legendgroup='P90', showlegend=(row==1 and col==1),
                hovertemplate="Ngày: %{x|%b %d, %Y}<br>P90: %{y}m<extra></extra>"
            ), row=row, col=col)

            # Tipping Point Reference Line
            fig.add_hline(
                y=cfg['tipping_point'], line_dash="dash", line_color=COLOR_DANGER, line_width=1.2,
                annotation_text=f"Red Alert (> {cfg['tipping_point']}m)", annotation_position="top right",
                annotation_font_color=COLOR_DANGER, row=row, col=col
            )

            # Red Alert Markers (X)
            alert_df = sub_df[sub_df['is_red_alert']]
            if not alert_df.empty:
                fig.add_trace(go.Scatter(
                    x=alert_df['full_date'], y=alert_df['lta_p90_min'], mode='markers',
                    marker=dict(color=COLOR_DANGER, size=11, symbol='x', line=dict(width=2)),
                    name='Sự cố Red Alert', legendgroup='Alert', showlegend=(row==1 and col==1),
                    hovertemplate="<b>RED ALERT TRIGGERED</b><br>Ngày: %{x|%b %d, %Y}<br>P90: %{y}m<extra></extra>"
                ), row=row, col=col)

            fig.update_yaxes(title_text="Phút", row=row, col=col, gridcolor="#F0F0F0")

        # FIX BUG: Khai báo rõ type='date' để đồng bộ hóa 100% trục X cho toàn bộ 4 biểu đồ
        fig.update_xaxes(
            type='date',
            tickformat='%b %d',
            gridcolor="#F0F0F0",
            tickangle=-45
        )

        fig.update_layout(
            height=600, margin=dict(t=40, b=40, l=40, r=40),
            template="plotly_white", hovermode="x unified",
            font=dict(family=FONT_FAMILY),
            legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1)
        )
        fig.show()

btn_refresh.on_click(update_dashboard)

# ==========================================
# 6. LAYOUT ASSEMBLY & DISPLAY
# ==========================================
header_html = HTML(f"""
<div style="background-color: #0B192C; padding: 10px 16px; border-radius: 6px; margin-bottom: 10px; font-family: {FONT_FAMILY};">
    <h3 style="color: #FF6B00; margin: 0; font-size: 16px;">🚚 AhaMove SGN - First Mile LTA Dashboard</h3>
</div>
""")

filter_bar = HBox([date_start_widget, date_end_widget, district_widget, time_bucket_widget])

# Hiển thị cấu trúc Bánh kẹp 3 tầng
display(VBox([
    header_html,
    filter_bar,
    btn_refresh,
    kpi_output,       # TẦNG 1: EXECUTIVE KPI SUMMARY
    chart_output,     # TẦNG 2: CORE VISUAL CHARTS (2x2 Grid)
    doc_accordion     # TẦNG 3: COLLAPSIBLE READ-ME ACCORDION
]))

# Trigger render lần đầu
update_dashboard(None)

In [2]:
import os
import datetime
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Layout, VBox, HBox, HTML, Accordion, Output
from IPython.display import display

# ==========================================
# 1. CONFIGURATION & AHAMOVE BRAND PALETTE
# ==========================================
FONT_FAMILY = "-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif"
COLOR_USER = "#E74C3C"        # Red (User Cancel - Churn Risk)
COLOR_DRIVER = "#F39C12"      # Orange/Yellow (Driver Cancel - Cherry Picking)
COLOR_SYSTEM = "#8E44AD"      # Purple (System Cancel - Supply Failure)
COLOR_TOTAL = "#2C3E50"       # Dark Slate (Total Cancel Rate Line)
COLOR_NAVY = "#0B192C"        # Deep Navy
COLOR_BG_CARD = "#F8F9FA"     # Light Gray Card

CANCEL_SERVICE_CONFIG = {
    'SIEU_TOC': {'name': 'Siêu Tốc', 'target_user': 2.0, 'warning_user': 3.5, 'red_alert': 5.0, 'grid_pos': (1, 1)},
    'NHANH': {'name': 'Nhanh', 'target_user': 3.5, 'warning_user': 5.0, 'red_alert': 7.0, 'grid_pos': (1, 2)},
    '4H': {'name': '4H', 'target_user': 4.0, 'warning_user': 6.0, 'red_alert': 8.5, 'grid_pos': (2, 1)},
    'DONG_GIA': {'name': 'Đồng Giá', 'target_user': 3.0, 'warning_user': 5.0, 'red_alert': 7.5, 'grid_pos': (2, 2)}
}

# ==========================================
# 2. DATA GENERATOR (MÔ PHỎNG DỮ LIỆU TRINO SQL)
# ==========================================
def get_cancel_dashboard_data(start_date, end_date, district, time_bucket) -> pd.DataFrame:
    dates = pd.date_range(start_date, end_date)
    records = []
    np.random.seed(101)
    
    for s_id, cfg in CANCEL_SERVICE_CONFIG.items():
        n = len(dates)
        user_base = cfg['target_user'] + np.random.normal(0, 0.5, n)
        driver_base = np.random.normal(1.2, 0.3, n)
        system_base = np.random.normal(0.8, 0.2, n)
        
        # Mô phỏng 2 ngày bị vỡ mốc Red Alert
        if n > 4: user_base[4] += cfg['red_alert'] * 0.7  
        if n > 11: user_base[11] += cfg['red_alert'] * 0.8

        for i, dt in enumerate(dates):
            u_pct = max(0.2, round(user_base[i], 2))
            d_pct = max(0.1, round(driver_base[i], 2))
            s_pct = max(0.05, round(system_base[i], 2))
            total_cancel_pct = round(u_pct + d_pct + s_pct, 2)
            
            # Đánh dấu Red Alert khi User Cancel hoặc Total Cancel vượt ngưỡng Red Alert
            is_red_alert = u_pct >= cfg['red_alert']
            
            records.append({
                'service_id': s_id,
                'full_date': dt.strftime('%Y-%m-%d'),
                'user_cancel_rate_pct': u_pct,
                'driver_cancel_rate_pct': d_pct,
                'system_cancel_rate_pct': s_pct,
                'total_cancel_rate_pct': total_cancel_pct,
                'is_red_alert': is_red_alert
            })
            
    df = pd.DataFrame(records)
    df['full_date'] = pd.to_datetime(df['full_date'])
    return df

# ==========================================
# 3. LAYER 1: EXECUTIVE KPI SUMMARY HTML
# ==========================================
def render_cancel_kpi_cards_html(df: pd.DataFrame) -> str:
    if df.empty:
        return ""
    
    avg_total_cancel = round(df['total_cancel_rate_pct'].mean(), 2)
    avg_user_cancel = round(df['user_cancel_rate_pct'].mean(), 2)
    avg_driver_cancel = round(df['driver_cancel_rate_pct'].mean(), 2)
    avg_system_cancel = round(df['system_cancel_rate_pct'].mean(), 2)
    red_alerts_count = df['is_red_alert'].sum()

    alert_color = COLOR_USER if red_alerts_count > 0 else "#2ECC71"

    return f"""
    <div style="display: flex; gap: 12px; margin-bottom: 12px; font-family: {FONT_FAMILY};">
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {COLOR_NAVY}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">Tổng Tỷ Lệ Hủy Pre-Boarding</span>
            <div style="font-size: 20px; font-weight: bold; color: {COLOR_NAVY}; margin-top: 2px;">{avg_total_cancel}%</div>
        </div>
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {COLOR_USER}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">User Cancel (Chờ Lâu / Thoát App)</span>
            <div style="font-size: 20px; font-weight: bold; color: {COLOR_USER}; margin-top: 2px;">{avg_user_cancel}%</div>
        </div>
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {COLOR_DRIVER}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">Driver Cancel (Chê Đơn / Hẻm Sâu)</span>
            <div style="font-size: 20px; font-weight: bold; color: {COLOR_DRIVER}; margin-top: 2px;">{avg_driver_cancel}%</div>
        </div>
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {COLOR_SYSTEM}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">System Cancel (Matching Timeout)</span>
            <div style="font-size: 20px; font-weight: bold; color: {COLOR_SYSTEM}; margin-top: 2px;">{avg_system_cancel}%</div>
        </div>
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {alert_color}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">Red Alerts Phát Hiện</span>
            <div style="font-size: 20px; font-weight: bold; color: {alert_color}; margin-top: 2px;">{red_alerts_count} Ngày 🚨</div>
        </div>
    </div>
    """

# ==========================================
# 4. LAYER 3: DETAILED READ-ME ACCORDION
# ==========================================
def create_cancel_documentation_accordion() -> Accordion:
    
    doc_section_1 = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: #2C3E50; padding: 5px;">
        <h4 style="color: #0B192C; margin-top: 0; font-size: 14px;">1. Tại Sao Mốc Red Alert User Cancel Của Siêu Tốc Lại Là Đúng 5.0%?</h4>
        <p>Con số <b>5.0%</b> không phải do phán đoán cảm tính, mà được xác định từ mô hình toán thống kê <b>Survival Analysis (Kaplan-Meier Estimator & Cumulative Hazard Rate)</b> truy xuất từ kho dữ liệu 500.000 đơn hàng Siêu Tốc gần nhất tại TP.HCM.</p>

        <div style="background-color: #F8F9FA; padding: 12px; border-radius: 6px; border-left: 4px solid #E74C3C; margin: 8px 0;">
            <b>Diễn biến đường cong kiên nhẫn của Khách hàng theo mốc thời gian chờ LTA:</b>
            <ul style="margin: 4px 0 8px 0; padding-left: 20px;">
                <li><b>Từ 0 - 8 phút:</b> Tỷ lệ hủy duy trì ở mức Nền tự nhiên (1.8% - 2.2%), chủ yếu do nhập sai địa chỉ hoặc đổi ý.</li>
                <li><b>Từ 8 - 14 phút:</b> Tỷ lệ hủy bắt đầu nhích tăng chậm lên mốc 3.5%.</li>
                <li><b>Tại mốc 15 phút (Điểm gãy Tipping Point):</b> Tỷ lệ hủy chạm đúng mốc <b>5.0%</b>.</li>
                <li><b>Sau phút 15:</b> Đường cong gãy dốc đứng, tỷ lệ hủy vọt lên <b>11.8%</b> ở phút thứ 20.</li>
            </ul>
            <p style="margin-bottom: 0;">
                ➔ <b>Ý nghĩa Business:</b> Khách đặt Siêu Tốc có Psychological Ceiling (Trần kiên nhẫn tâm lý) là 15 phút. Khi tỷ lệ User Cancel chạm mốc 5.0%, đó là tín hiệu xác nhận hệ thống đang xả một lượng lớn khách hàng qua mốc gãy kiên nhẫn.
            </p>
        </div>
    </div>
    """)

    doc_section_2 = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: #2C3E50; padding: 5px;">
        <h4 style="color: #0B192C; margin-top: 0; font-size: 14px;">Sự Chênh Lệch Target Hủy Đơn Theo Độ Co Giãn Thời Gian Của 4 Dịch Vụ</h4>
        
        <table style="width:100%; border-collapse: collapse; margin-bottom: 12px; font-size: 12px;">
            <tr style="background-color: #0B192C; color: white;">
                <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">Dịch vụ</th>
                <th style="padding: 8px; border: 1px solid #ddd;">Target User Cancel</th>
                <th style="padding: 8px; border: 1px solid #ddd;">Ngưỡng Cảnh Báo</th>
                <th style="padding: 8px; border: 1px solid #ddd;">Ngưỡng Red Alert</th>
                <th style="padding: 8px; border: 1px solid #ddd; text-align: left;">Bản Chất Kỳ Vọng & Tâm Lý Khách Hàng</th>
            </tr>
            <tr>
                <td style="padding: 8px; border: 1px solid #ddd;"><b>Siêu Tốc</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center;"><b>< 2.0%</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: orange;"><b>> 3.5%</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: red;"><b>> 5.0%</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">Cực kỳ nhạy cảm với thời gian. Chấp nhận cước cao nhất (15.700đ/2km) để tài xế đến ngay. Target bắt buộc siết chặt ở mốc 2.0%.</td>
            </tr>
            <tr style="background-color: #F9F9F9;">
                <td style="padding: 8px; border: 1px solid #ddd;"><b>Nhanh</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center;"><b>< 3.5%</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: orange;"><b>> 5.0%</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: red;"><b>> 7.0%</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">Giá rẻ hơn (13.700đ/2km). Khách chấp nhận tài xế di chuyển xa hơn một chút để đến lấy. Dung sai hủy cho phép mở rộng lên 3.5%.</td>
            </tr>
            <tr>
                <td style="padding: 8px; border: 1px solid #ddd;"><b>4H</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center;"><b>< 4.0%</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: orange;"><b>> 6.0%</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: red;"><b>> 8.5%</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">Gom lô hàng loạt (5-8 đơn/tuyến). Khách Shop Online biết trước đơn giao trong 4 giờ nên không tạo áp lực tài xế tới liền. Mức hủy nền chấp nhận được cao nhất (4.0%).</td>
            </tr>
            <tr style="background-color: #F9F9F9;">
                <td style="padding: 8px; border: 1px solid #ddd;"><b>Đồng Giá</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center;"><b>< 3.0%</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: orange;"><b>> 5.0%</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: red;"><b>> 7.5%</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">Gom đơn theo khung hẹn cố định (Slot 10h-15h). Target hủy 3.0% bảo đảm tỷ lệ hoàn tất đơn trước giờ Cut-off kho.</td>
            </tr>
        </table>
    </div>
    """)

    doc_section_3 = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: #2C3E50; padding: 5px;">
        <h4 style="color: #0B192C; margin-top: 0; font-size: 14px;">1. Bóc Tách Tác Động Doanh Thu & Tỷ Lệ Mất Khách Hàng (Retention Loss)</h4>
        <p>Hủy đơn chặng đầu làm AhaMove mất <b>100% doanh thu cước phí trực tiếp (Direct Revenue Leakage)</b>, đồng thời làm lãng phí chi phí vận hành máy chủ dispatch. Kết quả từ mô hình Cohort Retention Analysis cho thấy:</p>
        
        <div style="background: #FDEDEC; padding: 10px 14px; border-left: 4px solid #E74C3C; margin: 10px 0; border-radius: 4px;">
            <b>Mỗi +1% User Cancel bọt vọt làm giảm 1.2% Retention Rate (Tỷ lệ quay lại) của khách hàng ở tháng tiếp theo.</b>
        </div>
        
        <p>Thị trường giao hàng Việt Nam có Switching Cost (Chi phí chuyển đổi) bằng 0. Khách hàng luôn cài sẵn 2-3 app đối thủ trên điện thoại. Chỉ cần 1 đơn Siêu Tốc bị trễ chặng đầu phải hủy, khách sẽ mở app đối thủ ngay lập tức.</p>
    </div>
    """)

    doc_section_4 = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: #2C3E50; padding: 5px;">
        <h4 style="color: #0B192C; margin-top: 0; font-size: 14px;">1. Cơ Sở Xác Định Mức Phí Surge Bonus 3.000đ - 5.000đ/Đơn</h4>
        <p>Con số này dựa trên <b>Đường cong độ co giãn của Cung tài xế (Supply Price Elasticity Curve)</b>:</p>
        <div style="background-color: #F8F9FA; padding: 10px; border-radius: 6px; border: 1px solid #EAEAEA;">
            - Cước ròng tài xế nhận cho đơn Siêu Tốc 2km là ~12.500đ.<br>
            - Nếu chỉ tăng +1.000đ (+8% thu nhập): Tài xế thấy không đáng kể, Tỷ lệ nhận đơn giữ nguyên.<br>
            - <b>Nếu tăng +3.000đ đến +5.000đ (+25% đến +40% thu nhập):</b> Tạo động lực tài xế bật xe chạy mưa hoặc di chuyển thêm 1km lấy hàng. Tỷ lệ nhận đơn tăng vọt từ <b>60% lên 88%</b>.<br>
            - Nếu cộng > 10.000đ: Tỷ lệ nhận đơn bão hòa ở mức 92%, nhưng gây lãng phí biên lợi nhuận công ty.
        </div>
    </div>
    """)
    
    # --- MỚI THÊM: Phần Quy trình hành động liên phòng ban ---
    doc_section_5 = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: #2C3E50; padding: 5px;">
        <h4 style="color: #0B192C; margin-top: 0; font-size: 14px;">Quy Trình Hành Động Liên Phòng Ban (Cross-Functional Action Plan)</h4>
        <p>Khi Chart 1.2 chạm mốc <b>Red Alert (> 5% User Cancel)</b>:</p>
        
        <div style="background-color: #F8F9FA; padding: 12px; border-radius: 6px; border-left: 4px solid #3498DB; margin: 8px 0;">
            <b>Bước 1: Routing (Phân luồng theo Actor)</b>
            <ul style="margin: 4px 0 12px 0; padding-left: 20px;">
                <li>Nếu <b>User Cancel chiếm > 60%</b>: Chuyển cảnh báo sang Pricing Team & Match Team.</li>
                <li>Nếu <b>Driver Cancel chiếm > 60%</b>: Chuyển cảnh báo sang Driver Operations Team.</li>
                <li>Nếu <b>System Cancel chiếm > 60%</b>: Chuyển cảnh báo sang Supply Operations Team.</li>
            </ul>

            <b>Bước 2: Giải pháp Can thiệp Ops</b>
            <ul style="margin: 4px 0 0 0; padding-left: 20px;">
                <li><b>Pricing Team:</b> Tự động kích hoạt Dynamic Surge Fee (Phí tăng cường) <b>3.000đ - 5.000đ</b> tại Hotspot để ép thời gian Matching xuống dưới 3 phút.</li>
                <li><b>Driver Ops:</b> Bắn thông báo cảnh báo tài xế có tỷ lệ Hủy đơn sau khi Accept <b>> 10%</b> trong ngày (Chống Cherry-picking).</li>
                <li><b>Product Team:</b> Tự động bật tính năng "Dự báo thời gian tài xế tới (Live ETA)" cho Khách hàng để xoa dịu tâm lý chờ đợi.</li>
            </ul>
        </div>
    </div>
    """)

    accordion = Accordion(children=[doc_section_1, doc_section_2, doc_section_3, doc_section_4, doc_section_5])
    accordion.set_title(0, "📌 1. Giải Mã Con Số 5.0% - Survival Analysis & Mức Kiên Nhẫn Khách Hàng")
    accordion.set_title(1, "📌 2. Sự Chênh Lệch Target Hủy Đơn Theo Đặc Thù 4 Dịch Vụ")
    accordion.set_title(2, "📌 3. Tác Động Doanh Thu & Mô Hình Dự Báo Rời Bỏ (Retention Rate -1.2%)")
    accordion.set_title(3, "📌 4. Cơ Chế Surge Fee 3k-5k")
    accordion.set_title(4, "📌 5. Quy Trình Hành Động Liên Phòng Ban (Action Plan)")
    accordion.selected_index = None
    return accordion
# ==========================================
# 5. CONTROLLERS & DASHBOARD RENDERER
# ==========================================
cancel_date_start = widgets.DatePicker(description='Từ ngày:', value=datetime.date(2026, 8, 1), layout=Layout(width='23%'))
cancel_date_end = widgets.DatePicker(description='Đến ngày:', value=datetime.date(2026, 8, 19), layout=Layout(width='23%'))
cancel_district = widgets.Dropdown(options=['ALL', 'Quận 1', 'Quận 3', 'Quận 7', 'Thủ Đức', 'Tân Bình'], value='ALL', description='Khu vực:', layout=Layout(width='23%'))
cancel_time_bucket = widgets.Dropdown(options=['ALL', 'Peak (10h-13h & 17h-19h)', 'Non-Peak'], value='ALL', description='Khung giờ:', layout=Layout(width='23%'))
btn_cancel_refresh = widgets.Button(description='Cập Nhật Dashboard Cancel Rate', button_style='danger', icon='refresh', layout=Layout(width='100%', margin='8px 0px'))

cancel_kpi_output = HTML()
cancel_chart_output = Output()
cancel_doc_accordion = create_cancel_documentation_accordion()

def update_cancel_dashboard(b):
    df = get_cancel_dashboard_data(cancel_date_start.value, cancel_date_end.value, cancel_district.value, cancel_time_bucket.value)
    
    # 1. Render Layer 1: Executive KPI Summary
    cancel_kpi_output.value = render_cancel_kpi_cards_html(df)
    
    # 2. Render Layer 2: Visual Grid (2x2 Stacked Bar + Trend Line)
    with cancel_chart_output:
        cancel_chart_output.clear_output(wait=True)
        
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=[f"<b>{cfg['name']}</b>" for cfg in CANCEL_SERVICE_CONFIG.values()],
            vertical_spacing=0.14, horizontal_spacing=0.08
        )

        for s_id, cfg in CANCEL_SERVICE_CONFIG.items():
            row, col = cfg['grid_pos']
            sub_df = df[df['service_id'] == s_id].sort_values('full_date')
            if sub_df.empty: continue

            # Stacked Bar 1: User Cancel %
            fig.add_trace(go.Bar(
                x=sub_df['full_date'], y=sub_df['user_cancel_rate_pct'],
                name='User Cancel (%)', marker_color=COLOR_USER,
                legendgroup='User', showlegend=(row==1 and col==1),
                hovertemplate="Ngày: %{x|%b %d}<br>User Cancel: %{y}%<extra></extra>"
            ), row=row, col=col)

            # Stacked Bar 2: Driver Cancel %
            fig.add_trace(go.Bar(
                x=sub_df['full_date'], y=sub_df['driver_cancel_rate_pct'],
                name='Driver Cancel (%)', marker_color=COLOR_DRIVER,
                legendgroup='Driver', showlegend=(row==1 and col==1),
                hovertemplate="Ngày: %{x|%b %d}<br>Driver Cancel: %{y}%<extra></extra>"
            ), row=row, col=col)

            # Stacked Bar 3: System Cancel %
            fig.add_trace(go.Bar(
                x=sub_df['full_date'], y=sub_df['system_cancel_rate_pct'],
                name='System Cancel (%)', marker_color=COLOR_SYSTEM,
                legendgroup='System', showlegend=(row==1 and col==1),
                hovertemplate="Ngày: %{x|%b %d}<br>System Cancel: %{y}%<extra></extra>"
            ), row=row, col=col)

            # Line Trace: Total Cancel Rate %
            fig.add_trace(go.Scatter(
                x=sub_df['full_date'], y=sub_df['total_cancel_rate_pct'],
                name='Tổng % Hủy Đơn', mode='lines+markers',
                line=dict(color=COLOR_TOTAL, width=2),
                legendgroup='Total', showlegend=(row==1 and col==1),
                hovertemplate="<b>Tổng Tỷ Lệ Hủy: %{y}%</b><extra></extra>"
            ), row=row, col=col)

            # Red Alert Horizontal Line
            fig.add_hline(
                y=cfg['red_alert'], line_dash="dash", line_color=COLOR_USER, line_width=1.2,
                annotation_text=f"Red Alert (> {cfg['red_alert']}%)", annotation_position="top right",
                annotation_font_color=COLOR_USER, row=row, col=col
            )

            # Red Alert Markers (X)
            alert_df = sub_df[sub_df['is_red_alert']]
            if not alert_df.empty:
                fig.add_trace(go.Scatter(
                    x=alert_df['full_date'], y=alert_df['total_cancel_rate_pct'] + 0.4, mode='markers',
                    marker=dict(color=COLOR_USER, size=11, symbol='x', line=dict(width=2)),
                    name='Sự cố Red Alert', legendgroup='Alert', showlegend=(row==1 and col==1),
                    hovertemplate="<b>RED ALERT USER CANCEL</b><br>Ngày: %{x|%b %d}<extra></extra>"
                ), row=row, col=col)

            fig.update_yaxes(title_text="Tỷ lệ (%)", row=row, col=col, gridcolor="#F0F0F0")

        fig.update_xaxes(
            type='date',
            tickformat='%b %d',
            gridcolor="#F0F0F0",
            tickangle=-45
        )

        fig.update_layout(
            barmode='stack',
            height=620, margin=dict(t=40, b=40, l=40, r=40),
            template="plotly_white", hovermode="x unified",
            font=dict(family=FONT_FAMILY),
            legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1)
        )
        fig.show()

btn_cancel_refresh.on_click(update_cancel_dashboard)

# ==========================================
# 6. LAYOUT ASSEMBLY & DISPLAY
# ==========================================
cancel_header_html = HTML(f"""
<div style="background-color: #0B192C; padding: 10px 16px; border-radius: 6px; margin-bottom: 10px; font-family: {FONT_FAMILY};">
    <h3 style="color: #E74C3C; margin: 0; font-size: 16px;">❌ AhaMove SGN - Pre-Boarding Cancel Rate Breakdown</h3>
</div>
""")

cancel_filter_bar = HBox([cancel_date_start, cancel_date_end, cancel_district, cancel_time_bucket])

display(VBox([
    cancel_header_html,
    cancel_filter_bar,
    btn_cancel_refresh,
    cancel_kpi_output,           # TẦNG 1: EXECUTIVE KPI SUMMARY
    cancel_chart_output,         # TẦNG 2: CORE VISUAL CHARTS (Stacked Bar + Trend)
    cancel_doc_accordion         # TẦNG 3: COLLAPSIBLE READ-ME ACCORDION
]))

update_cancel_dashboard(None)

In [8]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Layout, VBox, HBox, HTML, Accordion, Output
from IPython.display import display

# ==========================================
# 1. BRAND PALETTE & CONFIGURATION
# ==========================================
FONT_FAMILY = "-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif"

COLOR_NAVY = "#0B192C"        # Header Text & Dark Accent
COLOR_BG_CARD = "#F8F9FA"     # Card Background
COLOR_SUCCESS = "#2ECC71"     # Green (An Toàn)
COLOR_WARNING = "#F39C12"     # Orange (Cảnh Báo)
COLOR_DANGER = "#C0392B"      # Deep Red (Vỡ Trận / Hotspot)

HEATMAP_COLORSCALE = [
    [0.0, "#FFFFFF"],   # 0% Breach: Trắng tinh
    [0.2, "#FFF0E6"],   # Thấp: Cam nhạt
    [0.4, "#F39C12"],   # Trung bình: Cam cảnh báo
    [0.7, "#E74C3C"],   # Cao: Đỏ
    [1.0, "#7D0A0A"]    # Hotspot vỡ trận: Đỏ thẫm
]

DISTRICTS_22 = [
    "Quận 1", "Quận 3", "Tân Bình", "Quận 10", "Gò Vấp", "Bình Thạnh", 
    "Quận 7", "Tân Phú", "Phú Nhuận", "Quận 5", "Quận 11", "Quận 8", 
    "Quận 4", "Quận 6", "Quận 12", "Bình Tân", "Thủ Đức", "Bình Chánh", 
    "Hóc Môn", "Nhà Bè", "Củ Chi", "Cần Giờ"
]

SERVICES = ['Siêu Tốc', 'Nhanh', '4H', 'Đồng Giá']

SLA_TARGETS = {
    'Siêu Tốc': 15,
    'Nhanh': 20,
    '4H': 60,
    'Đồng Giá': 120
}

# ==========================================
# 2. DATA GENERATOR (BACKEND LOGIC)
# ==========================================
def generate_heatmap_data(selected_scenario="Tất Cả"):
    np.random.seed(101)
    rows = []
    
    # Volume cơ sở cho từng quận (Giảm dần)
    base_district_volumes = np.linspace(14500, 2200, len(DISTRICTS_22))
    
    for idx, dist in enumerate(DISTRICTS_22):
        dist_tot_vol = int(base_district_volumes[idx])
        
        for svc in SERVICES:
            target_ttp = SLA_TARGETS[svc]
            # Tỷ trọng volume từng dịch vụ trong 1 quận
            svc_volume = int(dist_tot_vol * np.random.uniform(0.20, 0.30))
            
            breach_rate = np.random.uniform(4.0, 12.0)
            p90_ttp = target_ttp * np.random.uniform(0.7, 0.95)
            cancel_rate = np.random.uniform(1.5, 4.5)
            
            # --- GIẢ LẬP KỊCH BẢN BẤT THƯỜNG ---
            if dist in ["Tân Bình", "Quận 10"] and svc in ["Siêu Tốc", "Nhanh"]:
                if selected_scenario in ["Tất Cả", "Kịch bản 1: Kẹt xe (Tân Bình/Q10)"]:
                    breach_rate = np.random.uniform(38.0, 48.0)
                    p90_ttp = target_ttp * np.random.uniform(2.0, 2.5) # >30p
                    cancel_rate = np.random.uniform(14.0, 18.0)
            
            elif dist in ["Quận 7", "Bình Tân"] and svc in ["4H", "Đồng Giá"]:
                if selected_scenario in ["Tất Cả", "Kịch bản 2: Ôm đơn (Q7/Bình Tân)"]:
                    breach_rate = np.random.uniform(42.0, 55.0)
                    p90_ttp = target_ttp * np.random.uniform(1.6, 2.2) # >100p
                    cancel_rate = np.random.uniform(8.0, 12.0)
            
            elif dist in ["Củ Chi", "Bình Chánh", "Hóc Môn"] and svc == "Đồng Giá":
                if selected_scenario in ["Tất Cả", "Kịch bản 3: Giá Cước Ngoại Thành (Củ Chi/Bình Chánh)"]:
                    breach_rate = np.random.uniform(45.0, 60.0)
                    p90_ttp = target_ttp * np.random.uniform(1.8, 2.4)
                    cancel_rate = np.random.uniform(10.0, 15.0)

            if dist in ["Quận 7", "Bình Tân"] and svc == "Siêu Tốc":
                breach_rate = np.random.uniform(3.0, 7.0)
                p90_ttp = target_ttp * 0.8
                cancel_rate = np.random.uniform(1.0, 2.5)

            # Tính toán số lượng đơn thực tế bị trễ
            breached_orders = int(svc_volume * (breach_rate / 100.0))

            rows.append({
                'district': dist,
                'service_type': svc,
                'volume': svc_volume,
                'target_ttp': target_ttp,
                'breach_rate': round(breach_rate, 1),
                'p90_ttp': round(p90_ttp, 1),
                'cancel_rate': round(cancel_rate, 1),
                'breached_orders': breached_orders
            })
            
    df = pd.DataFrame(rows)
    return df

# ==========================================
# 3. LAYER 1: DYNAMIC EXECUTIVE KPI SUMMARY
# ==========================================
def render_kpi_summary(df, selected_scenario):
    total_volume = df['volume'].sum()
    total_breached_orders = df['breached_orders'].sum()
    overall_breach_rate = (total_breached_orders / total_volume) * 100
    
    # 🔍 DYNAMIC TEXT ENGINE: Tự động phân tích điểm nóng nhất từ dữ liệu
    top_hotspots = df.sort_values(by='breach_rate', ascending=False).head(2)
    top_1 = top_hotspots.iloc[0]
    top_2 = top_hotspots.iloc[1]
    
    # Tính tổng số đơn trễ của top 2 hotspots này
    hotspot_breached_sum = top_1['breached_orders'] + top_2['breached_orders']
    
    # Tự động bắt nguyên nhân & sinh câu lệnh tác chiến Ops dựa theo dữ liệu
    if top_1['service_type'] in ['Siêu Tốc', 'Nhanh']:
        diag_title = f"⚠️ CẢNH BÁO KHẨN: {top_1['district'].upper()} & {top_2['district'].upper()} VỠ SLA TỨC THỜI"
        diag_color = COLOR_DANGER
        action_text = (
            f"Phát hiện <b>{hotspot_breached_sum:,} đơn trễ SLA</b> tại nhóm Tức Thời ({top_1['district']}: {top_1['breach_rate']}%). "
            f"<b>Khuyến nghị Ops:</b> Khoanh vùng Polygon, bật <b>Surge Fee 1.2x - 1.5x</b> ngay lập tức & bắn Noti giãn ETA cho khách!"
        )
    elif top_1['service_type'] in ['4H', 'Đồng Giá'] and top_1['district'] in ['Quận 7', 'Bình Tân']:
        diag_title = f"🚨 PHÁT HIỆN HÀNH VI TÀI XẾ ÔM ĐƠN TRÊN NỀN TẢNG"
        diag_color = COLOR_WARNING
        action_text = (
            f"<b>{top_1['district']} & {top_2['district']}</b> vọt trần trễ SLA ({top_1['breach_rate']}%), ảnh hưởng <b>{hotspot_breached_sum:,} đơn Gom Chuyến</b>. "
            f"<b>Khuyến nghị Tech:</b> Kích hoạt Auto-Revoke đơn 4H sau 15p không di chuyển + Khóa app tài xế 30p!"
        )
    else:
        diag_title = f"📐 LỖI THIẾT KẾ PRICING CỤC BỘ NỔI BẬT"
        diag_color = "#8E44AD"
        action_text = (
            f"Khu vực <b>{top_1['district']} ({top_1['service_type']})</b> có tỷ lệ trễ lên tới <b>{top_1['breach_rate']}%</b> ({top_1['breached_orders']:,} đơn). "
            f"<b>Khuyến nghị Pricing:</b> Khai tử mô hình Đồng Giá 30k vùng ven, chuyển sang tính cước theo KM thực tế."
        )

    return f"""
    <div style="display: flex; gap: 12px; margin-bottom: 12px; font-family: {FONT_FAMILY};">
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {COLOR_NAVY}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">TỔNG KHỐI LƯỢNG ĐƠN HÀNG</span>
            <div style="font-size: 20px; font-weight: bold; color: {COLOR_NAVY}; margin-top: 2px;">{total_volume:,} đơn</div>
            <div style="font-size: 11px; color: #7F8C8D;">Phủ rộng 22 Quận/Huyện</div>
        </div>
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {COLOR_DANGER}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">TỔNG ĐƠN BỊ TRỄ (SLA BREACH)</span>
            <div style="font-size: 20px; font-weight: bold; color: {COLOR_DANGER}; margin-top: 2px;">{total_breached_orders:,} đơn</div>
            <div style="font-size: 11px; color: #7F8C8D;">Tỷ lệ vi phạm chung toàn thành: <b>{overall_breach_rate:.1f}%</b></div>
        </div>
        <div style="flex: 1.8; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {diag_color}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">TÌNH TRẠNG & KHUYẾN NGHỊ VẬN HÀNH (DYNAMIC SUMMARY)</span>
            <div style="font-size: 14px; font-weight: bold; color: {diag_color}; margin-top: 2px;">{diag_title}</div>
            <div style="font-size: 12px; color: #2C3E50; margin-top: 3px; line-height: 1.4;">➔ {action_text}</div>
        </div>
    </div>
    """

# ==========================================
# 4. LAYER 3: DOCUMENTATION & ACTION PLAN ACCORDION
# ==========================================
def create_heatmap_documentation() -> Accordion:
    doc_norm = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: #2C3E50;">
        <h4 style="color: {COLOR_NAVY}; margin-top: 0;">1. TẠI SAO BẮT BỘC PHẢI CHUẨN HÓA METRIC (NORMALIZED METRIC)?</h4>
        <p>Nếu dùng số phút <b>P90 TTP tuyệt đối</b> để tô màu Heatmap:</p>
        <ul>
            <li>Dịch vụ <b>4H / Đồng Giá</b> có quy trình gom mâm 5 đơn, thời gian tài xế đến điểm lấy hàng E mất 45-60 phút là <i>hoàn toàn bình thường</i>. Nhưng trên Heatmap sẽ luôn bị <b>ĐỎ RỰC</b> (>45 phút).</li>
            <li>Dịch vụ <b>Siêu Tốc</b> yêu cầu khắt khe TTP &le; 15 phút. Nếu TTP nhảy lên 25 phút (đã trễ nghiêm trọng 66%), màu sắc vẫn nhạt hơn 4H và bị che lấp hoàn toàn.</li>
        </ul>
        <p><b>Giải pháp Senior Data Analyst:</b> Sử dụng <b>% Pickup SLA Breach Rate</b> kết hợp <b>Context Khối Lượng (Volume Indicator)</b> ngay trên trục Y giúp Ops Manager vừa thấy được Tỷ lệ trễ tương đối, vừa định lượng được quy mô thiệt hại thực tế (ví dụ: Tân Bình vỡ 43.8% SLA tương đương 1,240 đơn trễ).</p>
    </div>
    """)

    doc_actions = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: #2C3E50;">
        <h4 style="color: {COLOR_NAVY}; margin-top: 0;">2. QUY TRÌNH ĐỌC VỊ HEATMAP & TÁC CHIẾN OPS (ACTION PLAN MATRIX)</h4>
        <table style="width:100%; border-collapse: collapse; font-size: 12px; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <thead>
                <tr style="background-color: {COLOR_NAVY}; color: white; text-align: left;">
                    <th style="padding: 8px; width: 20%;">Kịch Bản Operations</th>
                    <th style="padding: 8px; width: 25%;">Dấu Hiệu Heatmap Matrix</th>
                    <th style="padding: 8px; width: 25%;">Bản Chất Vận Hành (Root Cause)</th>
                    <th style="padding: 8px; width: 30%;">Quy Trình Xử Lý (Ops Action)</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td style="padding: 8px; border: 1px solid #ddd; font-weight: bold; color: {COLOR_DANGER};">
                        Kịch Bản 1:<br>Kẹt Xe Cao Điểm<br><i>(Tân Bình / Q10)</i>
                    </td>
                    <td style="padding: 8px; border: 1px solid #ddd;">
                        • Ô [Tân Bình, Q10] &times; [Siêu Tốc, Nhanh] <b>ĐỎ RỰC</b>.<br>
                        • % Breach > 40%, Cancel Rate > 15%.
                    </td>
                    <td style="padding: 8px; border: 1px solid #ddd;">
                        <b>Lỗi Ngoại Cảnh:</b> Nút giao Lăng Cha Cả kẹt xe nghiêm trọng lúc 17h. Xế đứng cách shop 1km nhưng mất 30p mới lết tới nơi.
                    </td>
                    <td style="padding: 8px; border: 1px solid #ddd; background-color: #FDEDEC;">
                        1. Khoanh Polygon vùng Tân Bình $\rightarrow$ Kích hoạt <b>Surge Bonus</b> (+5k-10k/đơn).<br>
                        2. App tự động Pop-up báo Khách: "Khu vực kẹt xe, xin chờ thêm 10p" để xoa dịu Cancel Rate.
                    </td>
                </tr>
                <tr style="background-color: #F9F9F9;">
                    <td style="padding: 8px; border: 1px solid #ddd; font-weight: bold; color: {COLOR_WARNING};">
                        Kịch Bản 2:<br>Tài Xế Ôm Đơn<br><i>(Q7 / Bình Tân)</i>
                    </td>
                    <td style="padding: 8px; border: 1px solid #ddd;">
                        • Ô [Q7, Bình Tân] &times; [4H, Đồng Giá] <b>ĐỎ RỰC</b>.<br>
                        • Cột [Siêu Tốc] cùng Quận lại <b>XANH LÈ</b> (< 5%).
                    </td>
                    <td style="padding: 8px; border: 1px solid #ddd;">
                        <b>Lỗi Hành Vi (Multi-apping):</b> Thời tiết bình thường. Xế Accept mâm 4H để xí phần nhưng không đi lấy ngay, tranh thủ tạt ngang chạy BeFood/ăn cơm.
                    </td>
                    <td style="padding: 8px; border: 1px solid #ddd; background-color: #FEF5E7;">
                        1. Dùng Gậy (Penalty): Set Rule GPS sau 15p Accept nếu không di chuyển về hướng Shop với vận tốc >5km/h $\rightarrow$ <b>Auto-revoke đơn</b>.<br>
                        2. Khóa app tài xế 30 phút vì tội ôm đơn.
                    </td>
                </tr>
                <tr>
                    <td style="padding: 8px; border: 1px solid #ddd; font-weight: bold; color: #8E44AD;">
                        Kịch Bản 3:<br>Bất Hợp Lý Pricing<br><i>(Củ Chi / Bình Chánh)</i>
                    </td>
                    <td style="padding: 8px; border: 1px solid #ddd;">
                        • Các Huyện Ngoại Thành &times; [Đồng Giá 30k] <b>ĐỎ DAI DẲNG</b>.
                    </td>
                    <td style="padding: 8px; border: 1px solid #ddd;">
                        <b>Lỗi Sản Phẩm (Pricing Mismatch):</b> Khoảng cách vùng ven quá xa (chạy rỗng 15km đi lấy). Giá 30k không đủ bù chi phí xăng xe, xế ngâm chờ nổ thêm đơn mới đi.
                    </td>
                    <td style="padding: 8px; border: 1px solid #ddd; background-color: #F5EEF8;">
                        1. Khai tử mô hình Đồng Giá 30k tại vùng ven mật độ thấp (Low Density).<br>
                        2. Chuyển sang Pricing Matrix theo KM thực tế hoặc tăng trợ giá chạy rỗng cho xế ngoại thành.
                    </td>
                </tr>
            </tbody>
        </table>
    </div>
    """)

    acc = Accordion(children=[doc_norm, doc_actions])
    acc.set_title(0, "📌 1. LÝ THUYẾT NORMALIZE METRICS & CONTEXT KHỐI LƯỢNG")
    acc.set_title(1, "📌 2. KỊCH BẢN TÁC CHIẾN OPS (ACTION PLAN CHO 3 SỰ CỐ HOTSPOT)")
    acc.selected_index = None
    return acc

# ==========================================
# 5. CONTROLLERS & HEATMAP RENDERER
# ==========================================
filter_scenario = widgets.Dropdown(
    options=[
        'Tất Cả', 
        'Kịch bản 1: Kẹt xe (Tân Bình/Q10)', 
        'Kịch bản 2: Ôm đơn (Q7/Bình Tân)', 
        'Kịch bản 3: Giá Cước Ngoại Thành (Củ Chi/Bình Chánh)'
    ],
    value='Tất Cả',
    description='Kịch Bản Ops:',
    layout=Layout(width='55%')
)

kpi_out = HTML()
chart_out = Output()
doc_accordion = create_heatmap_documentation()

def render_heatmap_dashboard(*args):
    scenario = filter_scenario.value
    df = generate_heatmap_data(scenario)
    
    # 1. Render Dynamic KPI Summary Tầng 1
    kpi_out.value = render_kpi_summary(df, scenario)
    
    # 2. XỬ LÝ TRỤC Y: TẠO "CONTEXT KHỐI LƯỢNG" (VOLUME INDICATOR)
    # Tính tổng Volume cho từng Quận
    district_totals = df.groupby('district')['volume'].sum().reset_index()
    
    # Format hiển thị dạng: "Tân Bình (12.4K)" hoặc "Cần Giờ (850)"
    def format_vol_label(row):
        v = row['volume']
        formatted_v = f"{v/1000:.1f}K" if v >= 1000 else f"{v}"
        return f"{row['district']} ({formatted_v})"
    
    district_totals['district_label'] = district_totals.apply(format_vol_label, axis=1)
    
    # Merge label mới vào DataFrame gốc
    df = df.merge(district_totals[['district', 'district_label']], on='district')
    
    # Sắp xếp Y-axis theo Volume tăng dần (để khi Plotly vẽ từ dưới lên, Quận to nhất nằm TRÊN CÙNG)
    dist_label_order = district_totals.sort_values(by='volume', ascending=True)['district_label'].tolist()
    
    # Pivot Data cho Heatmap Matrix theo district_label mới
    pivot_breach = df.pivot(index='district_label', columns='service_type', values='breach_rate').loc[dist_label_order, SERVICES]
    pivot_p90 = df.pivot(index='district_label', columns='service_type', values='p90_ttp').loc[dist_label_order, SERVICES]
    pivot_cancel = df.pivot(index='district_label', columns='service_type', values='cancel_rate').loc[dist_label_order, SERVICES]
    pivot_vol = df.pivot(index='district_label', columns='service_type', values='volume').loc[dist_label_order, SERVICES]
    pivot_breached_count = df.pivot(index='district_label', columns='service_type', values='breached_orders').loc[dist_label_order, SERVICES]

    # Matrix 4D Customdata cho Hover Tooltip
    customdata = np.dstack((
        pivot_p90.values,
        pivot_cancel.values,
        pivot_vol.values,
        pivot_breached_count.values
    ))

    with chart_out:
        chart_out.clear_output(wait=True)
        
        fig = go.Figure(data=go.Heatmap(
            z=pivot_breach.values,
            x=SERVICES,
            y=dist_label_order,
            customdata=customdata,
            colorscale=HEATMAP_COLORSCALE,
            zmin=0, zmax=55,
            colorbar=dict(
                title=dict(text="<b>SLA Breach %</b>", font=dict(size=12, family=FONT_FAMILY)),
                ticksuffix="%",
                len=0.9
            ),
            hovertemplate=(
                "<b>📍 Quận/Huyện (Volume):</b> %{y}<br>" +
                "<b>🚚 Dịch vụ:</b> %{x}<br>" +
                "----------------------------------<br>" +
                "<b>🚨 Tỷ lệ Vi phạm SLA:</b> %{z:.1f}%<br>" +
                "<b>💥 SỐ ĐƠN BỊ TRỄ SLA:</b> <b>%{customdata[3]:,} đơn</b><br>" +
                "<b>⏱️ P90 TTP (Thời gian đến lấy):</b> %{customdata[0]:.1f} phút<br>" +
                "<b>❌ Tỷ lệ Khách Hủy do Xế Trễ:</b> %{customdata[1]:.1f}%<br>" +
                "<b>📦 Volume Dịch vụ:</b> %{customdata[2]:,} đơn<extra></extra>"
            ),
            xgap=3, ygap=3
        ))

        # Hiển thị số phần trăm vi phạm trực tiếp lên ô Hotspot (>= 25%)
        annotations = []
        for i, dist_label in enumerate(dist_label_order):
            for j, svc in enumerate(SERVICES):
                val = pivot_breach.loc[dist_label, svc]
                if val >= 25.0:
                    annotations.append(dict(
                        x=svc, y=dist_label,
                        text=f"<b>{val:.1f}%</b>",
                        font=dict(color="white" if val > 35 else COLOR_NAVY, size=10, family=FONT_FAMILY),
                        showarrow=False
                    ))

        fig.update_layout(
            title=dict(
                text="<b>HEATMAP MATRIX 2A: BREAK-DOWN TOP HOTSPOTS VẠCH TRẦN LỖI VẬN HÀNH</b><br><sup>Tỷ lệ vi phạm SLA (% Breach Rate) & Khối lượng đơn hàng (Volume) từng Quận/Huyện</sup>",
                font=dict(family=FONT_FAMILY, color=COLOR_NAVY, size=15)
            ),
            annotations=annotations,
            xaxis=dict(
                title=dict(text="<b>Nhóm Dịch Vụ (Cột)</b>", font=dict(family=FONT_FAMILY)),
                side="top",
                tickfont=dict(family=FONT_FAMILY)
            ),
            yaxis=dict(
                title=dict(text="<b>22 Quận / Huyện TP.HCM & Tổng Volume (Sắp xếp giảm dần ↓)</b>", font=dict(family=FONT_FAMILY)),
                tickfont=dict(family=FONT_FAMILY)
            ),
            height=680,
            margin=dict(l=140, r=40, t=100, b=40),
            plot_bgcolor="#FFFFFF"
        )

        fig.show()

filter_scenario.observe(render_heatmap_dashboard, 'value')

# ==========================================
# 6. LAYOUT ASSEMBLY
# ==========================================
header_html = HTML(f"""
<div style="background-color: {COLOR_NAVY}; padding: 10px 16px; border-radius: 6px; margin-bottom: 12px; font-family: {FONT_FAMILY};">
    <h3 style="color: #FFFFFF; margin: 0; font-size: 16px;">🚀 AhaMove Ops Intelligence - Chart 2A: Hotspots Matrix Analysis (With Volume Context)</h3>
</div>
""")

display(VBox([
    header_html,
    HBox([filter_scenario], layout=Layout(margin='0 0 10px 0')),
    kpi_out,          # TẦNG 1: DYNAMIC KPI SUMMARY
    chart_out,        # TẦNG 2: CORE HEATMAP CHART (Có Volume Indicator trên Y-Axis)
    doc_accordion     # TẦNG 3: DOCUMENTATION & ACTION PLAN
]))

# Initial Run
render_heatmap_dashboard()

In [3]:
import os
import datetime
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Layout, VBox, HBox, HTML, Accordion, Output
from IPython.display import display

# ==========================================
# 1. CONFIGURATION & AHAMOVE BRAND PALETTE
# ==========================================
FONT_FAMILY = "-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif"

# Bảng màu chuẩn AhaMove Ops Dashboard
COLOR_NAVY = "#0B192C"        # Deep Navy (Main Text/Headers)
COLOR_BG_CARD = "#F8F9FA"     # Light Gray Card
COLOR_SUCCESS = "#2ECC71"     # Green (Ổn định)
COLOR_WARNING = "#F39C12"     # Orange (Cảnh báo)
COLOR_DANGER = "#E74C3C"      # Red (Nguy hiểm / Vỡ trận)

# Cấu hình màu cho Dịch vụ
OD_COLOR_BAR = "#3498DB"      # Blue (Pickup Distance)
OD_COLOR_LINE = "#E67E22"     # Orange (Reject Rate)
BA_COLOR_BAR = "#9B59B6"      # Purple (Wait-in-Pool)
BA_COLOR_LINE = "#34495E"     # Dark Slate (Batch Size)

# ==========================================
# 2. DATA GENERATOR (MÔ PHỎNG DỮ LIỆU)
# ==========================================
def get_dispatch_data(service_group, scenario):
    """Tạo dữ liệu linh hoạt dựa trên nhóm dịch vụ và kịch bản bất thường"""
    np.random.seed(42) # Cố định seed để demo mượt hơn
    
    if "On-Demand" in service_group:
        hours = [f"{h:02d}:00" for h in range(24)]
        pickup_dist = [1.1, 0.9, 0.8, 0.8, 0.9, 1.2, 1.4, 1.6, 1.8, 1.5, 1.4, 1.5, 1.6, 1.4, 1.3, 1.5, 1.7, 1.9, 1.7, 1.5, 1.3, 1.2, 1.1, 1.0]
        reject_rate = [12, 10, 8, 8, 9, 14, 18, 22, 24, 20, 19, 21, 23, 19, 18, 20, 24, 26, 22, 18, 16, 14, 13, 11]
        
        if scenario == 'Kịch bản: Mưa Lớn (Cạn Cung)':
            for h in range(15, 19): # Mưa chiều
                pickup_dist[h] = round(3.5 + np.random.uniform(0.2, 0.6), 2)
                reject_rate[h] = round(12.0 + np.random.uniform(-2, 2), 1)
        elif scenario == 'Kịch bản: Đơn Rác / Cồng Kềnh (Cherry-Picking)':
            for h in range(11, 15): # Trưa nắng tài xế kén cá chọn canh
                pickup_dist[h] = round(0.7 + np.random.uniform(0.1, 0.3), 2)
                reject_rate[h] = round(42.0 + np.random.uniform(2, 6), 1)
                
        return pd.DataFrame({'hour': hours, 'metric_1': pickup_dist, 'metric_2': reject_rate})

    else: # Batching (Chỉ lấy giờ hành chính 06:00 đến 18:00)
        hours = [f"{h:02d}:00" for h in range(6, 19)]
        wait_time = [45, 50, 60, 65, 55, 40, 50, 60, 55, 45, 50, 60, 45]
        batch_size = [3.5, 4.0, 4.5, 5.0, 4.8, 3.5, 4.0, 4.8, 4.5, 3.8, 4.0, 4.2, 3.0]
        
        if scenario == 'Kịch bản: Thiếu Demand (Mâm mỏng)':
            for idx, h in enumerate(range(6, 19)):
                if 9 <= h <= 11:
                    wait_time[idx] = int(105 + np.random.uniform(5, 15))
                    batch_size[idx] = round(1.2 + np.random.uniform(0.1, 0.4), 1)
        elif scenario == 'Kịch bản: Lỗi Thuật toán Routing (Zic-zac)':
            for idx, h in enumerate(range(6, 19)):
                if 14 <= h <= 16:
                    wait_time[idx] = int(115 + np.random.uniform(10, 20))
                    batch_size[idx] = round(5.8 + np.random.uniform(0.1, 0.5), 1)
                    
        return pd.DataFrame({'hour': hours, 'metric_1': wait_time, 'metric_2': batch_size})

# ==========================================
# 3. LAYER 1: EXECUTIVE KPI SUMMARY HTML
# ==========================================
def render_kpi_cards(service_group, df):
    m1, m2, hours = df['metric_1'], df['metric_2'], df['hour']
    peak_hour = hours.iloc[m2.idxmax() if "On-Demand" in service_group else m1.idxmax()]
    
    if "On-Demand" in service_group:
        max_pickup = m1.max()
        max_reject = m2.max()
        
        if max_pickup > 3.0 and max_reject < 25.0:
            diag, color, action = "🌩️ Hội Chứng Cạn Cung (Thiếu xế)", COLOR_DANGER, "Kích hoạt Surge Fee 5k / Bắn Noti kéo xế."
        elif max_pickup < 1.0 and max_reject > 35.0:
            diag, color, action = "🚫 Hội Chứng Chê Đơn (Cherry-Picking)", COLOR_WARNING, "Gắn cờ đơn cồng kềnh, thu Bulky Fee."
        else:
            diag, color, action = "✅ Vận Hành Ổn Định", COLOR_SUCCESS, "Giữ nguyên tham số gán đơn (Radius 1.5km)."

        return f"""
        <div style="display: flex; gap: 12px; margin-bottom: 12px; font-family: {FONT_FAMILY};">
            <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {OD_COLOR_BAR}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
                <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">AVG PICKUP DISTANCE</span>
                <div style="font-size: 20px; font-weight: bold; color: {COLOR_NAVY}; margin-top: 2px;">{round(m1.mean(), 2)} km</div>
                <div style="font-size: 11px; color: #7F8C8D;">Đỉnh điểm: {max_pickup}km</div>
            </div>
            <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {OD_COLOR_LINE}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
                <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">MAX REJECTION RATE</span>
                <div style="font-size: 20px; font-weight: bold; color: {OD_COLOR_LINE}; margin-top: 2px;">{max_reject}%</div>
                <div style="font-size: 11px; color: #7F8C8D;">Lúc: {peak_hour}</div>
            </div>
            <div style="flex: 1.5; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {color}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
                <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">CHẨN ĐOÁN (ROOT CAUSE)</span>
                <div style="font-size: 16px; font-weight: bold; color: {color}; margin-top: 2px;">{diag}</div>
                <div style="font-size: 12px; color: #2C3E50; margin-top: 2px;">➔ Ops Action: {action}</div>
            </div>
        </div>
        """
    else:
        max_wait = m1.max()
        min_batch = m2.min()
        
        if max_wait > 90 and min_batch < 2.0:
            diag, color, action = "📉 Mâm Cỗ Mỏng (Thiếu Demand)", COLOR_DANGER, "Bật Cross-service Batching. Push Sale chéo."
        elif max_wait > 90 and m2.max() >= 5.0:
            diag, color, action = "🗺️ Lỗi Thuật Toán Routing (Zic-zac)", COLOR_WARNING, "Giảm Max_Detour_Distance, chẻ mâm nhỏ."
        else:
            diag, color, action = "✅ Gom Chuyến Tối Ưu", COLOR_SUCCESS, "Mâm to, tài xế nhận nhanh, an toàn SLA."

        return f"""
        <div style="display: flex; gap: 12px; margin-bottom: 12px; font-family: {FONT_FAMILY};">
            <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {BA_COLOR_BAR}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
                <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">MAX WAIT-IN-POOL TIME</span>
                <div style="font-size: 20px; font-weight: bold; color: {COLOR_NAVY}; margin-top: 2px;">{max_wait} phút</div>
                <div style="font-size: 11px; color: #7F8C8D;">Lúc: {peak_hour} (Nguy hiểm khi > 90p)</div>
            </div>
            <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {BA_COLOR_LINE}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
                <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">AVG BATCH SIZE</span>
                <div style="font-size: 20px; font-weight: bold; color: {BA_COLOR_LINE}; margin-top: 2px;">{round(m2.mean(), 1)} đơn/chuyến</div>
                <div style="font-size: 11px; color: #7F8C8D;">Thấp nhất: {min_batch} đơn</div>
            </div>
            <div style="flex: 1.5; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {color}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
                <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">CHẨN ĐOÁN (ROOT CAUSE)</span>
                <div style="font-size: 16px; font-weight: bold; color: {color}; margin-top: 2px;">{diag}</div>
                <div style="font-size: 12px; color: #2C3E50; margin-top: 2px;">➔ Tech Action: {action}</div>
            </div>
        </div>
        """

# ==========================================
# 4. LAYER 3: DETAILED READ-ME ACCORDION
# ==========================================
def create_dispatch_documentation() -> Accordion:
    # --------------------------------------
    # TAB 1: VIEW TỨC THỜI (ON-DEMAND)
    # --------------------------------------
    doc_ondemand = HTML(f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; line-height: 1.6; color: #2C3E50;">
        <div style="background-color: #0B192C; color: #FFFFFF; padding: 8px 12px; border-radius: 4px; font-weight: bold; margin-bottom: 12px; font-size: 13px;">
            🎮 KỊCH BẢN TÁC CHIẾN OPS — DỊCH VỤ TỨC THỜI (SIÊU TỐC / NHANH)
        </div>
        
        <table style="width:100%; border-collapse: collapse; font-size: 12px; box-shadow: 0 1px 3px rgba(0,0,0,0.1);">
            <thead>
                <tr style="background-color: #F2F4F4; color: #0B192C; text-align: left; border-bottom: 2px solid #BDC3C7;">
                    <th style="padding: 10px; width: 22%;">Tình Huống Ops</th>
                    <th style="padding: 10px; width: 28%;">Nhận Diện Trên Chart</th>
                    <th style="padding: 10px; width: 25%;">Nguyên Nhân Cốt Lõi (Root Cause)</th>
                    <th style="padding: 10px; width: 25%;">Quy Trình Xử Lý (Standard Action)</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1; font-weight: bold; color: #E74C3C;">
                        Tình huống A:<br>Cạn Cung Cục Bộ<br><i>(Supply Shortage)</i>
                    </td>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1;">
                        • <b>Cột Pickup Distance:</b> > 2.0 km<br>
                        • <b>Tỷ lệ Reject Rate:</b> < 20% (Thấp)
                    </td>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1;">
                        Thiếu hụt mật độ tài xế thời điểm thực tại khu vực. Thuật toán phải quét rộng bán kính đón đơn nhưng tài xế vẫn hợp tác nhận đơn.
                    </td>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1; background-color: #FDEDEC;">
                        <b>1. Dynamic Pricing:</b> Bật Surge Fee (Thắt chặt nhân giá theo Zone).<br>
                        <b>2. Supply Re-location:</b> Bắn Noti Push kèm Bonus ngắn hạn để kéo xế từ Zone lân cận.
                    </td>
                </tr>
                <tr style="background-color: #FAFAFA;">
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1; font-weight: bold; color: #E67E22;">
                        Tình huống B:<br>Ăn Thịt Dịch Vụ<br><i>(Cannibalization)</i>
                    </td>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1;">
                        • <b>Cột Pickup Distance:</b> < 1.0 km<br>
                        • <b>Reject Siêu Tốc:</b> Thấp<br>
                        • <b>Reject Nhanh:</b> > 40% (Giật đỉnh)
                    </td>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1;">
                        Tài xế cố tình chê đơn "Nhanh" cước thấp để ôm máy chờ "giựt" đơn "Siêu Tốc" có thu nhập tốt hơn dù mật độ xe rất dày.
                    </td>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1; background-color: #FEF5E7;">
                        <b>1. Pricing Gap:</b> Thu hẹp chênh lệch giá giữa Siêu Tốc & Nhanh.<br>
                        <b>2. Destination Masking:</b> Ẩn điểm đến đơn Siêu Tốc đối với xế có tỷ lệ Reject đơn Nhanh cao.
                    </td>
                </tr>
                <tr>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1; font-weight: bold; color: #C0392B;">
                        Tình huống C:<br>Kén Đơn / Đơn Rác<br><i>(Cherry-Picking)</i>
                    </td>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1;">
                        • <b>Cột Pickup Distance:</b> Thấp (< 1km)<br>
                        • <b> Reject Rate (Cả 2 dịch vụ):</b> Vọt trần (> 35%)
                    </td>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1;">
                        Chất lượng đơn hàng kém (hàng cồng kềnh quá tải, giao hẻm sâu, khu vực ngập nước) khiến toàn bộ xế quanh đó đồng loạt từ chối.
                    </td>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1; background-color: #FDEDEC;">
                        <b>1. Drill-down Analysis:</b> Filter các đơn có Note cồng kềnh/hẻm.<br>
                        <b>2. Extra Surcharge:</b> Tự động áp Bulky Fee / Alley Fee để bù đắp công sức cho xế.
                    </td>
                </tr>
            </tbody>
        </table>
    </div>
    """)

    # --------------------------------------
    # TAB 2: VIEW GOM CHUYẾN (BATCHING)
    # --------------------------------------
    doc_batching = HTML(f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; line-height: 1.6; color: #2C3E50;">
        <div style="background-color: #0B192C; color: #FFFFFF; padding: 8px 12px; border-radius: 4px; font-weight: bold; margin-bottom: 12px; font-size: 13px;">
            🎮 KỊCH BẢN TÁC CHIẾN OPS — DỊCH VỤ GOM CHUYẾN (4H / ĐỒNG GIÁ)
        </div>
        
        <table style="width:100%; border-collapse: collapse; font-size: 12px; box-shadow: 0 1px 3px rgba(0,0,0,0.1);">
            <thead>
                <tr style="background-color: #F2F4F4; color: #0B192C; text-align: left; border-bottom: 2px solid #BDC3C7;">
                    <th style="padding: 10px; width: 22%;">Tình Huống Ops</th>
                    <th style="padding: 10px; width: 28%;">Nhận Diện Trên Chart</th>
                    <th style="padding: 10px; width: 25%;">Nguyên Nhân Cốt Lõi (Root Cause)</th>
                    <th style="padding: 10px; width: 25%;">Quy Trình Xử Lý (Standard Action)</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1; font-weight: bold; color: #D35400;">
                        Tình huống D:<br>Đói Đơn / Khô Thanh Khoản<br><i>(Low Liquidity Pool)</i>
                    </td>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1;">
                        • <b>Wait-in-Pool Time:</b> > 90 phút<br>
                        • <b>Batch Size:</b> < 2.0 đơn/chuyến
                    </td>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1;">
                        Mật độ đơn tạo mới quá mỏng (Low Demand), thuật toán không đủ dữ liệu trùng tuyến để ghép mâm lớn, dẫn đến ngâm đơn chờ quá thời lượng SLA.
                    </td>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1; background-color: #FEF5E7;">
                        <b>1. Cross-service Batching:</b> Cho phép thuật toán gom đơn 4H ghép chung đường với đơn Đồng Giá.<br>
                        <b>2. Growth Action:</b> Đẩy Voucher kích cầu Demand theo khung giờ thiếu hụt.
                    </td>
                </tr>
                <tr style="background-color: #FAFAFA;">
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1; font-weight: bold; color: #8E44AD;">
                        Tình huống E:<br>Lỗi Lộ Trình Zic-zac<br><i>(Routing & Detour Failure)</i>
                    </td>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1;">
                        • <b>Wait-in-Pool Time:</b> > 90 phút<br>
                        • <b>Batch Size:</b> ≥ 5.0 đơn (Mâm to)
                    </td>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1;">
                        Mâm cỗ to nhưng tài xế liên tục bỏ qua/từ chối do lộ trình tối ưu hóa kém (đường vòng vèo, băng qua nhiều điểm kẹt xe, tổng quãng đường quá xa).
                    </td>
                    <td style="padding: 10px; border-bottom: 1px solid #ECF0F1; background-color: #F5EEF8;">
                        <b>1. Tech Ticket:</b> Báo gấp cho Algorithmic Team hạ tham số <code>Max_Detour_Distance</code>.<br>
                        <b>2. Batch Splitting:</b> Ép hệ thống chẻ mâm lớn thành các mâm nhỏ (3 đơn) đi theo trục đường thẳng.
                    </td>
                </tr>
            </tbody>
        </table>
    </div>
    """)

    acc = Accordion(children=[doc_ondemand, doc_batching])
    acc.set_title(0, "📖 HƯỚNG DẪN XỬ LÝ SỰ CỐ: DỊCH VỤ TỨC THỜI (ON-DEMAND)")
    acc.set_title(1, "📖 HƯỚNG DẪN XỬ LÝ SỰ CỐ: DỊCH VỤ GOM CHUYẾN (BATCHING)")
    acc.selected_index = None
    return acc
# ==========================================
# 5. CONTROLLERS & DASHBOARD RENDERER
# ==========================================
filter_service = widgets.Dropdown(options=['Nhóm 1: Tức Thời (On-Demand) - Siêu Tốc/Nhanh', 'Nhóm 2: Gom Chuyến (Batching) - 4H/Đồng Giá'], value='Nhóm 1: Tức Thời (On-Demand) - Siêu Tốc/Nhanh', description='Dịch Vụ:', layout=Layout(width='45%'))
filter_scenario = widgets.Dropdown(options=['Bình Thường (Baseline)', 'Kịch bản: Mưa Lớn (Cạn Cung)', 'Kịch bản: Đơn Rác / Cồng Kềnh (Cherry-Picking)'], description='Tình huống:', layout=Layout(width='45%'))

def update_scenario_options(*args):
    if "On-Demand" in filter_service.value:
        filter_scenario.options = ['Bình Thường (Baseline)', 'Kịch bản: Mưa Lớn (Cạn Cung)', 'Kịch bản: Đơn Rác / Cồng Kềnh (Cherry-Picking)']
    else:
        filter_scenario.options = ['Bình Thường (Baseline)', 'Kịch bản: Thiếu Demand (Mâm mỏng)', 'Kịch bản: Lỗi Thuật toán Routing (Zic-zac)']
        
filter_service.observe(update_scenario_options, 'value')

kpi_out = HTML()
chart_out = Output()
doc_accordion = create_dispatch_documentation()

def render_dashboard(*args):
    svc = filter_service.value
    df = get_dispatch_data(svc, filter_scenario.value)
    
    kpi_out.value = render_kpi_cards(svc, df)
    
    with chart_out:
        chart_out.clear_output(wait=True)
        # Sử dụng secondary_y=True để sửa lỗi 2 trục đè nhau làm biến dạng cột Bar
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        if "On-Demand" in svc:
            fig.add_trace(go.Bar(
                x=df['hour'], y=df['metric_1'], name="Avg Pickup Distance (km)",
                marker_color=OD_COLOR_BAR, opacity=0.85,
                hovertemplate="Giờ: %{x}<br>Pickup Dist: %{y}km<extra></extra>"
            ), secondary_y=False)
            
            fig.add_trace(go.Scatter(
                x=df['hour'], y=df['metric_2'], name="Driver Reject Rate (%)",
                mode="lines+markers", line=dict(color=OD_COLOR_LINE, width=3),
                marker=dict(size=8, symbol='circle'),
                hovertemplate="Giờ: %{x}<br>Reject Rate: %{y}%<extra></extra>"
            ), secondary_y=True)
            
            # Reference Lines
            fig.add_hline(y=1.5, line_dash="dash", line_color=COLOR_SUCCESS, annotation_text="Target Pickup (1.5km)", secondary_y=False)
            fig.add_hline(y=25, line_dash="dash", line_color=COLOR_DANGER, annotation_text="Warning Reject (25%)", secondary_y=True)
            
            y1_title, y2_title = "Khoảng cách Pickup (km)", "Tỷ lệ Từ chối (%)"
            
        else:
            fig.add_trace(go.Bar(
                x=df['hour'], y=df['metric_1'], name="Wait-in-Pool Time (Phút)",
                marker_color=BA_COLOR_BAR, opacity=0.85,
                hovertemplate="Giờ: %{x}<br>Wait Time: %{y} phút<extra></extra>"
            ), secondary_y=False)
            
            fig.add_trace(go.Scatter(
                x=df['hour'], y=df['metric_2'], name="Avg Batch Size (Đơn/Chuyến)",
                mode="lines+markers", line=dict(color=BA_COLOR_LINE, width=3),
                marker=dict(size=8, symbol='diamond'),
                hovertemplate="Giờ: %{x}<br>Batch Size: %{y} đơn<extra></extra>"
            ), secondary_y=True)
            
            # Reference Lines
            fig.add_hline(y=90, line_dash="dash", line_color=COLOR_DANGER, annotation_text="Warning SLA (90 Phút)", secondary_y=False)
            
            y1_title, y2_title = "Thời gian Chờ Pool (Phút)", "Số Đơn / Chuyến"

        # Tinh chỉnh UI cho Chart
        fig.update_layout(
            title=dict(text=f"<b>PHÂN TÍCH LOGIC ĐIỀU PHỐI (DISPATCHING DYNAMICS)</b>", font=dict(family=FONT_FAMILY, color=COLOR_NAVY, size=16)),
            plot_bgcolor="#FFFFFF", hovermode="x unified",
            height=480, margin=dict(l=40, r=40, t=60, b=40),
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )
        fig.update_xaxes(title_text="Khung Giờ Trong Ngày", gridcolor="#F0F0F0", tickangle=0)
        fig.update_yaxes(title_text=y1_title, gridcolor="#F0F0F0", secondary_y=False)
        fig.update_yaxes(title_text=y2_title, showgrid=False, secondary_y=True)
        
        fig.show()

filter_service.observe(render_dashboard, 'value')
filter_scenario.observe(render_dashboard, 'value')

# ==========================================
# 6. LAYOUT ASSEMBLY & DISPLAY
# ==========================================
header_html = HTML(f"""
<div style="background-color: {COLOR_NAVY}; padding: 10px 16px; border-radius: 6px; margin-bottom: 10px; font-family: {FONT_FAMILY};">
    <h3 style="color: #FFFFFF; margin: 0; font-size: 16px;">🚀 AhaMove SGN - Cấp độ Phân Tích Logic Thuật Toán Điều Phối</h3>
</div>
""")

display(VBox([
    header_html,
    HBox([filter_service, filter_scenario], layout=Layout(margin='0 0 15px 0')),
    kpi_out,          # TẦNG 1: KPI SUMMARY
    chart_out,        # TẦNG 2: CORE VISUAL CHARTS (Đã fix lỗi scale trục Y)
    doc_accordion     # TẦNG 3: DOCUMENTATION
]))

render_dashboard()

In [11]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Layout, VBox, HBox, HTML, Accordion, Output
from IPython.display import display

# ==========================================
# 1. AHAMOVE BRAND PALETTE & CONFIGURATION
# ==========================================
FONT_FAMILY = "-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif"

COLOR_NAVY = "#0B192C"        # Header Text & Dark Accent
COLOR_BG_CARD = "#F8F9FA"     # Card Background
COLOR_NORMAL_BLUE = "#2980B9" # Đơn Bình Thường (Neutral Cool Blue)
COLOR_DANGER = "#E74C3C"      # Đơn Gian Lận / Vi Phạm Red Zone (High-contrast Red)

# ==========================================
# 2. MÔ PHỎNG DỮ LIỆU & TÍNH MẬT ĐỘ (DATA ENGINE)
# ==========================================
def generate_scatter_data(sample_size=1200):
    np.random.seed(42)
    
    # --- PANEL A: ON-DEMAND ---
    n_a = sample_size
    dist_a_norm = np.random.uniform(0.2, 4.8, int(n_a * 0.82))
    ttp_a_norm = 3.0 * dist_a_norm + np.random.normal(3.5, 2.0, int(n_a * 0.82))
    ttp_a_norm = np.clip(ttp_a_norm, 2.0, 14.0)
    
    dist_a_fraud = np.random.uniform(0.2, 1.45, int(n_a * 0.18))
    ttp_a_fraud = np.random.uniform(16.0, 52.0, int(n_a * 0.18))
    
    dist_a = np.concatenate([dist_a_norm, dist_a_fraud])
    ttp_a = np.concatenate([ttp_a_norm, ttp_a_fraud])
    
    xy_a = np.vstack([dist_a, ttp_a])
    kde_a = stats.gaussian_kde(xy_a)(xy_a)
    
    df_a = pd.DataFrame({
        'order_id': [f"ORD-A-{1000+i}" for i in range(len(dist_a))],
        'driver_id': [f"DRV-{np.random.randint(100, 999)}" for _ in range(len(dist_a))],
        'service_group': 'On-Demand (Siêu Tốc / Nhanh)',
        'distance_km': np.round(dist_a, 2),
        'ttp_minutes': np.round(ttp_a, 1),
        'density': kde_a,
        'is_red_zone': (dist_a < 1.5) & (ttp_a > 15.0)
    })
    
    # --- PANEL B: BATCHING ---
    n_b = sample_size
    dist_b_norm = np.random.uniform(0.3, 4.8, int(n_b * 0.80))
    ttp_b_norm = 28.0 + 3.0 * dist_b_norm + np.random.normal(0, 5.0, int(n_b * 0.80))
    ttp_b_norm = np.clip(ttp_b_norm, 15.0, 44.0)
    
    dist_b_fraud = np.random.uniform(0.2, 1.95, int(n_b * 0.20))
    ttp_b_fraud = np.random.uniform(46.0, 105.0, int(n_b * 0.20))
    
    dist_b = np.concatenate([dist_b_norm, dist_b_fraud])
    ttp_b = np.concatenate([ttp_b_norm, ttp_b_fraud])
    
    xy_b = np.vstack([dist_b, ttp_b])
    kde_b = stats.gaussian_kde(xy_b)(xy_b)
    
    df_b = pd.DataFrame({
        'order_id': [f"ORD-B-{2000+i}" for i in range(len(dist_b))],
        'driver_id': [f"DRV-{np.random.randint(100, 999)}" for _ in range(len(dist_b))],
        'service_group': 'Batching (4H / Đồng Giá)',
        'distance_km': np.round(dist_b, 2),
        'ttp_minutes': np.round(ttp_b, 1),
        'density': kde_b,
        'is_red_zone': (dist_b < 2.0) & (ttp_b > 45.0)
    })
    
    return df_a, df_b

# ==========================================
# 3. LAYER 1: DYNAMIC EXECUTIVE SUMMARY
# ==========================================
def render_kpi_summary(df_a, df_b):
    total_a = len(df_a)
    red_a = df_a['is_red_zone'].sum()
    pct_red_a = (red_a / total_a) * 100
    
    total_b = len(df_b)
    red_b = df_b['is_red_zone'].sum()
    pct_red_b = (red_b / total_b) * 100
    
    total_orders = total_a + total_b
    total_red = red_a + red_b
    
    return f"""
    <div style="display: flex; gap: 12px; margin-bottom: 12px; font-family: {FONT_FAMILY};">
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {COLOR_NAVY}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">MẪU ĐƠN HÀNG PHÂN TÍCH</span>
            <div style="font-size: 20px; font-weight: bold; color: {COLOR_NAVY}; margin-top: 2px;">{total_orders:,} đơn</div>
            <div style="font-size: 11px; color: #7F8C8D;">On-Demand: {total_a:,} | Batching: {total_b:,}</div>
        </div>
        <div style="flex: 1.2; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {COLOR_DANGER}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">VI PHẠM RED ZONE (GIAN LẬN/NGÂM ĐƠN)</span>
            <div style="font-size: 20px; font-weight: bold; color: {COLOR_DANGER}; margin-top: 2px;">{total_red:,} đơn ({total_red/total_orders*100:.1f}%)</div>
            <div style="font-size: 11px; color: #7F8C8D;">Siêu Tốc: <b>{pct_red_a:.1f}%</b> | 4H/Đồng Giá: <b>{pct_red_b:.1f}%</b></div>
        </div>
        <div style="flex: 1.8; background: {COLOR_BG_CARD}; padding: 10px 14px; border-radius: 8px; border-left: 5px solid {COLOR_DANGER}; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <span style="font-size: 11px; color: #7F8C8D; font-weight: bold; text-transform: uppercase;">CHẨN ĐOÁN HÀNH VI TÀI XẾ (OPS DIAGNOSIS)</span>
            <div style="font-size: 13px; font-weight: bold; color: {COLOR_DANGER}; margin-top: 2px;">🚨 Báo động: Siêu Tốc dính gian lận Multi-apping ({red_a} đơn) & 4H ngâm đơn ({red_b} đơn)</div>
            <div style="font-size: 11px; color: #2C3E50; margin-top: 2px;"><b>➔ Lằn ranh đỏ:</b> Đơn xanh (An toàn) nằm ở dải dưới, đơn đỏ (Vi phạm) tập trung trong Red Zone.</div>
        </div>
    </div>
    """

# ==========================================
# 4. LAYER 3: DOCUMENTATION ACCORDION
# ==========================================
def create_scatter_documentation() -> Accordion:
    doc_business = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: #2C3E50;">
        <h4 style="color: {COLOR_NAVY}; margin-top: 0;">1. BÓC TÁC HÀNH VI BẤT THƯỜNG TRÊN BIỂU ĐỒ (BUSINESS INSIGHTS)</h4>
        
        <p><b>👉 Panel A: Siêu Tốc & Nhanh (On-Demand) — Gian lận Multi-apping:</b></p>
        <ul>
            <li><b>Cam kết:</b> Siêu Tốc là dịch vụ thu giá cao nhất (15.7k/2km đầu), khách trả tiền mua thời gian. Vận tốc chuẩn nội thành là 20km/h (~ 3 phút/km).</li>
            <li><b>Hành vi gian lận (Red Zone màu Đỏ):</b> Tài xế đứng ngay gần shop (X &lt; 1.5 km), bấm Accept đơn AhaMove lập tức để "xí phần". Tuy nhiên họ đang cầm đơn ShopeeFood/GrabFood chưa giao xong. Tài xế "ngâm" đơn AhaMove 12-15 phút để chạy đi giao trà sữa rồi mới quay lại shop.</li>
            <li><b>Hậu quả:</b> TTP vọt lên &gt; 15 phút ở cự ly cực gần. Khách hàng xem app thấy tài xế chạy ngược hướng &rarr; Tỷ lệ hủy đơn và khiếu nại tăng vọt.</li>
        </ul>

        <p><b>👉 Panel B: 4H & Đồng Giá (Batching) — Hành vi Ngâm Đơn Nhởn Nhơ:</b></p>
        <ul>
            <li><b>Cam kết:</b> Dịch vụ cước rẻ, cho phép gom mâm 5 đơn với tổng SLA giao trong 4 tiếng. Lý thuyết tài xế được phép chờ ghép chuyến (~ 30 phút buffer).</li>
            <li><b>Rủi ro vỡ trận (Red Zone màu Đỏ):</b> Đứng cách shop &lt; 2.0 km nhưng ngâm quá 45 phút mới đến lấy. Dù chưa trễ SLA 4 tiếng của khách, việc ngâm lấy hàng quá lâu khiến rủi ro Fail SLA tổng dây chuyền vọt lên <b>80%</b>.</li>
        </ul>
    </div>
    """)

    doc_data_eng = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: #2C3E50;">
        <h4 style="color: {COLOR_NAVY}; margin-top: 0;">2. KIẾN THỨC DATA ENGINEERING & SPATIAL DATA CALCULATIONS</h4>
        <p><b>1. Trục X (Khoảng cách ST_Distance):</b> Tọa độ lúc Accept <code>(accept_lat, accept_lng)</code> và Tọa độ Shop <code>(from_lat, from_lng)</code>. Sử dụng hệ số <b>Circuity Factor (1.3 - 1.41)</b> để quy đổi từ đường chim bay sang đường thực tế.</p>
        <p><b>2. Trục Y (Time to Pickup) & Auto-board:</b> Bắt buộc filter cờ <code>is_auto_board = FALSE</code> để loại bỏ nhiễu khi tài xế quên bấm nút tại điểm lấy hàng.</p>
    </div>
    """)

    doc_ops_action = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: #2C3E50;">
        <h4 style="color: {COLOR_NAVY}; margin-top: 0;">3. KHUNG HÀNH ĐỘNG OPS MANAGER (RULES ENGINE & PENALTY)</h4>
        <p><b>Auto-Warning Bot:</b> Bắn Pop-up nhắc nhở khi phát hiện ngâm đơn Siêu Tốc &gt; 10 phút tại cự ly &lt; 1.5km.</p>
        <p><b>Trừng phạt (Penalty):</b> Khóa quyền nhận đơn Siêu Tốc 3 ngày nếu vi phạm Red Zone &ge; 3 lần/tuần.</p>
    </div>
    """)

    acc = Accordion(children=[doc_business, doc_data_eng, doc_ops_action])
    acc.set_title(0, "📌 1. BÓC TÁC BẢN CHẤT BUSINESS (MULTI-APPING VS SLOTH)")
    acc.set_title(1, "📌 2. KIẾN THỨC DATA ENGINEERING (SQL SPATIAL & CIRCUITY FACTOR)")
    acc.set_title(2, "📌 3. THIẾT LẬP RULES ENGINE TÁC CHIẾN CHO OPS MANAGER")
    acc.selected_index = None
    return acc

# ==========================================
# 5. CONTROLLERS & IMPROVED TRELLIS SCATTER RENDERER
# ==========================================
df_a, df_b = generate_scatter_data(sample_size=1200)

filter_view = widgets.Dropdown(
    options=['Tất Cả Đơn Hàng', 'Chỉ Hiển Thị Red Zone (Gian Lận/Ngâm Đơn)'],
    value='Tất Cả Đơn Hàng',
    description='Chế Độ Xem:',
    layout=Layout(width='45%')
)

kpi_out = HTML()
chart_out = Output()
doc_accordion = create_scatter_documentation()

def render_trellis_scatter_dashboard(*args):
    view_mode = filter_view.value
    
    if 'Red Zone' in view_mode:
        data_a = df_a[df_a['is_red_zone']].copy()
        data_b = df_b[df_b['is_red_zone']].copy()
    else:
        data_a = df_a.copy()
        data_b = df_b.copy()
        
    kpi_out.value = render_kpi_summary(df_a, df_b)

    with chart_out:
        chart_out.clear_output(wait=True)
        
        # Subplot Layout với tiêu đề rõ ràng
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=(
                "<b>PANEL A: ON-DEMAND (Siêu Tốc & Nhanh)</b>",
                "<b>PANEL B: BATCHING (4H & Đồng Giá)</b>"
            ),
            horizontal_spacing=0.08
        )

        # ----------------------------------------------------
        # PANEL A: ON-DEMAND SCATTER
        # ----------------------------------------------------
        # Red Zone Box
        fig.add_shape(
            type="rect", x0=0, y0=15, x1=1.5, y1=120,
            fillcolor="rgba(231, 76, 60, 0.12)",
            line=dict(color=COLOR_DANGER, width=2, dash="dash"),
            row=1, col=1
        )
        
        # Baseline Line
        x_base = np.linspace(0, 5, 100)
        fig.add_trace(go.Scatter(
            x=x_base, y=3.0 * x_base,
            mode='lines', line=dict(color='#5D6D7E', width=2, dash='dot'),
            name='Baseline (20km/h ~ 3p/km)',
            legendgroup='base_a', showlegend=True
        ), row=1, col=1)

        # 1. Đơn Bình Thường (Normal - Neutral Blue KDE)
        norm_a = data_a[~data_a['is_red_zone']]
        fig.add_trace(go.Scatter(
            x=norm_a['distance_km'],
            y=norm_a['ttp_minutes'],
            mode='markers',
            marker=dict(
                size=5.5,
                color=norm_a['density'],
                colorscale='Blues', # Neutral Cool Palette
                opacity=0.7,
                showscale=False
            ),
            customdata=np.dstack((norm_a['order_id'], norm_a['driver_id']))[0],
            hovertemplate=(
                "<b>📦 Mã Đơn:</b> %{customdata[0]}<br>" +
                "<b>🪪 Tài Xế:</b> %{customdata[1]}<br>" +
                "<b>📍 Khoảng Cách Accept:</b> %{x:.2f} km<br>" +
                "<b>⏱️ TTP:</b> %{y:.1f} phút (An Toàn)<extra></extra>"
            ),
            name='Đơn Bình Thường',
            legendgroup='norm', showlegend=True
        ), row=1, col=1)

        # 2. Đơn Vi Phạm Red Zone (High-contrast Alert Red)
        fraud_a = data_a[data_a['is_red_zone']]
        fig.add_trace(go.Scatter(
            x=fraud_a['distance_km'],
            y=fraud_a['ttp_minutes'],
            mode='markers',
            marker=dict(
                size=7.5,
                color=COLOR_DANGER,
                symbol='circle',
                opacity=0.9,
                line=dict(color='#900C3F', width=0.8)
            ),
            customdata=np.dstack((fraud_a['order_id'], fraud_a['driver_id']))[0],
            hovertemplate=(
                "<b>🚨 CẢNH BÁO GIAN LẬN</b><br>" +
                "<b>📦 Mã Đơn:</b> %{customdata[0]}<br>" +
                "<b>🪪 Tài Xế:</b> %{customdata[1]}<br>" +
                "<b>📍 Khoảng Cách Accept:</b> %{x:.2f} km<br>" +
                "<b>⏱️ TTP:</b> %{y:.1f} phút (&gt;15p)<extra></extra>"
            ),
            name='Gian Lận Red Zone',
            legendgroup='fraud', showlegend=True
        ), row=1, col=1)

        # Annotation Panel A
        fig.add_annotation(
            x=0.75, y=70,
            text="<b>RED ZONE GIAN LẬN</b><br>Multi-apping (Chạy chui)<br><i>(Dist &lt; 1.5km | TTP &gt; 15m)</i>",
            font=dict(color=COLOR_DANGER, size=10, family=FONT_FAMILY),
            showarrow=True, arrowhead=2, ax=45, ay=0,
            bgcolor="white", bordercolor=COLOR_DANGER, borderwidth=1.5,
            row=1, col=1
        )

        # ----------------------------------------------------
        # PANEL B: BATCHING SCATTER
        # ----------------------------------------------------
        # Red Zone Box
        fig.add_shape(
            type="rect", x0=0, y0=45, x1=2.0, y1=120,
            fillcolor="rgba(231, 76, 60, 0.12)",
            line=dict(color=COLOR_DANGER, width=2, dash="dash"),
            row=1, col=2
        )

        # Baseline Line
        fig.add_trace(go.Scatter(
            x=x_base, y=30.0 + 3.0 * x_base,
            mode='lines', line=dict(color='#5D6D7E', width=2, dash='dot'),
            name='Baseline 4H (Buffer 30p + 3p/km)',
            legendgroup='base_b', showlegend=True
        ), row=1, col=2)

        # 1. Đơn Bình Thường
        norm_b = data_b[~data_b['is_red_zone']]
        fig.add_trace(go.Scatter(
            x=norm_b['distance_km'],
            y=norm_b['ttp_minutes'],
            mode='markers',
            marker=dict(
                size=5.5,
                color=norm_b['density'],
                colorscale='Blues',
                opacity=0.7,
                showscale=False
            ),
            customdata=np.dstack((norm_b['order_id'], norm_b['driver_id']))[0],
            hovertemplate=(
                "<b>📦 Mã Đơn:</b> %{customdata[0]}<br>" +
                "<b>🪪 Tài Xế:</b> %{customdata[1]}<br>" +
                "<b>📍 Khoảng Cách Accept:</b> %{x:.2f} km<br>" +
                "<b>⏱️ TTP:</b> %{y:.1f} phút (An Toàn)<extra></extra>"
            ),
            showlegend=False
        ), row=1, col=2)

        # 2. Đơn Vi Phạm Red Zone
        fraud_b = data_b[data_b['is_red_zone']]
        fig.add_trace(go.Scatter(
            x=fraud_b['distance_km'],
            y=fraud_b['ttp_minutes'],
            mode='markers',
            marker=dict(
                size=7.5,
                color=COLOR_DANGER,
                symbol='circle',
                opacity=0.9,
                line=dict(color='#900C3F', width=0.8)
            ),
            customdata=np.dstack((fraud_b['order_id'], fraud_b['driver_id']))[0],
            hovertemplate=(
                "<b>🚨 CẢNH BÁO NGÂM ĐƠN</b><br>" +
                "<b>📦 Mã Đơn:</b> %{customdata[0]}<br>" +
                "<b>🪪 Tài Xế:</b> %{customdata[1]}<br>" +
                "<b>📍 Khoảng Cách Accept:</b> %{x:.2f} km<br>" +
                "<b>⏱️ TTP:</b> %{y:.1f} phút (&gt;45p)<extra></extra>"
            ),
            showlegend=False
        ), row=1, col=2)

        # Annotation Panel B
        fig.add_annotation(
            x=1.0, y=85,
            text="<b>RED ZONE NGÂM ĐƠN</b><br>Rủi ro vỡ SLA 4 tiếng<br><i>(Dist &lt; 2.0km | TTP &gt; 45m)</i>",
            font=dict(color=COLOR_DANGER, size=10, family=FONT_FAMILY),
            showarrow=True, arrowhead=2, ax=45, ay=0,
            bgcolor="white", bordercolor=COLOR_DANGER, borderwidth=1.5,
            row=1, col=2
        )

        # ----------------------------------------------------
        # LAYOUT & MARGIN FIXES (ĐÃ TỐI ƯU CHỐNG ĐÈ CHỮ)
        # ----------------------------------------------------
        fig.update_xaxes(
            title_text="<b>Accept Distance (Km)</b>",
            range=[0, 5.0],
            title_font=dict(family=FONT_FAMILY, size=11),
            tickfont=dict(family=FONT_FAMILY, size=10),
            row=1, col=1
        )
        fig.update_xaxes(
            title_text="<b>Accept Distance (Km)</b>",
            range=[0, 5.0],
            title_font=dict(family=FONT_FAMILY, size=11),
            tickfont=dict(family=FONT_FAMILY, size=10),
            row=1, col=2
        )

        fig.update_yaxes(
            title_text="<b>Time to Pickup - TTP (Phút)</b>",
            range=[0, 115],
            title_font=dict(family=FONT_FAMILY, size=11),
            tickfont=dict(family=FONT_FAMILY, size=10),
            row=1, col=1
        )
        fig.update_yaxes(
            title_text="<b>Time to Pickup - TTP (Phút)</b>",
            range=[0, 115],
            title_font=dict(family=FONT_FAMILY, size=11),
            tickfont=dict(family=FONT_FAMILY, size=10),
            row=1, col=2
        )

        # Tăng Margin Top (t=130) để tiêu đề không đè nhau
        fig.update_layout(
            title=dict(
                text="<b>CHART 3.1: ANALYTICAL SCATTER PLOT - PHÁT HIỆN GIAN LẬN & HÀNH VI NGÂM ĐƠN</b><br><sup>Tách biệt On-Demand vs Batching để định danh chính xác gian lận Chạy chui (Multi-apping) và Ngâm đơn (Sloth)</sup>",
                font=dict(family=FONT_FAMILY, color=COLOR_NAVY, size=14),
                x=0.01, xanchor='left', y=0.98
            ),
            height=580,
            margin=dict(l=60, r=40, t=130, b=50),
            plot_bgcolor="#FFFFFF",
            legend=dict(
                orientation="h",
                yanchor="bottom", y=1.04,
                xanchor="right", x=1,
                font=dict(family=FONT_FAMILY, size=10)
            )
        )

        # Điều chỉnh font cho Subplot Titles
        fig.for_each_annotation(lambda a: a.update(font=dict(size=12, family=FONT_FAMILY, color=COLOR_NAVY)) if "PANEL" in a.text else None)

        fig.show()

filter_view.observe(render_trellis_scatter_dashboard, 'value')

# ==========================================
# 6. LAYOUT ASSEMBLY
# ==========================================
header_html = HTML(f"""
<div style="background-color: {COLOR_NAVY}; padding: 10px 16px; border-radius: 6px; margin-bottom: 12px; font-family: {FONT_FAMILY};">
    <h3 style="color: #FFFFFF; margin: 0; font-size: 16px;">🚀 AhaMove Ops Intelligence - Chart 3.1: Spatial Anomaly Scatter Plot</h3>
</div>
""")

display(VBox([
    header_html,
    HBox([filter_view], layout=Layout(margin='0 0 10px 0')),
    kpi_out,
    chart_out,
    doc_accordion
]))

render_trellis_scatter_dashboard()

In [15]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Layout, VBox, HBox, HTML, Accordion, Output
from IPython.display import display

# ==========================================
# 1. SENIOR COLOR PALETTE & STYLING SYSTEM
# ==========================================
FONT_FAMILY = "-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif"

# Executive Palette Configuration
COLOR_AHA_NAVY = "#0F172A"       # Primary Dark Accent (Deep Slate Navy)
COLOR_BREACH = "#C0392B"         # Đỏ đậm pha cam (Burnt Crimson / High Alert)
COLOR_SAFE = "#78909C"           # Xám pha xanh nhạt (Cool Slate Gray / Safe Zone)
COLOR_BG_CARD = "#FFFFFF"        # Pure White Card
COLOR_BG_CONTAINER = "#F8FAFC"   # Modern Dashboard Soft Slate Background
COLOR_BORDER = "#E2E8F0"         # Subtle Gray Border
COLOR_TEXT_DARK = "#1E293B"      # Dark Charcoal Text
COLOR_TEXT_MUTED = "#64748B"     # Muted Slate Text

# ==========================================
# 2. MÔ PHỎNG DỮ LIỆU B2B MERCHANT (DATA ENGINE)
# ==========================================
def generate_merchant_friction_data():
    np.random.seed(101)
    
    # Group 1: On-Demand Merchants (Siêu Tốc / Nhanh - Mall, F&B, Retail)
    ondemand_partners = [
        ("An Nam Gourmet - Landmark 81", 24.8, 420),
        ("Highlands - Vincom Đồng Khởi", 22.5, 850),
        ("Uniqlo - Saigon Centre", 21.2, 630),
        ("Zara - Vincom Center Q1", 19.4, 510),
        ("Phúc Long - Takashimaya", 17.8, 790),
        ("Shopee Express - Hub Q10", 14.2, 1150),
        ("Pizza Hut - Lê Văn Sỹ", 11.5, 340),
        ("KFC - Nguyễn Trãi", 10.2, 480),
        ("Thế Giới Di Động - CMT8", 9.1, 920),
        ("Pharmacity - Hai Bà Trưng", 7.8, 1340)
    ]
    
    # Group 2: Batching Merchants (4H / Đồng Giá - Tổng Kho KCN, Logistics Hubs)
    batching_partners = [
        ("Kho Shopee Express - KCN Vĩnh Lộc", 49.2, 2100),
        ("Kho Boxme - KCN Cát Lái", 46.5, 1450),
        ("Tổng kho Lazada - KCN Tân Bình", 42.8, 3100),
        ("Kho ViettelPost - Sóng Thần", 39.4, 1850),
        ("Kho Tiki Logistics - KCN Tân Thuận", 37.6, 2400),
        ("Kho Ninjavan - Bình Tân", 32.1, 1200),
        ("Kho GHN - Tân Phú", 28.5, 1950),
        ("Kho Sendo - Q12", 24.8, 880),
        ("Kho Pharmacity Hub - Bình Dương", 21.3, 1600),
        ("Kho BEST Express - Củ Chi", 19.5, 750)
    ]
    
    records = []
    for partner, avg_time, orders in ondemand_partners:
        records.append({
            'partner_name': partner,
            'service_group': 'Siêu Tốc / Nhanh (On-Demand)',
            'avg_time_to_board': avg_time,
            'total_orders': orders,
            'sla_target': 15.0,
            'is_breached': avg_time > 15.0
        })
        
    for partner, avg_time, orders in batching_partners:
        records.append({
            'partner_name': partner,
            'service_group': '4H / Đồng Giá (Batching)',
            'avg_time_to_board': avg_time,
            'total_orders': orders,
            'sla_target': 45.0,
            'is_breached': avg_time > 45.0
        })
        
    return pd.DataFrame(records)

df_b2b_raw = generate_merchant_friction_data()

# ==========================================
# 3. LAYER 1: DYNAMIC EXECUTIVE SUMMARY (EXECUTIVE CARDS)
# ==========================================
def render_kpi_summary(df_filtered, service_selected):
    total_merchants = len(df_filtered)
    breached_merchants = df_filtered['is_breached'].sum()
    pct_breached = (breached_merchants / total_merchants) * 100 if total_merchants > 0 else 0
    
    worst_partner = df_filtered.sort_values(by='avg_time_to_board', ascending=False).iloc[0]
    avg_friction_all = df_filtered['avg_time_to_board'].mean()
    sla_target = df_filtered['sla_target'].iloc[0]
    target_text = "15 phút" if "On-Demand" in service_selected else "45 phút"
    
    return f"""
    <div style="display: flex; gap: 14px; margin-bottom: 16px; font-family: {FONT_FAMILY}; background: {COLOR_BG_CONTAINER}; padding: 12px; border-radius: 12px; border: 1px solid {COLOR_BORDER};">
        <div style="flex: 1; background: {COLOR_BG_CARD}; padding: 14px 16px; border-radius: 10px; border: 1px solid {COLOR_BORDER}; box-shadow: 0 2px 4px rgba(0,0,0,0.02);">
            <div style="font-size: 11px; color: {COLOR_TEXT_MUTED}; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px;">SỐ LƯỢNG PARTNER B2B</div>
            <div style="font-size: 22px; font-weight: 800; color: {COLOR_AHA_NAVY}; margin-top: 4px;">{total_merchants} Kho/Điểm</div>
            <div style="font-size: 11px; color: {COLOR_TEXT_MUTED}; margin-top: 2px;">📍 Dispatch Distance &lt; 2km</div>
        </div>
        <div style="flex: 1.2; background: {COLOR_BG_CARD}; padding: 14px 16px; border-radius: 10px; border-left: 4px solid {COLOR_BREACH}; border-top: 1px solid {COLOR_BORDER}; border-right: 1px solid {COLOR_BORDER}; border-bottom: 1px solid {COLOR_BORDER}; box-shadow: 0 2px 4px rgba(0,0,0,0.02);">
            <div style="font-size: 11px; color: {COLOR_TEXT_MUTED}; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px;">TỶ LỆ VƯỢT CHUẨN SLA ({target_text})</div>
            <div style="font-size: 22px; font-weight: 800; color: {COLOR_BREACH}; margin-top: 4px;">{pct_breached:.0f}% <span style="font-size: 14px; font-weight: 600;">({breached_merchants}/{total_merchants} kho)</span></div>
            <div style="font-size: 11px; color: {COLOR_TEXT_MUTED}; margin-top: 2px;">TTB TB Nhóm: <b>{avg_friction_all:.1f} phút</b></div>
        </div>
        <div style="flex: 1.8; background: {COLOR_BG_CARD}; padding: 14px 16px; border-radius: 10px; border-left: 4px solid {COLOR_BREACH}; border-top: 1px solid {COLOR_BORDER}; border-right: 1px solid {COLOR_BORDER}; border-bottom: 1px solid {COLOR_BORDER}; box-shadow: 0 2px 4px rgba(0,0,0,0.02);">
            <div style="font-size: 11px; color: {COLOR_TEXT_MUTED}; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px;">ĐIỂM NGHỄN B2B BÁO ĐỘNG NHẤT</div>
            <div style="font-size: 14px; font-weight: 700; color: {COLOR_BREACH}; margin-top: 4px;">🚨 {worst_partner['partner_name']}</div>
            <div style="font-size: 11px; color: {COLOR_TEXT_DARK}; margin-top: 2px;">Thời gian Board: <b>{worst_partner['avg_time_to_board']:.1f} phút</b> (Vượt <b>+{worst_partner['avg_time_to_board'] - sla_target:.1f}p</b> so với chuẩn).</div>
        </div>
    </div>
    """

# ==========================================
# 4. LAYER 3: DOCUMENTATION ACCORDION
# ==========================================
def create_friction_documentation() -> Accordion:
    doc_business = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: {COLOR_TEXT_DARK};">
        <h4 style="color: {COLOR_AHA_NAVY}; margin-top: 0;">1. PHÂN TÍCH MA SÁT B2B (BOARDING FRICTION ANALYSIS)</h4>
        <p><b>👉 Tại sao Dispatch Distance &lt; 2km mà thời gian Board lại mất 20-50 phút?</b></p>
        <ul>
            <li><b>TTM/Mall (On-Demand):</b> Tài xế di chuyển 500m mất 2 phút, nhưng tốn 15-20 phút gửi xe dưới hầm, đi bộ lên tầng cao, xếp hàng chờ F&B/Retail chuẩn bị đơn. Sóng GPS bị nghẽn dưới hầm làm vô hiệu hóa Geofencing.</li>
            <li><b>Tổng kho KCN (Batching 4H):</b> Tài xế mất 30-40 phút làm thủ tục an ninh (đổi CCCD, chờ Barie, phân luồng). Đây là thời gian "thẩm thấu" kho bãi, không phải lỗi tài xế di chuyển chậm.</li>
        </ul>
        <p><b>➔ Hậu quả:</b> Tài xế bị gãy SLA, ức chế và dẫn đến làn sóng <b>Boycott (tẩy chay)</b> không nhận đơn từ các Partner B2B này.</p>
    </div>
    """)

    doc_data_eng = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: {COLOR_TEXT_DARK};">
        <h4 style="color: {COLOR_AHA_NAVY}; margin-top: 0;">2. QUY TRÌNH DATA ENGINEERING & SQL METRIC</h4>
        <p><b>1. Hệ số Vòng Vèo (Circuity Factor 1.3):</b> <code>ST_Distance * 1.3 &lt; 2km</code> đảm bảo quy đổi từ đường chim bay sang quãng đường di chuyển thực tế của tài xế là hoàn toàn ngắn.</p>
        <p><b>2. Quy luật số lớn (Law of Large Numbers):</b> Lọc <code>HAVING COUNT(order_id) &gt; 100</code> để loại bỏ sai số ngẫu nhiên (ví dụ tài xế bị xịt lốp). Nếu 100+ tài xế đều mất 35 phút vào lấy hàng, 100% lỗi do quy trình vận hành của Kho/Mall.</p>
    </div>
    """)

    doc_ops_action = HTML(f"""
    <div style="font-family: {FONT_FAMILY}; font-size: 13px; line-height: 1.6; color: {COLOR_TEXT_DARK};">
        <h4 style="color: {COLOR_AHA_NAVY}; margin-top: 0;">3. CHƯƠNG TRÌNH HÀNH ĐỘNG CỦA BD & PRICING TEAM</h4>
        <ul>
            <li><b>Hành động 1 (BD Negotiation):</b> BD Director cầm Chart 3.2 làm việc trực tiếp với Quản lý Kho/Mall, yêu cầu mở Line ưu tiên cho AhaMove hoặc giản lược thủ tục an ninh.</li>
            <li><b>Hành động 2 (Waiting Fee / Phí Chờ):</b> Nếu Partner không thể sửa quy trình, Pricing Team sẽ cấu hình thuật toán tự động cộng thêm <b>10,000đ - 15,000đ Phí chờ</b> vào cước phí đơn hàng tại tọa độ kho đó. Tiền này chuyển cho tài xế và tính vào hóa đơn cuối tháng của Partner.</li>
            <li><b>Hành động 3 (Geofencing Fallback):</b> Mở rộng bán kính Geofencing từ 100m lên 300m tại các Mall lớn hoặc bật tính năng <i>GPS Bypass (Chụp hình bãi xe)</i> để tài xế kịp bấm Boarding.</li>
        </ul>
    </div>
    """)

    acc = Accordion(children=[doc_business, doc_data_eng, doc_ops_action])
    acc.set_title(0, "📌 1. BẢN CHẤT BUSINESS: MA SÁT KHO BÃI & CẠN KIỆT SỨC CHỜ CỦA TÀI XẾ")
    acc.set_title(1, "📌 2. QUY TRÌNH DATA ENGINEERING (CIRCUITY FACTOR & LAW OF LARGE NUMBERS)")
    acc.set_title(2, "📌 3. THƯƠNG LƯỢNG THƯƠNG MẠI (BD NEGOTIATION, PHÍ CHỜ & GEOFENCING)")
    acc.selected_index = None
    return acc

# ==========================================
# 5. CONTROLLERS & PLOTLY CHART (PLOTTING ENGINE)
# ==========================================
filter_service = widgets.Dropdown(
    options=['Siêu Tốc / Nhanh (On-Demand)', '4H / Đồng Giá (Batching)'],
    value='Siêu Tốc / Nhanh (On-Demand)',
    description='Nhóm Dịch Vụ:',
    layout=Layout(width='40%')
)

filter_distance_text = widgets.HTML(
    value=f"<span style='font-family: {FONT_FAMILY}; font-size: 12px; color: {COLOR_TEXT_MUTED}; background: #EDF2F7; padding: 6px 12px; border-radius: 6px;'>🔒 <b>Global Filter Active:</b> Dispatch Distance &lt; 2.0 km (Circuity Factor 1.3)</span>",
    layout=Layout(margin='0 0 0 10px')
)

kpi_out = HTML()
chart_out = Output()
doc_accordion = create_friction_documentation()

def render_merchant_friction_dashboard(*args):
    service_selected = filter_service.value
    
    df_filtered = df_b2b_raw[df_b2b_raw['service_group'] == service_selected].copy()
    df_filtered = df_filtered.sort_values(by='avg_time_to_board', ascending=True)
    
    kpi_out.value = render_kpi_summary(df_filtered, service_selected)
    
    sla_target = df_filtered['sla_target'].iloc[0]
    ref_title = "Chuẩn Siêu Tốc (15 phút)" if "On-Demand" in service_selected else "Chuẩn 4H (45 phút)"
    
    # Palette assignment: Đỏ đậm pha cam (Vi phạm) vs Xám pha xanh nhạt (Đạt SLA)
    colors = [COLOR_BREACH if breached else COLOR_SAFE for breached in df_filtered['is_breached']]
    
    with chart_out:
        chart_out.clear_output(wait=True)
        
        fig = go.Figure()
        
        # Horizontal Bars
        fig.add_trace(go.Bar(
            y=df_filtered['partner_name'],
            x=df_filtered['avg_time_to_board'],
            orientation='h',
            marker=dict(
                color=colors,
                line=dict(color='rgba(0,0,0,0.06)', width=1)
            ),
            text=[f" <b>{val:.1f} phút</b>" for val in df_filtered['avg_time_to_board']],
            textposition='outside',
            textfont=dict(family=FONT_FAMILY, size=11, color=COLOR_TEXT_DARK),
            customdata=df_filtered['total_orders'],
            hovertemplate=(
                "<b>🏢 Partner B2B:</b> %{y}<br>" +
                "<b>⏱️ Avg Time to Board:</b> %{x:.1f} phút<br>" +
                "<b>📦 Tổng Sản Lượng Đơn:</b> %{customdata:,} đơn<br>" +
                "<extra></extra>"
            ),
            showlegend=False
        ))
        
        # Reference Line (SLA Target Threshold)
        max_x = max(df_filtered['avg_time_to_board'].max() * 1.18, sla_target * 1.25)
        
        fig.add_shape(
            type="line",
            x0=sla_target, y0=-0.5,
            x1=sla_target, y1=len(df_filtered) - 0.5,
            line=dict(color=COLOR_BREACH, width=2, dash="dash")
        )
        
        # Annotation for SLA Line (Clean Badge layout anchored at top)
        fig.add_annotation(
            x=sla_target,
            y=1.05,
            yref='paper',
            text=f" <b>{ref_title}</b> ",
            font=dict(family=FONT_FAMILY, color="#FFFFFF", size=11),
            showarrow=False,
            bgcolor=COLOR_BREACH,
            bordercolor=COLOR_BREACH,
            borderwidth=1,
            borderpad=4
        )
        
        # Layout & Formatting (Generous Left Margin = 250px)
        fig.update_xaxes(
            title_text="<b>Thời Gian Trung Bình Từ Accept Đến Board - Avg Time to Board (Phút)</b>",
            range=[0, max_x],
            title_font=dict(family=FONT_FAMILY, size=11, color=COLOR_TEXT_DARK),
            tickfont=dict(family=FONT_FAMILY, size=10, color=COLOR_TEXT_MUTED),
            gridcolor="#EDF2F7",
            zeroline=False
        )
        
        fig.update_yaxes(
            title_text="<b>Đối Tác / Kho Hàng B2B</b>",
            title_font=dict(family=FONT_FAMILY, size=11, color=COLOR_TEXT_DARK),
            tickfont=dict(family=FONT_FAMILY, size=11, color=COLOR_AHA_NAVY),
            gridcolor="#F7FAFC"
        )
        
        fig.update_layout(
            title=dict(
                text="<b>CHART 3.2: B2B MERCHANT BOARDING FRICTION (XẾP HẠNG MA SÁT ĐIỂM LẤY HÀNG)</b><br><sup>Lọc các đơn có Dispatch Distance < 2km.sup>",
                font=dict(family=FONT_FAMILY, color=COLOR_AHA_NAVY, size=14),
                x=0.01, xanchor='left', y=0.98
            ),
            height=530,
            margin=dict(l=250, r=50, t=95, b=50),
            plot_bgcolor="#FFFFFF",
            paper_bgcolor="#FFFFFF"
        )
        
        fig.show()

filter_service.observe(render_merchant_friction_dashboard, 'value')

# ==========================================
# 6. LAYOUT ASSEMBLY
# ==========================================
header_html = HTML(f"""
<div style="background-color: {COLOR_AHA_NAVY}; padding: 12px 18px; border-radius: 8px; margin-bottom: 14px; font-family: {FONT_FAMILY}; display: flex; justify-content: space-between; align-items: center;">
    <h3 style="color: #FFFFFF; margin: 0; font-size: 16px; font-weight: 700;">🚀 AhaMove Ops Intelligence — Chart 3.2: B2B Merchant Boarding Friction Ranking</h3>
    <span style="color: #F8FAFC; font-weight: 600; font-size: 12px; background: rgba(255,255,255,0.12); padding: 4px 10px; border-radius: 4px;">EXECUTIVE REFINED PALETTE</span>
</div>
""")

display(VBox([
    header_html,
    HBox([filter_service, filter_distance_text], layout=Layout(align_items='center', margin='0 0 12px 0')),
    kpi_out,
    chart_out,
    doc_accordion
]))

render_merchant_friction_dashboard()